# Setup

## Imports

In [5]:
import pandas as pd
import requests
from pandas import json_normalize
import json
import numpy as np
import re
import altair as alt
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.express as px
import nfl_data_py as nfl
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from PIL import Image
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler

pd.options.mode.copy_on_write = True

This part is importing from the API, turning into a JSON file, and then turning the JSON file into a pandas DF.

# CLASS - LEAGUE

In [ ]:
class League:
    def __init__(self,year, id):
        self.year = year
        self.id = id
        self.SettingsJSONtoDF()
        self.UsersJSONtoDF()
        self.schedule = nfl.import_schedules([year])
        self.StructureWeekIDs()
        self.WeeklyNFLData = nfl.import_weekly_data([year])
        self.StructureNFLData()
        self.Rosters = nfl.import_seasonal_rosters([year])
        self.PlayerTeamImport()

    def SettingsJSONtoDF(self):
        league_settings_request = requests.get(f'https://api.sleeper.app/v1/league/{self.id}')
        league_settings_json = league_settings_request.json()

        self.league_settings = json_normalize(league_settings_json)
    
    def UsersJSONtoDF(self):
        league_user_response = requests.get(f'https://api.sleeper.app/v1/league/{self.id}/users')
        league_user_json = league_user_response.json()

        self.members = json_normalize(league_user_json)

    def StructureWeekIDs(self):
        self.schedule['teams'] = self.schedule[['home_team','away_team']].values.tolist()
        self.schedule = self.schedule.explode('teams')
        self.schedule['week_id'] = self.schedule['teams'] + '-' + self.schedule['week'].astype(str)

    def StructureNFLData(self):
        self.WeeklyNFLData['week_id'] = self.WeeklyNFLData['recent_team'] + '-' + self.WeeklyNFLData['week'].astype(str)

    def PlayerTeamImport(self):
        self.player_team_DF_Import = self.WeeklyNFLData[['player_display_name', 'recent_team']].drop_duplicates()
        self.player_team_DF = self.Rosters[['player_name', 'team']].drop_duplicates()

    def ImportPlayerData(self):
        player_data_response = requests.get('https://api.sleeper.app/v1/players/nfl')
        player_data_json = player_data_response.json()
        self.PlayerDataImport = player_data_json
        
    
    def Player_Pos_Dict(self,player_data):
        self.ImportPlayerData()
        player_df = pd.DataFrame.from_dict(self.PlayerDataImport, orient='index').reset_index()
        player_df = player_df.rename(columns={'index':'index_id'})
        player_mask = player_df['index_id'].astype(str).str.contains('[A-Z]')
        player_df.loc[player_mask,'full_name'] = player_df['index_id']
        player_names = player_df.set_index('player_id').to_dict()['full_name']
        player_pos = player_df.set_index('player_id').to_dict()['position']
        player_names['0'] = "Missing"
        player_names['OAK'] = 'OAK'
        player_pos['0'] = 'Missing'
        
        self.player_df = player_df
        self.player_names = player_names
        self.player_pos = player_pos
    
    def UserSetup(self):
        self.Members = self.Members.reset_index().rename({'index':'player_code'},axis=1)
        self.Members['player_code'] = self.Members['player_code'] + 1
        self.Teams = dict(zip(self.Members.player_code,self.Members.display_name))

    def ImportWeek(self, week):
        week_response = requests.get(f'https://api.sleeper.app/v1/league/{self.id}/matchups/{week}')
        week_json = week_response.json()
        return week_json
    

In [ ]:
leagueID_2019 = 464552024260734976
leagueID_2020 = 601250643570659328
leagueID_2021 = 726148502706589696
leagueID_2022 = 861038938700255232
leagueID_2023 = 992181788975620096
leagueID_2024 = 1122303756063965184

AllSeasonsList = [leagueID_2019,leagueID_2020,leagueID_2021,leagueID_2022,leagueID_2023,leagueID_2024]

leagueNumbers_Dict = {
    2019 : 464552024260734976,
    2020 : 601250643570659328,
    2021 : 726148502706589696,
    2022 : 861038938700255232,
    2023 : 992181788975620096,
    2024 : 1122303756063965184,
    2025 : None}
leagueID = leagueID_2023

In [ ]:
league_settings_response_2019 = requests.get(f'https://api.sleeper.app/v1/league/464552024260734976')
league_settings_response_2020 = requests.get(f'https://api.sleeper.app/v1/league/601250643570659328')
league_settings_response_2021 = requests.get(f'https://api.sleeper.app/v1/league/726148502706589696')
league_settings_response_2022 = requests.get(f'https://api.sleeper.app/v1/league/861038938700255232')
league_settings_response_2023 = requests.get(f'https://api.sleeper.app/v1/league/992181788975620096')
league_settings_response_2024 = requests.get(f'https://api.sleeper.app/v1/league/1122303756063965184')
league_settings_response_2025 = requests.get(f'https://api.sleeper.app/v1/league/')

league_settings_j_2019 = league_settings_response_2019.json()
league_settings_j_2020 = league_settings_response_2020.json()
league_settings_j_2021 = league_settings_response_2021.json()
league_settings_j_2022 = league_settings_response_2022.json()
league_settings_j_2023 = league_settings_response_2023.json()
league_settings_j_2024 = league_settings_response_2024.json()


league_settings_2019 = json_normalize(league_settings_j_2019)
league_settings_2020 = json_normalize(league_settings_j_2020)
league_settings_2021 = json_normalize(league_settings_j_2021)
league_settings_2022 = json_normalize(league_settings_j_2022)
league_settings_2023 = json_normalize(league_settings_j_2023)
league_settings_2024 = json_normalize(league_settings_j_2024)


In [11]:
league_settings_list = [league_settings_2019,league_settings_2020,league_settings_2021,league_settings_2022,league_settings_2023,league_settings_2024]

In [12]:
league_user_response = requests.get(f'https://api.sleeper.app/v1/league/{leagueID}/users')

In [13]:
league_user_response_2019 = requests.get(f'https://api.sleeper.app/v1/league/464552024260734976/users')
league_user_response_2020 = requests.get(f'https://api.sleeper.app/v1/league/601250643570659328/users')
league_user_response_2021 = requests.get(f'https://api.sleeper.app/v1/league/726148502706589696/users')
league_user_response_2022 = requests.get(f'https://api.sleeper.app/v1/league/861038938700255232/users')
league_user_response_2023 = requests.get(f'https://api.sleeper.app/v1/league/992181788975620096/users')
league_user_response_2024 = requests.get(f'https://api.sleeper.app/v1/league/1122303756063965184/users')

league_user_j_2019 = league_user_response_2019.json()
league_user_j_2020 = league_user_response_2020.json()
league_user_j_2021 = league_user_response_2021.json()
league_user_j_2022 = league_user_response_2022.json()
league_user_j_2023 = league_user_response_2023.json()
league_user_j_2024 = league_user_response_2024.json()

league_user_2019 = json_normalize(league_user_j_2019)
league_user_2020 = json_normalize(league_user_j_2020)
league_user_2021 = json_normalize(league_user_j_2021)
league_user_2022 = json_normalize(league_user_j_2022)
league_user_2023 = json_normalize(league_user_j_2023)
league_user_2024 = json_normalize(league_user_j_2024)



In [14]:
league_user_list = [league_user_2019,league_user_2020,league_user_2021,league_user_2022,league_user_2023,league_user_2024]

league_user_response = requests.get(f'https://api.sleeper.app/v1/league/{leagueID}/users')

league_settings_j = league_settings_response.json()
league_user_j = league_user_response.json()

league_settings = json_normalize(league_settings_j)
league_user = json_normalize(league_user_j)

### Schedule Imports

In [481]:
schedule24 = nfl.import_schedules([2024])
schedule23 = nfl.import_schedules([2023])
schedule22 = nfl.import_schedules([2022])
schedule21 = nfl.import_schedules([2021])
schedule20 = nfl.import_schedules([2020])
schedule19 = nfl.import_schedules([2019])

Schedule_Dict = {2019:schedule19, 2020:schedule20, 2021:schedule21, 2022:schedule22, 2023:schedule23, 2024:schedule24, }


for key, schedule in Schedule_Dict.items():
    schedule['teams'] = schedule[['home_team','away_team']].values.tolist()
    schedule = schedule.explode('teams')
    schedule['week_id'] = schedule['teams'] + '-' + schedule['week'].astype(str)
    Schedule_Dict[key] = schedule


In [16]:
schedule24

,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,...,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium,teams
6706,2024_01_BAL_KC,2024,REG,1,2024-09-05,Thursday,20:20,BAL,20.0,KC,...,00-0034796,00-0033873,Lamar Jackson,Patrick Mahomes,John Harbaugh,Andy Reid,Shawn Hochuli,KAN00,GEHA Field at Arrowhead Stadium,"[KC, BAL]"
6707,2024_01_GB_PHI,2024,REG,1,2024-09-06,Friday,20:15,GB,29.0,PHI,...,00-0036264,00-0036389,Jordan Love,Jalen Hurts,Matt LaFleur,Nick Sirianni,Ron Torbert,SAO00,Arena Corinthians,"[PHI, GB]"
6708,2024_01_PIT_ATL,2024,REG,1,2024-09-08,Sunday,13:00,PIT,18.0,ATL,...,00-0036945,00-0029604,Justin Fields,Kirk Cousins,Mike Tomlin,Raheem Morris,Brad Rogers,ATL97,Mercedes-Benz Stadium,"[ATL, PIT]"
6709,2024_01_ARI_BUF,2024,REG,1,2024-09-08,Sunday,13:00,ARI,28.0,BUF,...,00-0035228,00-0034857,Kyler Murray,Josh Allen,Jonathan Gannon,Sean McDermott,Tra Blake,BUF00,New Era Field,"[BUF, ARI]"
6710,2024_01_TEN_CHI,2024,REG,1,2024-09-08,Sunday,13:00,TEN,17.0,CHI,...,00-0039152,00-0039918,Will Levis,Caleb Williams,Brian Callahan,Matt Eberflus,Shawn Smith,CHI98,Soldier Field,"[CHI, TEN]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6973,2024_18_MIA_NYJ,2024,REG,18,2025-01-05,Sunday,13:00,MIA,NaN,NYJ,...,NaN,NaN,NaN,NaN,Mike McDaniel,Robert Saleh,NaN,NYC01,MetLife Stadium,"[NYJ, MIA]"
6974,2024_18_NYG_PHI,2024,REG,18,2025-01-05,Sunday,13:00,NYG,NaN,PHI,...,NaN,NaN,NaN,NaN,Brian Daboll,Nick Sirianni,NaN,PHI00,Lincoln Financial Field,"[PHI, NYG]"
6975,2024_18_CIN_PIT,2024,REG,18,2025-01-05,Sunday,13:00,CIN,NaN,PIT,...,NaN,NaN,NaN,NaN,Zac Taylor,Mike Tomlin,NaN,PIT00,Acrisure Stadium,"[PIT, CIN]"
6976,2024_18_NO_TB,2024,REG,18,2025-01-05,Sunday,13:00,NO,NaN,TB,...,NaN,NaN,NaN,NaN,Dennis Allen,Todd Bowles,NaN,TAM00,Raymond James Stadium,"[TB, NO]"


## NFL Data

In [11]:
player = nfl.import_seasonal_rosters([2019,2020,2021,2022,2023])
SeasonData = nfl.import_seasonal_data([2019,2020,2021,2022,2023], 'REG')
weekly = nfl.import_weekly_data([2019,2020,2021,2022,2023])
WeeklyNFLData_19 = nfl.import_weekly_data([2019])
WeeklyNFLData_20 = nfl.import_weekly_data([2020])
WeeklyNFLData_21 = nfl.import_weekly_data([2021])
WeeklyNFLData_22 = nfl.import_weekly_data([2022])
WeeklyNFLData_23 = nfl.import_weekly_data([2023])



URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)>

In [482]:
WeeklyNFLData_24 = nfl.import_weekly_data([2024])
WeeklyNFLData_Dict = {2019:WeeklyNFLData_19, 2020:WeeklyNFLData_20,2021:WeeklyNFLData_21,2022:WeeklyNFLData_22,2023:WeeklyNFLData_23,2024:WeeklyNFLData_24 }
SeasonPlayerData = SeasonData.merge(player,on='player_id',how='left')

Downcasting floats.


In [483]:
Rosters2019 = nfl.import_seasonal_rosters([2019])
Rosters2020 = nfl.import_seasonal_rosters([2020])
Rosters2021 = nfl.import_seasonal_rosters([2021])
Rosters2022 = nfl.import_seasonal_rosters([2022])
Rosters2023 = nfl.import_seasonal_rosters([2023])
Rosters2024 = nfl.import_seasonal_rosters([2024])

In [20]:
WeeklyNFLData_24.head()

,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,...,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
0,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,NYJ,2024,1,REG,...,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,8.580000,8.580000
1,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,NYJ,2024,2,REG,...,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,15.140000,15.140000
2,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,NYJ,2024,3,REG,...,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,21.040001,21.040001
3,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,NYJ,2024,4,REG,...,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,11.600000,11.600000
4,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,NYJ,2024,5,REG,...,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,11.760000,11.760000


In [484]:
WeeklyNFLData_24['week_id'] = WeeklyNFLData_24['recent_team'] + '-' + WeeklyNFLData_24['week'].astype(str)

In [22]:
#WeeklyNFLData_24 = WeeklyNFLData_24.merge(schedule,on = 'week_id')

In [23]:
WeeklyNFLData_24

,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,...,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr,week_id
0,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,NYJ,2024,1,REG,...,NaN,0,NaN,NaN,NaN,NaN,0.0,8.580000,8.580000,NYJ-1
1,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,NYJ,2024,2,REG,...,NaN,0,NaN,NaN,NaN,NaN,0.0,15.140000,15.140000,NYJ-2
2,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,NYJ,2024,3,REG,...,NaN,0,NaN,NaN,NaN,NaN,0.0,21.040001,21.040001,NYJ-3
3,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,NYJ,2024,4,REG,...,NaN,0,NaN,NaN,NaN,NaN,0.0,11.600000,11.600000,NYJ-4
4,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,NYJ,2024,5,REG,...,NaN,0,NaN,NaN,NaN,NaN,0.0,11.760000,11.760000,NYJ-5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4696,00-0039921,T.Benson,Trey Benson,RB,RB,https://static.www.nfl.com/image/upload/f_auto...,ARI,2024,10,REG,...,1.987017,0,-3.125,0.083333,-0.072072,0.074550,0.0,8.700000,10.700000,ARI-10
4697,00-0039921,T.Benson,Trey Benson,RB,RB,https://static.www.nfl.com/image/upload/f_auto...,ARI,2024,12,REG,...,NaN,0,NaN,NaN,NaN,NaN,0.0,1.800000,1.800000,ARI-12
4698,00-0039921,T.Benson,Trey Benson,RB,RB,https://static.www.nfl.com/image/upload/f_auto...,ARI,2024,13,REG,...,NaN,0,NaN,NaN,NaN,NaN,0.0,2.000000,2.000000,ARI-13
4699,00-0039921,T.Benson,Trey Benson,RB,RB,https://static.www.nfl.com/image/upload/f_auto...,ARI,2024,14,REG,...,-0.144496,0,-0.800,0.026316,-0.031847,0.017181,0.0,1.900000,2.900000,ARI-14


In [485]:
player_team_DF_Import = WeeklyNFLData_24[['player_display_name', 'recent_team']].drop_duplicates()

In [ ]:
player_team_DF_Import

In [486]:
player_team_DF2019 = Rosters2019[['player_name', 'team']].drop_duplicates()
player_team_DF2020 = Rosters2020[['player_name', 'team']].drop_duplicates()
player_team_DF2021 = Rosters2021[['player_name', 'team']].drop_duplicates()
player_team_DF2022 = Rosters2022[['player_name', 'team']].drop_duplicates()
player_team_DF2023 = Rosters2023[['player_name', 'team']].drop_duplicates()
player_team_DF2024 = Rosters2024[['player_name', 'team']].drop_duplicates()

In [26]:
player_team_DF_Import

,player_display_name,recent_team
0,Aaron Rodgers,NYJ
15,Marcedes Lewis,CHI
16,Joe Flacco,IND
21,Josh Johnson,BAL
24,Matthew Stafford,LA
...,...,...
4651,Michael Penix Jr.,ATL
4654,Caleb Williams,CHI
4669,Rome Odunze,CHI
4684,Malachi Corley,NYJ


## PlayerData():

Sleeper doesn't want us running the code below alot. Make sure you don't run it a bunch or they'll block the IP.

In [27]:
def PlayerData():
    player_resp = requests.get('https://api.sleeper.app/v1/players/nfl')
    player_data_json = player_resp.json()
    return player_data_json

In [28]:
runPlayerData = True
if runPlayerData == True:
    player_data_json = PlayerData()

In [29]:
player_data = player_data_json

# League User Setup

Below, we take in the league user data that covers the different teams. We're adding a column called 'player_code' so we can make a dictionary and change the display names on the dataframes. 

In [487]:
league_user_2019 = league_user_2019.reset_index().rename({'index':'player_code'},axis=1)
league_user_2020 = league_user_2020.reset_index().rename({'index':'player_code'},axis=1)
league_user_2021 = league_user_2021.reset_index().rename({'index':'player_code'},axis=1)
league_user_2022 = league_user_2022.reset_index().rename({'index':'player_code'},axis=1)
league_user_2023 = league_user_2023.reset_index().rename({'index':'player_code'},axis=1)
league_user_2024 = league_user_2024.reset_index().rename({'index':'player_code'},axis=1)

league_user_2019['player_code'] = league_user_2019['player_code'] + 1
league_user_2020['player_code'] = league_user_2020['player_code'] + 1
league_user_2021['player_code'] = league_user_2021['player_code'] + 1
league_user_2022['player_code'] = league_user_2022['player_code'] + 1
league_user_2023['player_code'] = league_user_2023['player_code'] + 1
league_user_2024['player_code'] = league_user_2024['player_code'] + 1

In [31]:
league_user_2019['player_code']

0      1
1      2
2      3
3      4
4      5
5      6
6      7
7      8
8      9
9     10
10    11
11    12
Name: player_code, dtype: int64

Below we're creating some dictionaries to help us update titles in the dataframes. 

In [32]:
league_teams_2019 = dict(zip(league_user_2019.player_code,league_user_2019.display_name))
league_teams_2020 = dict(zip(league_user_2020.player_code,league_user_2020.display_name))
league_teams_2021 = dict(zip(league_user_2021.player_code,league_user_2021.display_name))
league_teams_2022 = dict(zip(league_user_2022.player_code,league_user_2022.display_name))
league_teams_2023 = dict(zip(league_user_2023.player_code,league_user_2023.display_name))
league_teams_2024 = dict(zip(league_user_2024.player_code,league_user_2024.display_name))

LeagueTeamsDict = {}

LeagueTeamsDict[2019] = league_teams_2019
LeagueTeamsDict[2020] = league_teams_2020
LeagueTeamsDict[2021] = league_teams_2021
LeagueTeamsDict[2022] = league_teams_2022
LeagueTeamsDict[2023] = league_teams_2023
LeagueTeamsDict[2024] = league_teams_2024


positions = {0:'QB',1:'RB1', 2:'RB2', 3: 'WR1', 4:'WR2',5:'TE',6:'WRT',7:'K',8:'DEF'}

position_list = list(positions.values())
position_list

['QB', 'RB1', 'RB2', 'WR1', 'WR2', 'TE', 'WRT', 'K', 'DEF']

In [33]:
roster_ids_2019 = {1: 'bgmaddox',
 2: 'jlglover',
 3: 'akbrown29',
 4: 'RascalHazard',
 5: 'BMoreBallers88',
 6: 'eegrady',
 7: 'YouthPastor',
 8: 'BillyRayGonnaGetcha',
 9: 'GurlyGirls',
 10: 'SweetDizzzzzle',
 11: 'RossLikeSauce',
 12: 'JTizzzzle'}

In [34]:
roster_ids_2020 = {1: 'bgmaddox',
 2: 'jlglover',
 3: 'JTizzzzle',
 4: 'RascalHazard',
 5: 'BMoreBallers88',
 6: 'eegrady',
 7: 'YouthPastor',
 8: 'jhuntmadd',
 9: 'RReclam',
 10: 'SweetDizzzzzle'}

In [35]:
roster_ids_2021 = {1: 'bgmaddox',
 2: 'jlglover',
 3: 'JTizzzzle',
 4: 'RascalHazard',
 5: 'BMoreBallers88',
 6: 'eegrady',
 7: 'YouthPastor',
 8: 'jhuntmadd',
 9: 'RReclam',
 10: 'SweetDizzzzzle',
 11: 'RossLikeSauce',
 12: 'BillyRayGonnaGetcha'}

In [ ]:
roster_ids_2022 = {
 1: 'bgmaddox',
 2: 'jlglover',
 3: 'JTizzzzle',
 4: 'RascalHazard',
 5: 'BMoreBallers88',
 6: 'eegrady',
 7: 'DirtyCommie',
 8: 'jhuntmadd',
 9: 'RReclam',
 10: 'sgmaddox',
 11: 'RossLikeSauce',
 12: 'Just_Here_For_The_Snacks'}

In [37]:
roster_ids_2023 = {1: 'bgmaddox',
 2: 'jlglover',
 3: 'JTizzzzle',
 4: 'RascalHazard',
 5: 'BMoreBallers88',
 6: 'eegrady',
 7: 'DirtyCommie',
 8: 'jhuntmadd',
 9: 'RReclam',
 10: 'sgmaddox',
 11: 'RossLikeSauce',
 12: 'InfiniteJesse'}

In [38]:
roster_ids = {2019:roster_ids_2019, 2020:roster_ids_2020, 2021:roster_ids_2021, 2022:roster_ids_2022, 2023:roster_ids_2023,2024:roster_ids_2023 }

teamlist = PosistionPivot_scaled.team.unique()
teamdict = dict(enumerate(team_list,1))
teamlistorder = teamdict.values()

league_ids.update(roster_ids)

## Player_Pos_Dict():

In [ ]:
def Player_Pos_Dict(player_data):
    player_df = pd.DataFrame.from_dict(player_data, orient='index').reset_index()
    player_df = player_df.rename(columns={'index':'index_id'})
    player_mask = player_df['index_id'].astype(str).str.contains('[A-Z]')
    player_df.loc[player_mask,'full_name'] = player_df['index_id']
    player_names = player_df.set_index('player_id').to_dict()['full_name']
    player_pos = player_df.set_index('player_id').to_dict()['position']
    player_names['0'] = "Missing"
    player_names['OAK'] = 'OAK'
    player_pos['0'] = 'Missing'
    return player_df, player_names, player_pos


player_df, player_names, player_pos = Player_Pos_Dict(player_data)

In [41]:
player_names

{'6462': 'Ellis Richardson',
 '11255': 'Nick Amoah',
 '8842': 'Malkelm Morrison',
 '7926': 'Carl Tucker',
 '1875': 'C.J. Mosley',
 '8478': 'Samuel Womack',
 '995': 'Ron Parker',
 '1408': "Le'Veon Bell",
 '8074': 'Matt Seybert',
 '156': 'William Gay',
 '7311': 'Javin White',
 '2091': 'Bashaud Breeland',
 '11533': 'Brandon Aubrey',
 '2508': 'Amarlo Herrera',
 '2199': 'Kasim Edebali',
 '2881': 'Tyler Slavin',
 '2064': 'DeMarcus Lawrence',
 '1284': 'Lance Dunbar',
 '2801': 'DeAndrew White',
 '4501': 'Devaroe Lawrence',
 '5852': 'DeAndre Baker',
 '1073': 'Taylor Thompson',
 '3374': 'Kevon Seymour',
 '8012': 'Corey Straughter',
 '7918': 'A.J. Rose',
 '777': 'Erik Pears',
 '5428': 'Ricky Jeune',
 '1982': 'Aaron Colvin',
 '12376': 'Marquis Wilson',
 '7882': 'Christian Uphoff',
 '8733': 'Jacob Hummel',
 '11522': 'Jerome Kapp',
 '2474': 'Nick Boyle',
 '8852': 'Ty Clary',
 '8809': 'Aaron Shampklin',
 '9487': 'Parker Washington',
 '6923': 'Geno Stone',
 '1907': 'Johnny Manziel',
 '7325': 'Joe Gazi

In [42]:
player_names

{'6462': 'Ellis Richardson',
 '11255': 'Nick Amoah',
 '8842': 'Malkelm Morrison',
 '7926': 'Carl Tucker',
 '1875': 'C.J. Mosley',
 '8478': 'Samuel Womack',
 '995': 'Ron Parker',
 '1408': "Le'Veon Bell",
 '8074': 'Matt Seybert',
 '156': 'William Gay',
 '7311': 'Javin White',
 '2091': 'Bashaud Breeland',
 '11533': 'Brandon Aubrey',
 '2508': 'Amarlo Herrera',
 '2199': 'Kasim Edebali',
 '2881': 'Tyler Slavin',
 '2064': 'DeMarcus Lawrence',
 '1284': 'Lance Dunbar',
 '2801': 'DeAndrew White',
 '4501': 'Devaroe Lawrence',
 '5852': 'DeAndre Baker',
 '1073': 'Taylor Thompson',
 '3374': 'Kevon Seymour',
 '8012': 'Corey Straughter',
 '7918': 'A.J. Rose',
 '777': 'Erik Pears',
 '5428': 'Ricky Jeune',
 '1982': 'Aaron Colvin',
 '12376': 'Marquis Wilson',
 '7882': 'Christian Uphoff',
 '8733': 'Jacob Hummel',
 '11522': 'Jerome Kapp',
 '2474': 'Nick Boyle',
 '8852': 'Ty Clary',
 '8809': 'Aaron Shampklin',
 '9487': 'Parker Washington',
 '6923': 'Geno Stone',
 '1907': 'Johnny Manziel',
 '7325': 'Joe Gazi

In [43]:
player_pos

{'6462': 'TE',
 '11255': 'OL',
 '8842': 'CB',
 '7926': 'TE',
 '1875': 'LB',
 '8478': 'CB',
 '995': 'FS',
 '1408': 'RB',
 '8074': 'TE',
 '156': 'CB',
 '7311': 'LB',
 '2091': 'CB',
 '11533': 'K',
 '2508': 'LB',
 '2199': 'DE',
 '2881': 'WR',
 '2064': 'DE',
 '1284': 'RB',
 '2801': 'WR',
 '4501': 'DT',
 '5852': 'CB',
 '1073': 'TE',
 '3374': 'CB',
 '8012': 'CB',
 '7918': 'RB',
 '777': 'T',
 '5428': 'WR',
 '1982': 'CB',
 '12376': 'DB',
 '7882': 'DB',
 '8733': 'LB',
 '11522': 'WR',
 '2474': 'TE',
 '8852': 'OL',
 '8809': 'RB',
 '9487': 'WR',
 '6923': 'DB',
 '1907': 'QB',
 '7325': 'DE',
 '8716': 'OL',
 '6648': 'OT',
 '3342': 'WR',
 '1684': 'WR',
 '3268': 'DB',
 '4449': 'LB',
 '11072': 'DL',
 '6918': 'RB',
 '6137': 'TE',
 '11340': 'OL',
 '5668': 'G',
 '3561': 'TE',
 '832': 'WR',
 '6751': None,
 '3356': 'OT',
 '6786': 'WR',
 '5038': 'WR',
 '10938': 'LB',
 '4433': 'CB',
 '11584': 'RB',
 '1195': 'RB',
 '2023': 'WR',
 '1461': 'LB',
 '7116': 'OL',
 '4907': None,
 '11649': 'RB',
 '6783': 'WR',
 '5592':

In [44]:
player_df

,index_id,full_name,swish_id,height,injury_notes,player_id,active,search_first_name,first_name,team_abbr,...,weight,news_updated,position,opta_id,age,birth_date,hashtag,birth_state,oddsjam_id,last_name
0,6462,Ellis Richardson,NaN,75,None,6462,True,ellis,Ellis,NaN,...,245,NaN,TE,NaN,26.0,1995-02-12,#ellisrichardson-NFL-FA-45,NaN,None,Richardson
1,11255,Nick Amoah,1052592.0,74,None,11255,True,nick,Nick,NaN,...,306,NaN,OL,NaN,NaN,None,#nickamoah-NFL-FA-63,NaN,None,Amoah
2,8842,Malkelm Morrison,1131137.0,70,None,8842,True,malkelm,Malkelm,NaN,...,186,1.653352e+12,CB,NaN,NaN,None,#malkelmmorrison-NFL-FA-0,NaN,None,Morrison
3,7926,Carl Tucker,866047.0,74,None,7926,True,carl,Carl,NaN,...,250,1.631568e+12,TE,NaN,24.0,1997-02-06,#carltucker-NFL-FA-0,NaN,None,Tucker
4,1875,C.J. Mosley,557173.0,74,None,1875,True,cj,C.J.,NaN,...,231,1.733618e+12,LB,NaN,32.0,1992-06-19,#cjmosley-NFL-NYJ-57,NaN,97CDFFA7FA4D,Mosley
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10585,CHI,CHI,NaN,NaN,NaN,CHI,True,NaN,Chicago,NaN,...,NaN,NaN,DEF,NaN,NaN,NaN,NaN,NaN,NaN,Bears
10586,CIN,CIN,NaN,NaN,NaN,CIN,True,NaN,Cincinnati,NaN,...,NaN,NaN,DEF,NaN,NaN,NaN,NaN,NaN,NaN,Bengals
10587,DEN,DEN,NaN,NaN,NaN,DEN,True,NaN,Denver,NaN,...,NaN,NaN,DEF,NaN,NaN,NaN,NaN,NaN,NaN,Broncos
10588,KC,KC,NaN,NaN,NaN,KC,True,NaN,Kansas City,NaN,...,NaN,NaN,DEF,NaN,NaN,NaN,NaN,NaN,NaN,Chiefs


# Importing Weekly Data

# CLASS - WEEK

In [ ]:
Matches_2019 = {}
Matches_2020 = {}
Matches_2021 = {}
Matches_2022 = {}
Matches_2023 = {}
Matches_2024 = {}
Matches_2025 = {}

Breakout_Matches_2019 = {}
Breakout_Matches_2020 = {}
Breakout_Matches_2021 = {}
Breakout_Matches_2022 = {}
Breakout_Matches_2023 = {}
Breakout_Matches_2024 = {}
Breakout_Matches_2025 = {}

In [ ]:
class Week:
    def __init__(self,week, league):
        self.week = week
        self.league = league
        self.id = self.league.id
        self.year = self.league.year
        self.ImportWeek()
    
    def ImportWeek(self):
        week_response = requests.get(f'https://api.sleeper.app/v1/league/{self.id}/matchups/{self.week}')
        week_json = week_response.json()
        self.json = week_json

    def PlayerBreakout(self):
        # Initialize an empty list to hold the rows
        JSON_data = self.json
        player_data = self.league.player_team_DF
        rows = []
        
        Regular = list(range(1,15))
        Playoff = list(range(15,19))

        WeeklyNFLData = self.league.WeeklyNFLData
        schedule = self.league.schedule
        NFLTeamList =self.league.player_team_DF_Import['recent_team'].unique()

        Defence = pd.DataFrame(NFLTeamList)
        Defence = Defence.rename(columns={0:'player_name'})
        Defence['team'] = NFLTeamList
        
        player_team_DF = pd.concat([player_data,Defence])
        player_team_DF['player_name'] = player_team_DF['player_name'].replace(' Jr.','', regex=True).replace(' Sr.','', regex=True).replace(' III','',regex=True)
        
        #Player Corrections
        player_team_DF = player_team_DF.drop(1035)
        player_team_DF = player_team_DF.replace('Audric Estimé','Audric Estime')
        player_team_DF.loc[-1] = ['LAR','LA']

        
        player_team_DF.index = player_team_DF.index+1
        player_team_DF = player_team_DF.sort_index()
        for team in JSON_data:
            # Extract relevant information
            matchup_id = team['matchup_id']
            roster_id = team['roster_id']
            players_points = team['players_points']
            starters = team['starters']

            # Iterate through each player and their points
            for player, points in players_points.items():
                # Determine if the player is a starter
                is_starter = player in starters
                
                player_name = player_names.get(player, player)  # Default to player ID if name not found
                player_positions = player_pos.get(player, 0)

                # Create a row for each player
                rows.append({
                    'team': roster_id,
                    'matchup': matchup_id,
                    'player': player_name,
                    'points': points,
                    'starter': int(is_starter),
                    'position': player_positions
                })
        # Convert the list of rows into a DataFrame
        dfBreakout = pd.DataFrame(rows)
        league_names = roster_ids[self.year]
        dfBreakout['team'] = dfBreakout['team'].replace(league_names)
        dfBreakout['week'] = self.week
        dfBreakout = dfBreakout.merge(player_team_DF, left_on='player', right_on='player_name', how = 'left')
        dfBreakout['week_id'] = dfBreakout['team_y'] + '-' + dfBreakout['week'].astype(str)
        
        if self.week in Regular:
            dfBreakout['Season'] = 'Regular'
        elif self.week in Playoff:
            dfBreakout['Season'] = 'Playoff'

        dfBreakout['player_week_id'] = dfBreakout['player'] + ' - ' + dfBreakout['week'].astype(str)
        WeeklyNFLData['player_week_id'] = WeeklyNFLData['player_display_name'] + ' - ' + WeeklyNFLData['week'].astype(str)
        
        
        dfBreakout = dfBreakout.merge(schedule, on = 'week_id', how = 'left')
        
        dfBreakout = dfBreakout.merge(WeeklyNFLData, on = 'player_week_id', how = 'left', suffixes=('','_NFL'))
        dfBreakout['gametime'] = pd.to_datetime(dfBreakout['gametime']).dt.strftime('%I %p')
        dfBreakout['Game_date_time'] = dfBreakout['weekday'] + ' ' + dfBreakout['gametime'].astype(str).replace(r'0', "", regex=True)
        dfBreakout = dfBreakout.rename(columns={'team_x':'team','team_y':'recent_teams'})
        teamcolors = {'JTizzzzle': 'lightsalmon', 'bgmaddox':'mediumpurple', 'jlglover':'cyan', 'RascalHazard':'pink', 
                    'BMoreBallers88':'gold', 'eegrady':'teal', 'DirtyCommie':'lime', 'jhuntmadd':'orange', 'RReclam':'green', 
                    'sgmaddox':'darkseagreen', 'RossLikeSauce':'red', 'InfiniteJesse':'magenta'}
        dfBreakout['color'] = dfBreakout['team'].map(teamcolors)
        
        self.Breakout = dfBreakout
        if self.year == 2019:
            Breakout_Matches_2019[self.week] = dfBreakout
        elif self.year == 2020:
            Breakout_Matches_2020[self.week] = dfBreakout
        elif self.year == 2021:
            Breakout_Matches_2021[self.week] = dfBreakout
        elif self.year == 2022:
            Breakout_Matches_2022[self.week] = dfBreakout
        elif self.year == 2023:
            Breakout_Matches_2023[self.week] = dfBreakout   
        elif self.year == 2024:
            Breakout_Matches_2024[self.week] = dfBreakout
        elif self.year == 2025:
            Breakout_Matches_2025[self.week] = dfBreakout
        
        return dfBreakout
    
    
    def WeeklyDataframe(self):
        # Create an empty dictionary to hold the DataFrame data
        df_dict = {}
        matchup_dict = {}

        # Iterate through each team and their data
        for team in self.json:
            roster_id = team["roster_id"]
            starters = team['starters']
            starters_points = team['starters_points']
            matchup_id = team['matchup_id']
            
            # Replace player IDs with player names
            starters_with_names = [player_names[player] for player in starters]
            
            # Combine players and their points into a list where each entry is a list [dictionary, matchup_id]
            df_dict[roster_id] = [{player: points} for player, points in zip(starters_with_names, starters_points)]
            matchup_dict[roster_id] = matchup_id

        # Create a DataFrame from the dictionary
        WeeklyDf = pd.DataFrame.from_dict(df_dict, orient='index')

        WeeklyDf['Matchup'] = WeeklyDf.index.map(matchup_dict)
        # Define a function to sum the values in the dictionaries in each row
        def sum_points(row):
            total = 0
            for entry in row:
                if isinstance(entry, dict):
                    total += sum(entry.values())
            return total

        league_names = roster_ids[year]
        # Apply the function to each row to create the 'Total' column
        WeeklyDf['Total'] = WeeklyDf.apply(sum_points, axis=1)
        WeeklyDf = WeeklyDf.rename(index = league_names)
        
        # Step 1: Group by 'Matchup_ID' and get the maximum 'Total' for each group
        max_scores = WeeklyDf.groupby('Matchup')['Total'].transform('max')
    
    

        # Step 2: Create a new column 'Won' that checks if the team's 'Total' equals the max score
        WeeklyDf['Won'] = WeeklyDf['Total'] == max_scores

        # Step 3: Optional - Convert the boolean 'Won' column to 1 (win) and 0 (loss)
        WeeklyDf['Won'] = WeeklyDf['Won'].astype(int)
        
        
        WeeklyDf['Opp'] = np.where(WeeklyDf['Won'] == 1,WeeklyDf.groupby('Matchup')['Total'].transform('min'),WeeklyDf.groupby('Matchup')['Total'].transform('max'))
        
        WeeklyDf['Margin'] = WeeklyDf['Total'] - WeeklyDf['Opp']
        #WeeklyDf.loc[[WeeklyDf['Won']] == 1,'Opp Score'] = WeeklyDf.groupby('Matchup')['Total'].transform('max')
        #WeeklyDf.loc[[WeeklyDf['Won']] == 0,'Opp Score'] = WeeklyDf.groupby('Matchup')['Total'].transform('min')
        
        
        SeasonMultiplier = {2019:0, 2020:1, 2021:2, 2022:3, 2023:4, 2024:5}
        
        WeeklyDf = WeeklyDf.rename(columns=positions).sort_values('Matchup')
        WeeklyDf = WeeklyDf.reset_index().rename({'index':'Team'}, axis = 1)
        WeeklyDf['Week'] = self.week
        WeeklyDf['Season'] = "Regular" if self.week < 15 else "Playoff"
        WeeklyDf['Week Index'] = self.week + (14 * SeasonMultiplier[self.year])
        WeeklyDf['Year'] = self.year
        
        percent = WeeklyDf.groupby('Week')['Total'].sum()
        percent = percent.reset_index()
        WeekTotal = dict(zip(percent['Week'],percent['Total']))
        
        WeeklyDf['LeagueTotal'] = WeeklyDf['Week'].map(WeekTotal)
        WeeklyDf['PercentTotal'] = ((WeeklyDf['Total'] / WeeklyDf['LeagueTotal']) * 100).round(1)
        # Use groupby to assign the opposing team's name
        #WeeklyDf['Opp_team'] = WeeklyDf.groupby('Matchup')['Team'].transform(lambda x: x.shift(-1) if x.index[0] % 2 == 0 else x.shift(1))
        # Define a function to get the opponent team name for each group
        def get_opponent(teams):
            return teams[::-1]  # Reverse the order so each team gets the other team as opponent

        # Apply the function group-wise to assign opposing teams
        #WeeklyDf['Opp_team'] = WeeklyDf.groupby('Matchup')['Team'].transform(get_opponent)
        # Define a function to assign the opponent team name
        def assign_opponents(group):
            # Assuming there are exactly two teams per matchup
            teams = group['Team'].values
            if len(teams) == 2:  # For valid matchups with two teams
                group['Opp_team'] = [teams[1], teams[0]]  # Swap the teams
            return group

        # Apply the function to each matchup
        WeeklyDf = WeeklyDf.groupby('Matchup').apply(assign_opponents)
            
        WeeklyDfMatches= WeeklyDf.set_index(['Matchup','Team'])
        WeeklyDfNoMatches = WeeklyDf.set_index('Team')
        WeeklyDfClean = WeeklyDf.set_index('Team').drop(axis = 1,columns = ['Total','Won','Week','Opp','Margin'])
        
        self.WeeklyMatches = WeeklyDfMatches
        self.WeeklyNoMatches = WeeklyDfNoMatches
        self.WeeklyClean = WeeklyDfClean
        if self.year == 2019:
            Matches_2019[self.week] = self.WeeklyNoMatches.reset_index()
        elif self.year == 2020:
            Matches_2020[self.week] = self.WeeklyNoMatches.reset_index() 
        elif self.year == 2021:
            Matches_2021[self.week] = self.WeeklyNoMatches.reset_index() 
        elif self.year == 2022:
            Matches_2022[self.week] = self.WeeklyNoMatches.reset_index() 
        elif self.year == 2023:
            Matches_2023[self.week] = self.WeeklyNoMatches.reset_index()    
        elif self.year == 2024:
            Matches_2024[self.week] = self.WeeklyNoMatches.reset_index()  
        elif self.year == 2025:
            Matches_2025[self.week] = self.WeeklyNoMatches.reset_index()    
        
    
    def WeeklyGraph(self):
    
        #points = MatchDF.drop(axis = 1,columns = ['Total','Won','Week','Opp','Margin']).applymap(lambda if isinstance(x,dict): float(list(x.values())[0]))
        points = self.WeeklyMatches.applymap(lambda x: float(list(x.values())[0]) if isinstance(x, dict) else x)
        points = points.round(2).reset_index()
        
        fig1 = px.bar(points, y='Team',x=position_list,template = 'plotly_dark',color = "Matchup", barmode='group',text_auto=True, title = f'Week {self.week} Matchups', orientation='h')
        
        #Update the layout to hide the legend:
        fig1.update(layout_coloraxis_showscale=False)
        
        fig1.update_layout(barcornerradius=13)
        # Adjust the figure size
        fig1.update_layout(width=800, height=1200)
        fig1.update_layout(title=dict(
                font=dict(
                    size=50,
                    family="Courier New")))  # Set the width and height in pixels
        fig1.update_traces(textfont_size=12, textangle=0, cliponaxis=True, textposition = 'inside', textfont=dict(weight='bold',family='Courier New',  # Font family
                size=12   # Font color
            ))
        
        # Customize the x-axis labels
        fig1.update_xaxes(
            tickfont=dict(
                family='Courier New',  # Font family
                size=16,         # Font size
                color='white'    # Font color
            ),
            title = None
        )

        # Customize the y-axis labels
        fig1.update_yaxes(
            tickfont=dict(
                family='Courier New',  # Font family
                size=18,         # Font size
                color='white'    # Font color
            ),
            title=None
        )

        fig1.add_annotation(
        text="VS.",
        xref="paper", yref="paper",
        x=-.1, y=.93, # Position relative to figure (right side, middle)
        showarrow=False,
        font=dict(
            family="Courier New, monospace",
            size=15,
            color="magenta",
            weight ='bold'
        ))

        fig1.add_annotation(
        text="VS.",
        xref="paper", yref="paper",
        x=-.1, y=.76, # Position relative to figure (right side, middle)
        showarrow=False,
        font=dict(
            family="Courier New, monospace",
            size=15,
            color="magenta"
        )
        )

        fig1.add_annotation(
        text="VS.",
        xref="paper", yref="paper",
        x=-.1, y=.58, # Position relative to figure (right side, middle)
        showarrow=False,
        font=dict(
            family="Courier New, monospace",
            size=15,
            color="magenta"
        )
        )

        fig1.add_annotation(
        text="VS.",
        xref="paper", yref="paper",
        x=-.1, y=.41, # Position relative to figure (right side, middle)
        showarrow=False,
        font=dict(
            family="Courier New, monospace",
            size=15,
            color="magenta"
        )
        )

        fig1.add_annotation(
        text="VS.",
        xref="paper", yref="paper",
        x=-.1, y=.24, # Position relative to figure (right side, middle)
        showarrow=False,
        font=dict(
            family="Courier New, monospace",
            size=15,
            color="magenta"
        )
        )

        fig1.add_annotation(
        text="VS.",
        xref="paper", yref="paper",
        x=-.1, y=.07, # Position relative to figure (right side, middle)
        showarrow=False,
        font=dict(
            family="Courier New, monospace",
            size=15,
            color="magenta"
        )
        )

        fig1.add_annotation(
        text="Bar Order: QB ------------> DEF",
        xref="paper", yref="paper",
        x=0.05, y=-.06, # Position relative to figure (right side, middle)
        showarrow=False,
        font=dict(
            family="Courier New, monospace",
            size=15,
            color="white"
        )
        )
        fig1.update_traces(marker_line_width=2,marker_line_color='black')
        
        fig1.update_traces(insidetextanchor= 'middle')
        fig1.update_layout(
            uniformtext_minsize=12,
            uniformtext_mode='hide'
            )
        
        fig1.show()
        return fig1
        
    



# CLASS - SEASON

In [ ]:
class Season:
    def __init__(self, league):
        self.league = league
        self.id = self.league.id
        self.year = self.league.year
        self.AllMatchesConcat()
    
    def BreakoutConcat(self, Breakout_dict, last_week):
        # Filter out the dataframes corresponding to months up to and including the current month
        dataframes_to_concat = [Breakout_dict[week] for week in range(1, last_week + 1)]
        
        # Concatenate the selected dataframes
        Weeks= pd.concat(dataframes_to_concat, axis=0)
        
        Weeks['Score YTD'] = Weeks.groupby('player')['points'].cumsum()
        Weeks['TotalYTD'] = Weeks.groupby('player')['points'].transform('sum')
        Weeks['Year'] = self.year

        self.BreakoutSeason = Weeks

    def SetBest(self,TotalYTD):
        self.Best = self.BreakoutSeason[self.BreakoutSeason['TotalYTD'] > TotalYTD]
        print(f'Best set at {self.Best}')

    
    def AllMatchesConcat(self):
        if self.year == 2019:
            self.Matches = pd.concat(Matches_2019.values())
        elif self.year == 2020:
            self.Matches = pd.concat(Matches_2020.values())
        elif self.year == 2021:
            self.Matches = pd.concat(Matches_2021.values())
        elif self.year == 2022:
            self.Matches = pd.concat(Matches_2022.values())
        elif self.year == 2023:
            self.Matches = pd.concat(Matches_2023.values())
        elif self.year == 2024:
            self.Matches = pd.concat(Matches_2024.values())
        elif self.year == 2025:
            self.Matches = pd.concat(Matches_2025.values())
    
    def WeeklyWins(self, last_week):
        # Filter out the dataframes corresponding to months up to and including the current month
        #dataframes_to_concat = [df_dict[week] for week in range(1, last_week + 1)]
        
        # Concatenate the selected dataframes
        #Weeks= pd.concat(dataframes_to_concat, axis=0)
        Weeks = self.Matches
        Weeks['Total Wins'] = Weeks.groupby('Team')['Won'].cumsum()

        Weeks['Score YTD'] = Weeks.groupby('Team')['Total'].cumsum()

        Weeks['Opp YTD'] = Weeks.groupby('Team')['Opp'].cumsum()
        
        Week_Pivot = Weeks.pivot(index = 'Team', columns = 'Week',values = 'Total Wins')
        
        self.Week_Pivot = Week_Pivot
        self.ConcatinatedWeeks = Weeks

    def RecentTrends(self):
        WonGraph = self.Matches.pivot(columns = 'Week',index='Team',values = 'Won')
        WonGraph = WonGraph.replace(0,'L').replace(1,'W')
        last_five_cols = WonGraph.columns[-5:]
        
        WonGraph['Trend'] = WonGraph[last_five_cols].apply(lambda row: ' '.join(row.astype(str)), axis=1)
        self.WonGraph = WonGraph.reset_index()
        self.RecentTrend = dict(zip(WonGraph.Team, WonGraph.Trend))
    
    def WeeklyWinsGraph(self):
        df = self.Matches
        fig2 = px.line(df,x='Week',y='Total Wins', color = 'Team',template='plotly_dark',line_shape = 'spline')
        fig2.update_xaxes(dtick=1,
                        tickfont=dict(
                family='Courier New',  # Font family
                size=18,         # Font size
                color='white'    # Font color
            ))
        fig2.update_yaxes(dtick=1)
        fig2.update_layout(width=900, height=600)
        # Adjust the thickness of the lines
        fig2.update_traces(line=dict(width=4))  # Set the line width (e.g., 3 pixels)
        fig2.update_layout(legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1,
            title = ''
        ))

        
        # Customize the y-axis labels
        fig2.update_yaxes(
            tickfont=dict(
                family='Courier New',  # Font family
                size=20,         # Font size
                color='white'    # Font color
            )
        )

        fig2.update_layout(uniformtext_minsize=10, uniformtext_mode='hide')

        # Update x-axis and y-axis titles with font customization
        fig2.update_layout(
            xaxis_title="Week",  # Set x-axis title
            yaxis_title="Wins",   # Set y-axis title
            xaxis=dict(
                title_font=dict(
                    family="Courier New",  # Set font family for x-axis title
                    size=20          # Set font size for x-axis title
                )
            ),
            yaxis=dict(
                title_font=dict(
                    family="Courier New",  # Set font family for y-axis title
                    size=20          # Set font size for y-axis title
                )
            )
        )
        
        # Determine final wins and sort by them
        final_scores = [(d.name, d.x[-1], d.y[-1], d.line.color) for d in fig2.data]
        final_scores.sort(key=lambda x: -x[2])  # Sort by final win count, descending

        # Define a list of potential text positions to avoid overlap
        text_positions = ['middle right','top right', 'bottom right', 'top left', 'bottom left', 'middle left','top center','bottom center']

        previous_score = None
        position_index = 0

        for team_name, x_final, y_final, color in final_scores:
            if y_final == previous_score:
                # Cycle through text positions to avoid overlap if scores are the same
                position_index = (position_index + 1) % len(text_positions)
            else:
                position_index = 0  # Reset position index when score changes

            text_position = text_positions[position_index]
            previous_score = y_final

            fig2.add_scatter(
                x=[x_final], y=[y_final],
                mode='markers+text',
                text=[team_name],
                textfont=dict(family="Courier New",color=color, size=14, weight = 'bold'),
                textposition=text_position,
                marker=dict(color=color, size=12),
                showlegend=False
                )    
            fig2.update_layout(
            margin=dict(l=50, r=100, t=50, b=50)  # Set left, right, top, bottom padding within the plot area
                )
        fig2.update_layout(xaxis=dict(range=[0, week + 1])) 
        fig2.update_layout(font_family="Courier New")
        self.GRAPH_WeeklyWins = fig2.show()
        return fig2   
        

# CLASS - ALL-TIME

In [ ]:
AllSeasonsBreakoutList = [Breakout_Matches_2019, Breakout_Matches_2020, Breakout_Matches_2021, Breakout_Matches_2022, Breakout_Matches_2023, Breakout_Matches_2024]
AllMatches = [Matches_2019, Matches_2020, Matches_2021, Matches_2022, Matches_2023, Matches_2024, Matches_2025]

In [ ]:
class AllTime:
    def __init__(self, MatchesList,BreakOutList):
        self.Matches =  pd.concat(MatchesList)
        self.MatchProcessing()
        self.Breakout = pd.concat(BreakOutList)
        
        self.Breakout_Playoffs = self.Breakout[self.Breakout.Season == 'Playoff']
        self.Breakout_Regular = self.Breakout[self.Breakout.Season == 'Regular']
        
    def MatchProcessing(self):    
        self.Matches['Abs Margin'] = abs(self.Matches['Margin'].astype(float)).round(2)
        self.Matches['Margin'] = round(self.Matches['Margin'],2) 

    def TopPlayoffScores(self):
        self.TopPlayoffScores = self.Matches.sort_values('Total', ascending=False)[:10]
        self.TopPlayoffScores['Names'] = self.TopPlayoffScores.Team + ' [W' + self.TopPlayoffScores.Week.astype(str) + ' ' + self.TopPlayoffScores.Year.astype(str)+ ']' + ' - ' + self.TopPlayoffScores.Total.round(1).astype(str) 
        self.TopPlayoffScores['Year'] = self.TopPlayoffScores['Year'].astype(int)

    def OppWinPercentage(self,All_df, team, opp):
        OppTable = pd.pivot_table(All_df, values='Won',index='Team',columns='Opp_team',aggfunc='mean').round(2).fillna('')
        result = OppTable.loc[team,opp]
        return result
    
    def OppWinPercentageTable(self,All_df):
        result = pd.pivot_table(self.Matches, values='Won',index='Team',columns='Opp_team',aggfunc='mean').round(2).fillna('')
        result = result[result.columns.intersection(roster_ids_2023.values())].reset_index()
        result = result[result['Team'].isin(roster_ids_2023.values())].set_index('Team')
        self.OppWinPercentageTable = result
        return result
    


## ImportWeek():

In [45]:
def ImportWeek(week,year):
    LeagueID = leagueNumbers_Dict[year]
    week_resp = requests.get(f'https://api.sleeper.app/v1/league/{LeagueID}/matchups/{week}')
    week_json = week_resp.json()
    return week_json

## Importing Weeks

In [306]:
Week1_json_2019 = ImportWeek(1,2019)
Week2_json_2019 = ImportWeek(2,2019)
Week3_json_2019 = ImportWeek(3,2019)
Week4_json_2019 = ImportWeek(4,2019)
Week5_json_2019 = ImportWeek(5,2019)
Week6_json_2019 = ImportWeek(6,2019)
Week7_json_2019 = ImportWeek(7,2019)
Week8_json_2019 = ImportWeek(8,2019)
Week9_json_2019 = ImportWeek(9,2019)
Week10_json_2019 = ImportWeek(10,2019)
Week11_json_2019 = ImportWeek(11,2019)
Week12_json_2019 = ImportWeek(12,2019)
Week13_json_2019 = ImportWeek(13,2019)
Week14_json_2019 = ImportWeek(14,2019)
Week15_json_2019 = ImportWeek(15,2019)
Week16_json_2019 = ImportWeek(16,2019)
Week17_json_2019 = ImportWeek(17,2019)
Week18_json_2019 = ImportWeek(18,2019)



In [307]:
Week1_json_2020 = ImportWeek(1,2020)
Week2_json_2020 = ImportWeek(2,2020)
Week3_json_2020 = ImportWeek(3,2020)
Week4_json_2020 = ImportWeek(4,2020)
Week5_json_2020 = ImportWeek(5,2020)
Week6_json_2020 = ImportWeek(6,2020)
Week7_json_2020 = ImportWeek(7,2020)
Week8_json_2020 = ImportWeek(8,2020)
Week9_json_2020 = ImportWeek(9,2020)
Week10_json_2020 = ImportWeek(10,2020)
Week11_json_2020 = ImportWeek(11,2020)
Week12_json_2020 = ImportWeek(12,2020)
Week13_json_2020 = ImportWeek(13,2020)
Week14_json_2020 = ImportWeek(14,2020)
Week15_json_2020 = ImportWeek(15,2020)
Week16_json_2020 = ImportWeek(16,2020)
Week17_json_2020 = ImportWeek(17,2020)
Week18_json_2020 = ImportWeek(18,2020)



In [308]:
Week1_json_2021 = ImportWeek(1,2021)
Week2_json_2021 = ImportWeek(2,2021)
Week3_json_2021 = ImportWeek(3,2021)
Week4_json_2021 = ImportWeek(4,2021)
Week5_json_2021 = ImportWeek(5,2021)
Week6_json_2021 = ImportWeek(6,2021)
Week7_json_2021 = ImportWeek(7,2021)
Week8_json_2021 = ImportWeek(8,2021)
Week9_json_2021 = ImportWeek(9,2021)
Week10_json_2021 = ImportWeek(10,2021)
Week11_json_2021 = ImportWeek(11,2021)
Week12_json_2021 = ImportWeek(12,2021)
Week13_json_2021 = ImportWeek(13,2021)
Week14_json_2021 = ImportWeek(14,2021)
Week15_json_2021 = ImportWeek(15,2021)
Week16_json_2021 = ImportWeek(16,2021)
Week17_json_2021 = ImportWeek(17,2021)
Week18_json_2021 = ImportWeek(18,2021)



In [309]:
Week1_json_2022 = ImportWeek(1,2022)
Week2_json_2022 = ImportWeek(2,2022)
Week3_json_2022 = ImportWeek(3,2022)
Week4_json_2022 = ImportWeek(4,2022)
Week5_json_2022 = ImportWeek(5,2022)
Week6_json_2022 = ImportWeek(6,2022)
Week7_json_2022 = ImportWeek(7,2022)
Week8_json_2022 = ImportWeek(8,2022)
Week9_json_2022 = ImportWeek(9,2022)
Week10_json_2022 = ImportWeek(10,2022)
Week11_json_2022 = ImportWeek(11,2022)
Week12_json_2022 = ImportWeek(12,2022)
Week13_json_2022 = ImportWeek(13,2022)
Week14_json_2022 = ImportWeek(14,2022)
Week15_json_2022 = ImportWeek(15,2022)
Week16_json_2022 = ImportWeek(16,2022)
Week17_json_2022 = ImportWeek(17,2022)
Week18_json_2022 = ImportWeek(18,2022)



In [310]:
Week1_json_2023 = ImportWeek(1,2023)
Week2_json_2023 = ImportWeek(2,2023)
Week3_json_2023 = ImportWeek(3,2023)
Week4_json_2023 = ImportWeek(4,2023)
Week5_json_2023 = ImportWeek(5,2023)
Week6_json_2023 = ImportWeek(6,2023)
Week7_json_2023 = ImportWeek(7,2023)
Week8_json_2023 = ImportWeek(8,2023)
Week9_json_2023 = ImportWeek(9,2023)
Week10_json_2023 = ImportWeek(10,2023)
Week11_json_2023 = ImportWeek(11,2023)
Week12_json_2023 = ImportWeek(12,2023)
Week13_json_2023 = ImportWeek(13,2023)
Week14_json_2023 = ImportWeek(14,2023)
Week15_json_2023 = ImportWeek(15,2023)
Week16_json_2023 = ImportWeek(16,2023)
Week17_json_2023 = ImportWeek(17,2023)
Week18_json_2023 = ImportWeek(18,2023)




In [488]:
Week1_json_2024 = ImportWeek(1,2024)
Week2_json_2024 = ImportWeek(2,2024)
Week3_json_2024 = ImportWeek(3,2024)
Week4_json_2024 = ImportWeek(4,2024)
Week5_json_2024 = ImportWeek(5,2024)
Week6_json_2024 = ImportWeek(6,2024)
Week7_json_2024 = ImportWeek(7,2024)
Week8_json_2024 = ImportWeek(8,2024)
Week9_json_2024 = ImportWeek(9,2024)
Week10_json_2024 = ImportWeek(10,2024)
Week11_json_2024 = ImportWeek(11,2024)
Week12_json_2024 = ImportWeek(12,2024)
Week13_json_2024 = ImportWeek(13,2024)
Week14_json_2024 = ImportWeek(14,2024)
Week15_json_2024 = ImportWeek(15,2024)
Week16_json_2024 = ImportWeek(16,2024)
Week17_json_2024 = ImportWeek(17,2024)
Week18_json_2024 = ImportWeek(18,2024)



# Breakout Into Player Rows

## PlayerBreakout():

In [ ]:
def PlayerBreakout(json_data, week, year):
    # Initialize an empty list to hold the rows
    JSON_data = json_data
    rows = []
    
    Regular = list(range(1,15))
    Playoff = list(range(15,19))

    if year == 2024:
        player_team_DFYear = player_team_DF2024.drop_duplicates(subset='player_name')
    elif year == 2023:
        player_team_DFYear = player_team_DF2023.drop_duplicates(subset='player_name')
    elif year == 2022:
        player_team_DFYear = player_team_DF2022.drop_duplicates(subset='player_name')
    elif year == 2021:
        player_team_DFYear = player_team_DF2021.drop_duplicates(subset='player_name')
    elif year == 2020:
        player_team_DFYear = player_team_DF2020.drop_duplicates(subset='player_name')
    elif year == 2019:
        player_team_DFYear = player_team_DF2019.drop_duplicates(subset='player_name')
    
    WeeklyNFLData = WeeklyNFLData_Dict[year]
    schedule = Schedule_Dict[year]
    
    NFLTeamList = player_team_DF_Import['recent_team'].unique()
    Defence = pd.DataFrame(NFLTeamList)
    Defence = Defence.rename(columns={0:'player_name'})
    Defence['team'] = NFLTeamList
    
    player_team_DF = pd.concat([player_team_DFYear,Defence])
    player_team_DF['player_name'] = player_team_DF['player_name'].replace(' Jr.','', regex=True).replace(' Sr.','', regex=True).replace(' III','',regex=True)
    
    #Player Corrections
    player_team_DF = player_team_DF.drop(1035)
    player_team_DF = player_team_DF.replace('Audric Estimé','Audric Estime')
    player_team_DF.loc[-1] = ['LAR','LA']
    
    
    
    player_team_DF.index = player_team_DF.index+1
    player_team_DF = player_team_DF.sort_index()
    for team in JSON_data:
        # Extract relevant information
        matchup_id = team['matchup_id']
        roster_id = team['roster_id']
        players_points = team['players_points']
        starters = team['starters']

        # Iterate through each player and their points
        for player, points in players_points.items():
            # Determine if the player is a starter
            is_starter = player in starters
            
            player_name = player_names.get(player, player)  # Default to player ID if name not found
            player_positions = player_pos.get(player, 0)

            # Create a row for each player
            rows.append({
                'team': roster_id,
                'matchup': matchup_id,
                'player': player_name,
                'points': points,
                'starter': int(is_starter),
                'position': player_positions
            })
    # Convert the list of rows into a DataFrame
    dfBreakout = pd.DataFrame(rows)
    league_names = roster_ids[year]
    dfBreakout['team'] = dfBreakout['team'].replace(league_names)
    dfBreakout['week'] = week
    dfBreakout = dfBreakout.merge(player_team_DF, left_on='player', right_on='player_name', how = 'left')
    dfBreakout['week_id'] = dfBreakout['team_y'] + '-' + dfBreakout['week'].astype(str)
    
    if week in Regular:
        dfBreakout['Season'] = 'Regular'
    elif week in Playoff:
        dfBreakout['Season'] = 'Playoff'

    dfBreakout['player_week_id'] = dfBreakout['player'] + ' - ' + dfBreakout['week'].astype(str)
    WeeklyNFLData['player_week_id'] = WeeklyNFLData['player_display_name'] + ' - ' + WeeklyNFLData['week'].astype(str)
    
    
    dfBreakout = dfBreakout.merge(schedule, on = 'week_id', how = 'left')
    
    dfBreakout = dfBreakout.merge(WeeklyNFLData, on = 'player_week_id', how = 'left', suffixes=('','_NFL'))
    dfBreakout['gametime'] = pd.to_datetime(dfBreakout['gametime']).dt.strftime('%I %p')
    dfBreakout['Game_date_time'] = dfBreakout['weekday'] + ' ' + dfBreakout['gametime'].astype(str).replace(r'0', "", regex=True)
    dfBreakout = dfBreakout.rename(columns={'team_x':'team','team_y':'recent_teams'})
    teamcolors = {'JTizzzzle': 'lightsalmon', 'bgmaddox':'mediumpurple', 'jlglover':'cyan', 'RascalHazard':'pink', 
                'BMoreBallers88':'gold', 'eegrady':'teal', 'DirtyCommie':'lime', 'jhuntmadd':'orange', 'RReclam':'green', 
                'sgmaddox':'darkseagreen', 'RossLikeSauce':'red', 'InfiniteJesse':'magenta'}
    dfBreakout['color'] = dfBreakout['team'].map(teamcolors)
    return dfBreakout


In [53]:
player_team_DF2024

,player_name,team
0,Jason Peters,SEA
1,Aaron Rodgers,NYJ
2,Jon Ryan,SEA
3,Matt Prater,ARI
4,Marcedes Lewis,CHI
...,...,...
3207,Mike Smith Jr.,IND
3208,Devon Garrison,SEA
3209,Christian McCarroll,NE
3210,Aaron Beasley,CAR


## Breakouts for each week

In [318]:
week1Breakout2019 = PlayerBreakout(Week1_json_2019,1, 2019)
week2Breakout2019 = PlayerBreakout(Week2_json_2019,2, 2019)
week3Breakout2019 = PlayerBreakout(Week3_json_2019,3, 2019)
week4Breakout2019 = PlayerBreakout(Week4_json_2019,4, 2019)
week5Breakout2019 = PlayerBreakout(Week5_json_2019,5, 2019)
week6Breakout2019 = PlayerBreakout(Week6_json_2019,6, 2019)
week7Breakout2019 = PlayerBreakout(Week7_json_2019,7, 2019)
week8Breakout2019 = PlayerBreakout(Week8_json_2019,8, 2019)
week9Breakout2019 = PlayerBreakout(Week9_json_2019,9, 2019)
week10Breakout2019 = PlayerBreakout(Week10_json_2019,10, 2019)
week11Breakout2019 = PlayerBreakout(Week11_json_2019,11, 2019)
week12Breakout2019 = PlayerBreakout(Week12_json_2019,12, 2019)
week13Breakout2019 = PlayerBreakout(Week13_json_2019,13, 2019)
week14Breakout2019 = PlayerBreakout(Week14_json_2019,14, 2019)
week15Breakout2019 = PlayerBreakout(Week15_json_2019,15, 2019)
week16Breakout2019 = PlayerBreakout(Week16_json_2019,16, 2019)
week17Breakout2019 = PlayerBreakout(Week17_json_2019,17, 2019)


In [319]:
week17Breakout2019

,team,matchup,player,points,starter,position,week_x,player_name,recent_teams,week_id,...,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr,Game_date_time,color
0,bgmaddox,None,Eric Ebron,0.00,0,TE,17,Eric Ebron,IND,IND-17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sunday 4 PM,mediumpurple
1,bgmaddox,None,David Johnson,0.00,0,RB,17,David Johnson,ARI,ARI-17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sunday 4 PM,mediumpurple
2,bgmaddox,None,Raheem Mostert,19.80,1,RB,17,Raheem Mostert,SF,SF-17,...,0.0,-5.333333,0.090909,-0.020408,0.122078,0.0,19.299999,20.299999,Sunday 8 PM,mediumpurple
3,bgmaddox,None,Drew Brees,28.02,1,QB,17,Drew Brees,NO,NO-17,...,0.0,NaN,NaN,NaN,NaN,0.0,22.020000,22.020000,Sunday 1 PM,mediumpurple
4,bgmaddox,None,Jacoby Brissett,2.18,0,QB,17,Jacoby Brissett,IND,IND-17,...,0.0,NaN,NaN,NaN,NaN,0.0,4.180000,4.180000,Sunday 4 PM,mediumpurple
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
165,JTizzzzle,None,David Montgomery,17.30,1,RB,17,David Montgomery,CHI,CHI-17,...,0.0,NaN,NaN,NaN,NaN,0.0,17.299999,17.299999,Sunday 1 PM,lightsalmon
166,JTizzzzle,None,Mason Crosby,11.00,1,K,17,Mason Crosby,GB,GB-17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sunday 1 PM,lightsalmon
167,JTizzzzle,None,Darius Slayton,7.00,0,WR,17,Darius Slayton,NYG,NYG-17,...,0.0,0.303030,0.191489,0.316699,0.508923,0.0,5.000000,9.000000,Sunday 4 PM,lightsalmon
168,JTizzzzle,None,Aaron Rodgers,23.02,1,QB,17,Aaron Rodgers,GB,GB-17,...,0.0,NaN,NaN,NaN,NaN,0.0,19.020000,19.020000,Sunday 1 PM,lightsalmon


In [320]:
week1Breakout2020 = PlayerBreakout(Week1_json_2020,1, 2020)
week2Breakout2020 = PlayerBreakout(Week2_json_2020,2, 2020)
week3Breakout2020 = PlayerBreakout(Week3_json_2020,3, 2020)
week4Breakout2020 = PlayerBreakout(Week4_json_2020,4, 2020)
week5Breakout2020 = PlayerBreakout(Week5_json_2020,5, 2020)
week6Breakout2020 = PlayerBreakout(Week6_json_2020,6, 2020)
week7Breakout2020 = PlayerBreakout(Week7_json_2020,7, 2020)
week8Breakout2020 = PlayerBreakout(Week8_json_2020,8, 2020)
week9Breakout2020 = PlayerBreakout(Week9_json_2020,9, 2020)
week10Breakout2020 = PlayerBreakout(Week10_json_2020,10, 2020)
week11Breakout2020 = PlayerBreakout(Week11_json_2020,11, 2020)
week12Breakout2020 = PlayerBreakout(Week12_json_2020,12, 2020)
week13Breakout2020 = PlayerBreakout(Week13_json_2020,13, 2020)
week14Breakout2020 = PlayerBreakout(Week14_json_2020,14, 2020)
week15Breakout2020 = PlayerBreakout(Week15_json_2020,15, 2020)
week16Breakout2020 = PlayerBreakout(Week16_json_2020,16, 2020)
week17Breakout2020 = PlayerBreakout(Week17_json_2020,17, 2020)


In [321]:
week1Breakout2021 = PlayerBreakout(Week1_json_2021,1, 2021)
week2Breakout2021 = PlayerBreakout(Week2_json_2021,2, 2021)
week3Breakout2021 = PlayerBreakout(Week3_json_2021,3, 2021)
week4Breakout2021 = PlayerBreakout(Week4_json_2021,4, 2021)
week5Breakout2021 = PlayerBreakout(Week5_json_2021,5, 2021)
week6Breakout2021 = PlayerBreakout(Week6_json_2021,6, 2021)
week7Breakout2021 = PlayerBreakout(Week7_json_2021,7, 2021)
week8Breakout2021 = PlayerBreakout(Week8_json_2021,8, 2021)
week9Breakout2021 = PlayerBreakout(Week9_json_2021,9, 2021)
week10Breakout2021 = PlayerBreakout(Week10_json_2021,10, 2021)
week11Breakout2021 = PlayerBreakout(Week11_json_2021,11, 2021)
week12Breakout2021 = PlayerBreakout(Week12_json_2021,12, 2021)
week13Breakout2021 = PlayerBreakout(Week13_json_2021,13, 2021)
week14Breakout2021 = PlayerBreakout(Week14_json_2021,14, 2021)
week15Breakout2021 = PlayerBreakout(Week15_json_2021,15, 2021)
week16Breakout2021 = PlayerBreakout(Week16_json_2021,16, 2021)
week17Breakout2021 = PlayerBreakout(Week17_json_2021,17, 2021)


In [322]:
week1Breakout2022 = PlayerBreakout(Week1_json_2022,1, 2022)
week2Breakout2022 = PlayerBreakout(Week2_json_2022,2, 2022)
week3Breakout2022 = PlayerBreakout(Week3_json_2022,3, 2022)
week4Breakout2022 = PlayerBreakout(Week4_json_2022,4, 2022)
week5Breakout2022 = PlayerBreakout(Week5_json_2022,5, 2022)
week6Breakout2022 = PlayerBreakout(Week6_json_2022,6, 2022)
week7Breakout2022 = PlayerBreakout(Week7_json_2022,7, 2022)
week8Breakout2022 = PlayerBreakout(Week8_json_2022,8, 2022)
week9Breakout2022 = PlayerBreakout(Week9_json_2022,9, 2022)
week10Breakout2022 = PlayerBreakout(Week10_json_2022,10, 2022)
week11Breakout2022 = PlayerBreakout(Week11_json_2022,11, 2022)
week12Breakout2022 = PlayerBreakout(Week12_json_2022,12, 2022)
week13Breakout2022 = PlayerBreakout(Week13_json_2022,13, 2022)
week14Breakout2022 = PlayerBreakout(Week14_json_2022,14, 2022)
week15Breakout2022 = PlayerBreakout(Week15_json_2022,15, 2022)
week15Breakout2022 = PlayerBreakout(Week15_json_2022,15, 2022)
week16Breakout2022 = PlayerBreakout(Week16_json_2022,16, 2022)
week17Breakout2022 = PlayerBreakout(Week17_json_2022,17, 2022)


In [323]:
week1Breakout2023 = PlayerBreakout(Week1_json_2023,1, 2023)
week2Breakout2023 = PlayerBreakout(Week2_json_2023,2, 2023)
week3Breakout2023 = PlayerBreakout(Week3_json_2023,3, 2023)
week4Breakout2023 = PlayerBreakout(Week4_json_2023,4, 2023)
week5Breakout2023 = PlayerBreakout(Week5_json_2023,5, 2023)
week6Breakout2023 = PlayerBreakout(Week6_json_2023,6, 2023)
week7Breakout2023 = PlayerBreakout(Week7_json_2023,7, 2023)
week8Breakout2023 = PlayerBreakout(Week8_json_2023,8, 2023)
week9Breakout2023 = PlayerBreakout(Week9_json_2023,9, 2023)
week10Breakout2023 = PlayerBreakout(Week10_json_2023,10, 2023)
week11Breakout2023 = PlayerBreakout(Week11_json_2023,11, 2023)
week12Breakout2023 = PlayerBreakout(Week12_json_2023,12, 2023)
week13Breakout2023 = PlayerBreakout(Week13_json_2023,13, 2023)
week14Breakout2023 = PlayerBreakout(Week14_json_2023,14, 2023)
week15Breakout2023 = PlayerBreakout(Week15_json_2023,15, 2023)
week16Breakout2023 = PlayerBreakout(Week16_json_2023,16, 2023)
week17Breakout2023 = PlayerBreakout(Week17_json_2023,17, 2023)


### 2024

In [489]:
week1Breakout2024 = PlayerBreakout(Week1_json_2024,1, 2024)
week2Breakout2024 = PlayerBreakout(Week2_json_2024,2, 2024)
week3Breakout2024 = PlayerBreakout(Week3_json_2024,3, 2024)
week4Breakout2024 = PlayerBreakout(Week4_json_2024,4, 2024)
week5Breakout2024 = PlayerBreakout(Week5_json_2024,5, 2024)
week6Breakout2024 = PlayerBreakout(Week6_json_2024,6, 2024)
week7Breakout2024 = PlayerBreakout(Week7_json_2024,7, 2024)
week8Breakout2024 = PlayerBreakout(Week8_json_2024,8, 2024)
week9Breakout2024 = PlayerBreakout(Week9_json_2024,9, 2024)
week10Breakout2024 = PlayerBreakout(Week10_json_2024,10, 2024)
week11Breakout2024 = PlayerBreakout(Week11_json_2024,11, 2024)
week12Breakout2024 = PlayerBreakout(Week12_json_2024,12, 2024)
week13Breakout2024 = PlayerBreakout(Week13_json_2024,13, 2024)
week14Breakout2024 = PlayerBreakout(Week14_json_2024,14, 2024)
week15Breakout2024 = PlayerBreakout(Week15_json_2024,15, 2024)
week16Breakout2024 = PlayerBreakout(Week16_json_2024,16, 2024)
week17Breakout2024 = PlayerBreakout(Week17_json_2024,17, 2024)


In [325]:
week15Breakout2024['Season']

0      Playoff
1      Playoff
2      Playoff
3      Playoff
4      Playoff
        ...   
167    Playoff
168    Playoff
169    Playoff
170    Playoff
171    Playoff
Name: Season, Length: 172, dtype: object

## JSON_Matches Dictionary of Weeks

In [326]:
Breakout_Matches_2019 = {
    1: week1Breakout2019,
    2: week2Breakout2019,
    3: week3Breakout2019,
    4: week4Breakout2019,
    5: week5Breakout2019,
    6: week6Breakout2019,
    7: week7Breakout2019,
    8: week8Breakout2019,
    9: week9Breakout2019,
    10: week10Breakout2019,
    11: week11Breakout2019,
    12: week12Breakout2019,
    13: week13Breakout2019,
    14: week14Breakout2019,
    15: week15Breakout2019,
    16: week16Breakout2019,
    17: week17Breakout2019
}

In [327]:
Breakout_Matches_2020 = {
    1: week1Breakout2020,
    2: week2Breakout2020,
    3: week3Breakout2020,
    4: week4Breakout2020,
    5: week5Breakout2020,
    6: week6Breakout2020,
    7: week7Breakout2020,
    8: week8Breakout2020,
    9: week9Breakout2020,
    10: week10Breakout2020,
    11: week11Breakout2020,
    12: week12Breakout2020,
    13: week13Breakout2020,
    14: week14Breakout2020,
    15: week15Breakout2020,
    16: week16Breakout2020,
    17: week17Breakout2020
}

In [328]:
Breakout_Matches_2021 = {
    1: week1Breakout2021,
    2: week2Breakout2021,
    3: week3Breakout2021,
    4: week4Breakout2021,
    5: week5Breakout2021,
    6: week6Breakout2021,
    7: week7Breakout2021,
    8: week8Breakout2021,
    9: week9Breakout2021,
    10: week10Breakout2021,
    11: week11Breakout2021,
    12: week12Breakout2021,
    13: week13Breakout2021,
    14: week14Breakout2021,
    15: week15Breakout2021,
    16: week16Breakout2021,
    17: week17Breakout2021,

}

In [329]:
Breakout_Matches_2022 = {
    1: week1Breakout2022,
    2: week2Breakout2022,
    3: week3Breakout2022,
    4: week4Breakout2022,
    5: week5Breakout2022,
    6: week6Breakout2022,
    7: week7Breakout2022,
    8: week8Breakout2022,
    9: week9Breakout2022,
    10: week10Breakout2022,
    11: week11Breakout2022,
    12: week12Breakout2022,
    13: week13Breakout2022,
    14: week14Breakout2022,
    15: week15Breakout2022,
    16: week16Breakout2022,
    17: week17Breakout2022,
}

In [330]:
Breakout_Matches_2023 = {
    1: week1Breakout2023,
    2: week2Breakout2023,
    3: week3Breakout2023,
    4: week4Breakout2023,
    5: week5Breakout2023,
    6: week6Breakout2023,
    7: week7Breakout2023,
    8: week8Breakout2023,
    9: week9Breakout2023,
    10: week10Breakout2023,
    11: week11Breakout2023,
    12: week12Breakout2023,
    13: week13Breakout2023,
    14: week14Breakout2023,
    15: week15Breakout2023,
    16: week16Breakout2023,
    17: week17Breakout2023,
}

In [490]:
Breakout_Matches_2024 = {
    1: week1Breakout2024,
    2: week2Breakout2024,
    3: week3Breakout2024,
    4: week4Breakout2024,
    5: week5Breakout2024,
    6: week6Breakout2024,
    7: week7Breakout2024,
    8: week8Breakout2024,
    9: week9Breakout2024,
    10: week10Breakout2024,
    11: week11Breakout2024,
    12: week12Breakout2024,
    13: week13Breakout2024,
    14: week14Breakout2024,
    15: week15Breakout2024,
    16: week16Breakout2024,
    17: week17Breakout2024,
}

In [340]:
week17Breakout2024.Season

0      Playoff
1      Playoff
2      Playoff
3      Playoff
4      Playoff
        ...   
167    Playoff
168    Playoff
169    Playoff
170    Playoff
171    Playoff
Name: Season, Length: 172, dtype: object

## BreakoutConcat():

In [332]:
def BreakoutConcat(df_dict, week_num):
    # Filter out the dataframes corresponding to months up to and including the current month
    dataframes_to_concat = [df_dict[week] for week in range(1, week_num + 1)]
    
    # Concatenate the selected dataframes
    Weeks= pd.concat(dataframes_to_concat, axis=0)
    
    #Weeks['Total Wins'] = Weeks.groupby('Team')['Won'].cumsum()

    Weeks['Score YTD'] = Weeks.groupby('player')['points'].cumsum()
    Weeks['TotalYTD'] = Weeks.groupby('player')['points'].transform('sum')

    #Weeks['Opp YTD'] = Weeks.groupby(Weeks.index)['Opp'].cumsum()
    
    
    #Week_Pivot = Weeks.pivot(index = 'Team', columns = 'Week',values = 'Total Wins')
    
    return Weeks

## SeparateColumns()

In [69]:
def SeparateColumns(df):

    for col in df.iloc[:,3:12]:
        # Expand the dictionary into two columns: 'key' and 'value'
        expanded_df = df[col].apply(lambda x: pd.Series(x) if isinstance(x, dict) else pd.Series()).stack().reset_index()
        
        # Rename the new columns to reflect their origin
        expanded_df.columns = ['original_index', f'{col}_player', f'{col}_pts']
        
        # Merge the expanded data back to the original DataFrame
        df = df.drop(columns=[col]).merge(expanded_df, left_index=True, right_on='original_index', how='left')
        
        # Drop the 'original_index' as it's no longer needed
        df.drop(columns='original_index', inplace=True)
    df.loc['Totals'] = df.select_dtypes(include='number').sum()
    return df

In [70]:
def SeparateColumnsMatches(df):

    for col in df.iloc[:,0:9]:
        # Expand the dictionary into two columns: 'key' and 'value'
        expanded_df = df[col].apply(lambda x: pd.Series(x) if isinstance(x, dict) else pd.Series()).stack().reset_index()
        
        # Rename the new columns to reflect their origin
        expanded_df.columns = ['original_index', f'{col}_player', f'{col}_pts']
        
        # Merge the expanded data back to the original DataFrame
        df = df.drop(columns=[col]).merge(expanded_df, left_index=True, right_on='original_index', how='left')
        
        # Drop the 'original_index' as it's no longer needed
        df.drop(columns='original_index', inplace=True)
    df.loc['Totals'] = df.select_dtypes(include='number').sum()
    return df

## Breakouts, Bests

In [491]:
Breakout_2019 = BreakoutConcat(Breakout_Matches_2019,17)
Breakout_2020 = BreakoutConcat(Breakout_Matches_2020,17)
Breakout_2021 = BreakoutConcat(Breakout_Matches_2021,17)
Breakout_2022 = BreakoutConcat(Breakout_Matches_2022,17)
Breakout_2023 = BreakoutConcat(Breakout_Matches_2023,17)
Breakout_2024 = BreakoutConcat(Breakout_Matches_2024,17)

In [492]:
Breakout_2019['Year'] = 2019
Breakout_2020['Year'] = 2020
Breakout_2021['Year'] = 2021
Breakout_2022['Year'] = 2022
Breakout_2023['Year'] = 2023
Breakout_2024['Year'] = 2024

In [493]:
Breakout_2021.columns.value_counts()

team             1
interceptions    1
carries          1
dakota           1
pacr             1
                ..
home_rest        1
away_rest        1
ftn              1
espn             1
Year             1
Length: 117, dtype: int64

In [494]:
AllSeasonsBreakoutList = [Breakout_2019, Breakout_2020, Breakout_2021, Breakout_2022, Breakout_2023, Breakout_2024]
AllSeasonsBreakoutDF = pd.concat(AllSeasonsBreakoutList)

In [495]:
AllSeasonsPlayoffs = AllSeasonsBreakoutDF[AllSeasonsBreakoutDF.Season == 'Playoff']
AllSeasonsRegular = AllSeasonsBreakoutDF[AllSeasonsBreakoutDF.Season == 'Regular']

In [496]:
AllSeasonsPlayoffs

,team,matchup,player,points,starter,position,week_x,player_name,recent_teams,week_id,...,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr,Game_date_time,color,Score YTD,TotalYTD,Year,week_id_NFL
0,bgmaddox,5.0,Eric Ebron,0.00,0,TE,15,Eric Ebron,IND,IND-15,...,NaN,NaN,NaN,NaN,Monday 8 PM,mediumpurple,56.00,56.00,2019,NaN
1,bgmaddox,5.0,David Johnson,0.60,0,RB,15,David Johnson,ARI,ARI-15,...,NaN,0.0,0.700000,0.700000,Sunday 4 PM,mediumpurple,121.30,122.50,2019,NaN
2,bgmaddox,5.0,Raheem Mostert,9.40,1,RB,15,Raheem Mostert,SF,SF-15,...,0.144344,0.0,11.900000,12.900000,Sunday 4 PM,mediumpurple,72.60,103.70,2019,NaN
3,bgmaddox,5.0,Drew Brees,36.28,1,QB,15,Drew Brees,NO,NO-15,...,NaN,0.0,28.280001,28.280001,Monday 8 PM,mediumpurple,221.88,278.76,2019,NaN
4,bgmaddox,5.0,Jacoby Brissett,7.30,0,QB,15,Jacoby Brissett,IND,IND-15,...,NaN,0.0,7.300000,7.300000,Monday 8 PM,mediumpurple,225.84,243.18,2019,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166,InfiniteJesse,2.0,Javonte Williams,1.00,0,RB,17,Javonte Williams,DEN,DEN-17,...,0.096774,0.0,0.000000,2.000000,Saturday 4 PM,magenta,122.60,122.60,2024,DEN-17
167,InfiniteJesse,2.0,Christian Watson,0.00,0,WR,17,Christian Watson,GB,GB-17,...,NaN,NaN,NaN,NaN,Sunday 4 PM,magenta,47.20,47.20,2024,NaN
168,InfiniteJesse,2.0,De'Von Achane,4.80,1,RB,17,De'Von Achane,MIA,MIA-17,...,NaN,NaN,NaN,NaN,Sunday 4 PM,magenta,243.08,243.08,2024,NaN
169,InfiniteJesse,2.0,C.J. Stroud,6.10,0,QB,17,C.J. Stroud,HOU,HOU-17,...,NaN,0.0,6.100000,6.100000,Wednesday 4 PM,magenta,250.38,250.38,2024,HOU-17


In [497]:
Best_2019 = Breakout_2019[Breakout_2019['TotalYTD'] > 130]
Best_2020 = Breakout_2020[Breakout_2020['TotalYTD'] > 130]
Best_2021 = Breakout_2021[Breakout_2021['TotalYTD'] > 130]
Best_2022 = Breakout_2022[Breakout_2022['TotalYTD'] > 130]
Best_2023 = Breakout_2023[Breakout_2023['TotalYTD'] > 130]
Best_2024 = Breakout_2024[Breakout_2024['TotalYTD'] > 130]

In [498]:

Best_2024['player'].unique

<bound method Series.unique of 2          Derrick Henry
3           Alvin Kamara
8          D'Andre Swift
9            Jalen Hurts
10     Amon-Ra St. Brown
             ...        
159           Jake Bates
162          Cooper Kupp
164          CeeDee Lamb
168        De'Von Achane
169          C.J. Stroud
Name: player, Length: 1604, dtype: object>

In [499]:
player_groups = Best_2019.groupby('player')

In [500]:
Breakout_2024

,team,matchup,player,points,starter,position,week_x,player_name,recent_teams,week_id,...,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr,week_id_NFL,Game_date_time,color,Score YTD,TotalYTD,Year
0,bgmaddox,1.0,Kirk Cousins,8.2,0,QB,1,Kirk Cousins,ATL,ATL-1,...,NaN,0.0,6.2,6.2,ATL-1,Sunday 1 PM,mediumpurple,8.20,87.36,2024
1,bgmaddox,1.0,Devaughn Vele,7.9,0,WR,1,Devaughn Vele,DEN,DEN-1,...,0.330790,0.0,3.9,11.9,DEN-1,Sunday 4 PM,mediumpurple,7.90,10.00,2024
2,bgmaddox,1.0,Derrick Henry,10.6,1,RB,1,Derrick Henry,BAL,BAL-1,...,0.048783,0.0,10.6,10.6,BAL-1,Thursday 8 PM,mediumpurple,10.60,295.80,2024
3,bgmaddox,1.0,Alvin Kamara,19.5,1,RB,1,Alvin Kamara,NO,NO-1,...,0.255288,0.0,17.0,22.0,NO-1,Sunday 1 PM,mediumpurple,19.50,230.30,2024
4,bgmaddox,1.0,Younghoe Koo,3.4,1,K,1,Younghoe Koo,ATL,ATL-1,...,NaN,NaN,NaN,NaN,NaN,Sunday 1 PM,mediumpurple,3.40,123.30,2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166,InfiniteJesse,2.0,Javonte Williams,1.0,0,RB,17,Javonte Williams,DEN,DEN-17,...,0.096774,0.0,0.0,2.0,DEN-17,Saturday 4 PM,magenta,122.60,122.60,2024
167,InfiniteJesse,2.0,Christian Watson,0.0,0,WR,17,Christian Watson,GB,GB-17,...,NaN,NaN,NaN,NaN,NaN,Sunday 4 PM,magenta,47.20,47.20,2024
168,InfiniteJesse,2.0,De'Von Achane,4.8,1,RB,17,De'Von Achane,MIA,MIA-17,...,NaN,NaN,NaN,NaN,NaN,Sunday 4 PM,magenta,243.08,243.08,2024
169,InfiniteJesse,2.0,C.J. Stroud,6.1,0,QB,17,C.J. Stroud,HOU,HOU-17,...,NaN,0.0,6.1,6.1,HOU-17,Wednesday 4 PM,magenta,250.38,250.38,2024


In [501]:
RushingYardsperWeek = Breakout_2024.groupby('week_x')['rushing_yards'].sum()

# Weekly Matchup Dataframes

In [355]:
def WeeklyDataframe(json_data, week_num, year):
    # Create an empty dictionary to hold the DataFrame data
    df_dict = {}
    matchup_dict = {}

    # Iterate through each team and their data
    for team in json_data:
        roster_id = team["roster_id"]
        starters = team['starters']
        starters_points = team['starters_points']
        matchup_id = team['matchup_id']
        
        # Replace player IDs with player names
        starters_with_names = [player_names[player] for player in starters]
        
        # Combine players and their points into a list where each entry is a list [dictionary, matchup_id]
        df_dict[roster_id] = [{player: points} for player, points in zip(starters_with_names, starters_points)]
        matchup_dict[roster_id] = matchup_id

    # Create a DataFrame from the dictionary
    WeeklyDf = pd.DataFrame.from_dict(df_dict, orient='index')

    WeeklyDf['Matchup'] = WeeklyDf.index.map(matchup_dict)
    # Define a function to sum the values in the dictionaries in each row
    def sum_points(row):
        total = 0
        for entry in row:
            if isinstance(entry, dict):
                total += sum(entry.values())
        return total

    league_names = roster_ids[year]
    # Apply the function to each row to create the 'Total' column
    WeeklyDf['Total'] = WeeklyDf.apply(sum_points, axis=1)
    WeeklyDf = WeeklyDf.rename(index = league_names)
    
    # Step 1: Group by 'Matchup_ID' and get the maximum 'Total' for each group
    max_scores = WeeklyDf.groupby('Matchup')['Total'].transform('max')
   
   

    # Step 2: Create a new column 'Won' that checks if the team's 'Total' equals the max score
    WeeklyDf['Won'] = WeeklyDf['Total'] == max_scores

    # Step 3: Optional - Convert the boolean 'Won' column to 1 (win) and 0 (loss)
    WeeklyDf['Won'] = WeeklyDf['Won'].astype(int)
    
    
    WeeklyDf['Opp'] = np.where(WeeklyDf['Won'] == 1,WeeklyDf.groupby('Matchup')['Total'].transform('min'),WeeklyDf.groupby('Matchup')['Total'].transform('max'))
    
    WeeklyDf['Margin'] = WeeklyDf['Total'] - WeeklyDf['Opp']
    #WeeklyDf.loc[[WeeklyDf['Won']] == 1,'Opp Score'] = WeeklyDf.groupby('Matchup')['Total'].transform('max')
    #WeeklyDf.loc[[WeeklyDf['Won']] == 0,'Opp Score'] = WeeklyDf.groupby('Matchup')['Total'].transform('min')
    
    
    SeasonMultiplier = {2019:0, 2020:1, 2021:2, 2022:3, 2023:4, 2024:5}
    
    WeeklyDf = WeeklyDf.rename(columns=positions).sort_values('Matchup')
    WeeklyDf = WeeklyDf.reset_index().rename({'index':'Team'}, axis = 1)
    WeeklyDf['Week'] = week_num
    WeeklyDf['Season'] = "Regular" if week_num < 15 else "Playoff"
    WeeklyDf['Week Index'] = week_num + (14 * SeasonMultiplier[year])
    WeeklyDf['Year'] = year
    
    percent = WeeklyDf.groupby('Week')['Total'].sum()
    percent = percent.reset_index()
    WeekTotal = dict(zip(percent['Week'],percent['Total']))
    
    WeeklyDf['LeagueTotal'] = WeeklyDf['Week'].map(WeekTotal)
    WeeklyDf['PercentTotal'] = ((WeeklyDf['Total'] / WeeklyDf['LeagueTotal']) * 100).round(1)
    # Use groupby to assign the opposing team's name
    #WeeklyDf['Opp_team'] = WeeklyDf.groupby('Matchup')['Team'].transform(lambda x: x.shift(-1) if x.index[0] % 2 == 0 else x.shift(1))
    # Define a function to get the opponent team name for each group
    def get_opponent(teams):
        return teams[::-1]  # Reverse the order so each team gets the other team as opponent

    # Apply the function group-wise to assign opposing teams
    #WeeklyDf['Opp_team'] = WeeklyDf.groupby('Matchup')['Team'].transform(get_opponent)
    # Define a function to assign the opponent team name
    def assign_opponents(group):
        # Assuming there are exactly two teams per matchup
        teams = group['Team'].values
        if len(teams) == 2:  # For valid matchups with two teams
            group['Opp_team'] = [teams[1], teams[0]]  # Swap the teams
        return group

    # Apply the function to each matchup
    WeeklyDf = WeeklyDf.groupby('Matchup').apply(assign_opponents)
        
    WeeklyDfMatches= WeeklyDf.set_index(['Matchup','Team'])
    WeeklyDfNoMatches = WeeklyDf.set_index('Team')
    WeeklyDfClean = WeeklyDf.set_index('Team').drop(axis = 1,columns = ['Total','Won','Week','Opp','Margin'])
    return WeeklyDfMatches, WeeklyDfNoMatches, WeeklyDfClean



### Match Dictionaries

In [371]:
Matches_2019 = {}
Matches_2020 = {}
Matches_2021 = {}
Matches_2022 = {}
Matches_2023 = {}
Matches_2023 = {}
Matches_2024 = {}

In [372]:
def WeekMatch(week, week_json, year):
    WeekMatches, WeekNoMatches, WeekClean = WeeklyDataframe(week_json, week, year)
    if year == 2019:
        Matches_2019[week] = WeekNoMatches.reset_index()
    elif year == 2020:
        Matches_2020[week] = WeekNoMatches.reset_index() 
    elif year == 2021:
        Matches_2021[week] = WeekNoMatches.reset_index() 
    elif year == 2022:
        Matches_2022[week] = WeekNoMatches.reset_index() 
    elif year == 2023:
        Matches_2023[week] = WeekNoMatches.reset_index()    
    elif year == 2024:
        Matches_2024[week] = WeekNoMatches.reset_index()    
    return WeekMatches, WeekNoMatches, WeekClean

In [373]:
Week1Matches_2019, Week1NoMatches_2019, Week1Clean_2019 = WeekMatch(1,Week1_json_2019, 2019)
Week2Matches_2019, Week2NoMatches_2019, Week2Clean_2019 = WeekMatch(2,Week2_json_2019, 2019)
Week3Matches_2019, Week3NoMatches_2019, Week3Clean_2019 = WeekMatch(3,Week3_json_2019, 2019)
Week4Matches_2019, Week4NoMatches_2019, Week4Clean_2019 = WeekMatch(4,Week4_json_2019, 2019)
Week5Matches_2019, Week5NoMatches_2019, Week5Clean_2019 = WeekMatch(5,Week5_json_2019, 2019)
Week6Matches_2019, Week6NoMatches_2019, Week6Clean_2019 = WeekMatch(6,Week6_json_2019, 2019)
Week7Matches_2019, Week7NoMatches_2019, Week7Clean_2019 = WeekMatch(7,Week7_json_2019, 2019)
Week8Matches_2019, Week8NoMatches_2019, Week8Clean_2019 =  WeekMatch(8,Week8_json_2019, 2019)
Week9Matches_2019, Week9NoMatches_2019, Week9Clean_2019 = WeekMatch(9,Week9_json_2019, 2019)
Week10Matches_2019, Week10NoMatches_2019, Week10Clean_2019 = WeekMatch(10,Week10_json_2019, 2019)
Week11Matches_2019, Week11NoMatches_2019, Week11Clean_2019 = WeekMatch(11,Week11_json_2019, 2019)
Week12Matches_2019, Week12NoMatches_2019, Week12Clean_2019 = WeekMatch(12,Week12_json_2019, 2019)
Week13Matches_2019, Week13NoMatches_2019, Week13Clean_2019 = WeekMatch(13,Week13_json_2019, 2019)
Week14Matches_2019, Week14NoMatches_2019, Week14Clean_2019 = WeekMatch(14,Week14_json_2019, 2019)
Week15Matches_2019, Week15NoMatches_2019, Week15Clean_2019 = WeekMatch(15,Week15_json_2019, 2019)
Week16Matches_2019, Week16NoMatches_2019, Week16Clean_2019 = WeekMatch(16,Week16_json_2019, 2019)
Week17Matches_2019, Week17NoMatches_2019, Week17Clean_2019 = WeekMatch(17,Week17_json_2019, 2019)

/var/folders/xg/y9j6k4pn725g1jx4yx1dfv2m0000gn/T/ipykernel_20956/3839760998.py:88: FutureWarning:

Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)

/var/folders/xg/y9j6k4pn725g1jx4yx1dfv2m0000gn/T/ipykernel_20956/3839760998.py:88: FutureWarning:

Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)

/var/folders/xg/y9j6k4pn725g1jx4yx1dfv

In [374]:
Week1Matches_2020, Week1NoMatches_2020, Week1Clean_2020 = WeekMatch(1,Week1_json_2020, 2020)
Week2Matches_2020, Week2NoMatches_2020, Week2Clean_2020 = WeekMatch(2,Week2_json_2020, 2020)
Week3Matches_2020, Week3NoMatches_2020, Week3Clean_2020 = WeekMatch(3,Week3_json_2020, 2020)
Week4Matches_2020, Week4NoMatches_2020, Week4Clean_2020 = WeekMatch(4,Week4_json_2020, 2020)
Week5Matches_2020, Week5NoMatches_2020, Week5Clean_2020 = WeekMatch(5,Week5_json_2020, 2020)
Week6Matches_2020, Week6NoMatches_2020, Week6Clean_2020 = WeekMatch(6,Week6_json_2020, 2020)
Week7Matches_2020, Week7NoMatches_2020, Week7Clean_2020 = WeekMatch(7,Week7_json_2020, 2020)
Week8Matches_2020, Week8NoMatches_2020, Week8Clean_2020 =  WeekMatch(8,Week8_json_2020, 2020)
Week9Matches_2020, Week9NoMatches_2020, Week9Clean_2020 = WeekMatch(9,Week9_json_2020, 2020)
Week10Matches_2020, Week10NoMatches_2020, Week10Clean_2020 = WeekMatch(10,Week10_json_2020, 2020)
Week11Matches_2020, Week11NoMatches_2020, Week11Clean_2020 = WeekMatch(11,Week11_json_2020, 2020)
Week12Matches_2020, Week12NoMatches_2020, Week12Clean_2020 = WeekMatch(12,Week12_json_2020, 2020)
Week13Matches_2020, Week13NoMatches_2020, Week13Clean_2020 = WeekMatch(13,Week13_json_2020, 2020)
Week14Matches_2020, Week14NoMatches_2020, Week14Clean_2020 = WeekMatch(14,Week14_json_2020, 2020)
Week15Matches_2020, Week15NoMatches_2020, Week15Clean_2020 = WeekMatch(15,Week15_json_2020, 2020)
Week16Matches_2020, Week16NoMatches_2020, Week16Clean_2020 = WeekMatch(16,Week16_json_2020, 2020)
Week17Matches_2020, Week17NoMatches_2020, Week17Clean_2020 = WeekMatch(17,Week17_json_2020, 2020)




/var/folders/xg/y9j6k4pn725g1jx4yx1dfv2m0000gn/T/ipykernel_20956/3839760998.py:88: FutureWarning:

Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)

/var/folders/xg/y9j6k4pn725g1jx4yx1dfv2m0000gn/T/ipykernel_20956/3839760998.py:88: FutureWarning:

Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)

/var/folders/xg/y9j6k4pn725g1jx4yx1dfv

In [375]:
Week1Matches_2021, Week1NoMatches_2021, Week1Clean_2021 = WeekMatch(1,Week1_json_2021, 2021)
Week2Matches_2021, Week2NoMatches_2021, Week2Clean_2021 = WeekMatch(2,Week2_json_2021, 2021)
Week3Matches_2021, Week3NoMatches_2021, Week3Clean_2021 = WeekMatch(3,Week3_json_2021, 2021)
Week4Matches_2021, Week4NoMatches_2021, Week4Clean_2021 = WeekMatch(4,Week4_json_2021, 2021)
Week5Matches_2021, Week5NoMatches_2021, Week5Clean_2021 = WeekMatch(5,Week5_json_2021, 2021)
Week6Matches_2021, Week6NoMatches_2021, Week6Clean_2021 = WeekMatch(6,Week6_json_2021, 2021)
Week7Matches_2021, Week7NoMatches_2021, Week7Clean_2021 = WeekMatch(7,Week7_json_2021, 2021)
Week8Matches_2021, Week8NoMatches_2021, Week8Clean_2021 =  WeekMatch(8,Week8_json_2021, 2021)
Week9Matches_2021, Week9NoMatches_2021, Week9Clean_2021 = WeekMatch(9,Week9_json_2021, 2021)
Week10Matches_2021, Week10NoMatches_2021, Week10Clean_2021 = WeekMatch(10,Week10_json_2021, 2021)
Week11Matches_2021, Week11NoMatches_2021, Week11Clean_2021 = WeekMatch(11,Week11_json_2021, 2021)
Week12Matches_2021, Week12NoMatches_2021, Week12Clean_2021 = WeekMatch(12,Week12_json_2021, 2021)
Week13Matches_2021, Week13NoMatches_2021, Week13Clean_2021 = WeekMatch(13,Week13_json_2021, 2021)
Week14Matches_2021, Week14NoMatches_2021, Week14Clean_2021 = WeekMatch(14,Week14_json_2021, 2021)
Week15Matches_2021, Week15NoMatches_2021, Week15Clean_2021 = WeekMatch(15,Week15_json_2021, 2021)
Week16Matches_2021, Week16NoMatches_2021, Week16Clean_2021 = WeekMatch(16,Week16_json_2021, 2021)
Week17Matches_2021, Week17NoMatches_2021, Week17Clean_2021 = WeekMatch(17,Week17_json_2021, 2021)




/var/folders/xg/y9j6k4pn725g1jx4yx1dfv2m0000gn/T/ipykernel_20956/3839760998.py:88: FutureWarning:

Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)

/var/folders/xg/y9j6k4pn725g1jx4yx1dfv2m0000gn/T/ipykernel_20956/3839760998.py:88: FutureWarning:

Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)

/var/folders/xg/y9j6k4pn725g1jx4yx1dfv

In [376]:
Week1Matches_2022, Week1NoMatches_2022, Week1Clean_2022 = WeekMatch(1,Week1_json_2022, 2022)
Week2Matches_2022, Week2NoMatches_2022, Week2Clean_2022 = WeekMatch(2,Week2_json_2022, 2022)
Week3Matches_2022, Week3NoMatches_2022, Week3Clean_2022 = WeekMatch(3,Week3_json_2022, 2022)
Week4Matches_2022, Week4NoMatches_2022, Week4Clean_2022 = WeekMatch(4,Week4_json_2022, 2022)
Week5Matches_2022, Week5NoMatches_2022, Week5Clean_2022 = WeekMatch(5,Week5_json_2022, 2022)
Week6Matches_2022, Week6NoMatches_2022, Week6Clean_2022 = WeekMatch(6,Week6_json_2022, 2022)
Week7Matches_2022, Week7NoMatches_2022, Week7Clean_2022 = WeekMatch(7,Week7_json_2022, 2022)
Week8Matches_2022, Week8NoMatches_2022, Week8Clean_2022 =  WeekMatch(8,Week8_json_2022, 2022)
Week9Matches_2022, Week9NoMatches_2022, Week9Clean_2022 = WeekMatch(9,Week9_json_2022, 2022)
Week10Matches_2022, Week10NoMatches_2022, Week10Clean_2022 = WeekMatch(10,Week10_json_2022, 2022)
Week11Matches_2022, Week11NoMatches_2022, Week11Clean_2022 = WeekMatch(11,Week11_json_2022, 2022)
Week12Matches_2022, Week12NoMatches_2022, Week12Clean_2022 = WeekMatch(12,Week12_json_2022, 2022)
Week13Matches_2022, Week13NoMatches_2022, Week13Clean_2022 = WeekMatch(13,Week13_json_2022, 2022)
Week14Matches_2022, Week14NoMatches_2022, Week14Clean_2022 = WeekMatch(14,Week14_json_2022, 2022)
Week15Matches_2022, Week15NoMatches_2022, Week15Clean_2022 = WeekMatch(15,Week15_json_2022, 2022)
Week16Matches_2022, Week16NoMatches_2022, Week16Clean_2022 = WeekMatch(16,Week16_json_2022, 2022)
Week17Matches_2022, Week17NoMatches_2022, Week17Clean_2022 = WeekMatch(17,Week17_json_2022, 2022)




/var/folders/xg/y9j6k4pn725g1jx4yx1dfv2m0000gn/T/ipykernel_20956/3839760998.py:88: FutureWarning:

Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)

/var/folders/xg/y9j6k4pn725g1jx4yx1dfv2m0000gn/T/ipykernel_20956/3839760998.py:88: FutureWarning:

Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)

/var/folders/xg/y9j6k4pn725g1jx4yx1dfv

In [377]:
Week1Matches_2023, Week1NoMatches_2023, Week1Clean_2023 = WeekMatch(1,Week1_json_2023, 2023)
Week2Matches_2023, Week2NoMatches_2023, Week2Clean_2023 = WeekMatch(2,Week2_json_2023, 2023)
Week3Matches_2023, Week3NoMatches_2023, Week3Clean_2023 = WeekMatch(3,Week3_json_2023, 2023)
Week4Matches_2023, Week4NoMatches_2023, Week4Clean_2023 = WeekMatch(4,Week4_json_2023, 2023)
Week5Matches_2023, Week5NoMatches_2023, Week5Clean_2023 = WeekMatch(5,Week5_json_2023, 2023)
Week6Matches_2023, Week6NoMatches_2023, Week6Clean_2023 = WeekMatch(6,Week6_json_2023, 2023)
Week7Matches_2023, Week7NoMatches_2023, Week7Clean_2023 = WeekMatch(7,Week7_json_2023, 2023)
Week8Matches_2023, Week8NoMatches_2023, Week8Clean_2023 =  WeekMatch(8,Week8_json_2023, 2023)
Week9Matches_2023, Week9NoMatches_2023, Week9Clean_2023 = WeekMatch(9,Week9_json_2023, 2023)
Week10Matches_2023, Week10NoMatches_2023, Week10Clean_2023 = WeekMatch(10,Week10_json_2023, 2023)
Week11Matches_2023, Week11NoMatches_2023, Week11Clean_2023 = WeekMatch(11,Week11_json_2023, 2023)
Week12Matches_2023, Week12NoMatches_2023, Week12Clean_2023 = WeekMatch(12,Week12_json_2023, 2023)
Week13Matches_2023, Week13NoMatches_2023, Week13Clean_2023 = WeekMatch(13,Week13_json_2023, 2023)
Week14Matches_2023, Week14NoMatches_2023, Week14Clean_2023 = WeekMatch(14,Week14_json_2023, 2023)
Week15Matches_2023, Week15NoMatches_2023, Week15Clean_2023 = WeekMatch(15,Week15_json_2023, 2023)
Week16Matches_2023, Week16NoMatches_2023, Week16Clean_2023 = WeekMatch(16,Week16_json_2023, 2023)
Week17Matches_2023, Week17NoMatches_2023, Week17Clean_2023 = WeekMatch(17,Week17_json_2023, 2023)



/var/folders/xg/y9j6k4pn725g1jx4yx1dfv2m0000gn/T/ipykernel_20956/3839760998.py:88: FutureWarning:

Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)

/var/folders/xg/y9j6k4pn725g1jx4yx1dfv2m0000gn/T/ipykernel_20956/3839760998.py:88: FutureWarning:

Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)

/var/folders/xg/y9j6k4pn725g1jx4yx1dfv

In [502]:
Week1Matches_2024, Week1NoMatches_2024, Week1Clean_2024 = WeekMatch(1,Week1_json_2024, 2024)
Week2Matches_2024, Week2NoMatches_2024, Week2Clean_2024 = WeekMatch(2,Week2_json_2024, 2024)
Week3Matches_2024, Week3NoMatches_2024, Week3Clean_2024 = WeekMatch(3,Week3_json_2024, 2024)
Week4Matches_2024, Week4NoMatches_2024, Week4Clean_2024 = WeekMatch(4,Week4_json_2024, 2024)
Week5Matches_2024, Week5NoMatches_2024, Week5Clean_2024 = WeekMatch(5,Week5_json_2024, 2024)
Week6Matches_2024, Week6NoMatches_2024, Week6Clean_2024 = WeekMatch(6,Week6_json_2024, 2024)
Week7Matches_2024, Week7NoMatches_2024, Week7Clean_2024 = WeekMatch(7,Week7_json_2024, 2024)
Week8Matches_2024, Week8NoMatches_2024, Week8Clean_2024 = WeekMatch(8,Week8_json_2024, 2024)
Week9Matches_2024, Week9NoMatches_2024, Week9Clean_2024 = WeekMatch(9,Week9_json_2024, 2024)
Week10Matches_2024, Week10NoMatches_2024, Week10Clean_2024 = WeekMatch(10,Week10_json_2024, 2024)
Week11Matches_2024, Week11NoMatches_2024, Week11Clean_2024 = WeekMatch(11,Week11_json_2024, 2024)
Week12Matches_2024, Week12NoMatches_2024, Week12Clean_2024 = WeekMatch(12,Week12_json_2024, 2024)
Week13Matches_2024, Week13NoMatches_2024, Week13Clean_2024 = WeekMatch(13,Week13_json_2024, 2024)
Week14Matches_2024, Week14NoMatches_2024, Week14Clean_2024 = WeekMatch(14,Week14_json_2024, 2024)
Week15Matches_2024, Week15NoMatches_2024, Week15Clean_2024 = WeekMatch(15,Week15_json_2024, 2024)
Week16Matches_2024, Week16NoMatches_2024, Week16Clean_2024 = WeekMatch(16,Week16_json_2024, 2024)
Week17Matches_2024, Week17NoMatches_2024, Week17Clean_2024 = WeekMatch(17,Week17_json_2024, 2024)



/var/folders/xg/y9j6k4pn725g1jx4yx1dfv2m0000gn/T/ipykernel_20956/3839760998.py:88: FutureWarning:

Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)

/var/folders/xg/y9j6k4pn725g1jx4yx1dfv2m0000gn/T/ipykernel_20956/3839760998.py:88: FutureWarning:

Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)

/var/folders/xg/y9j6k4pn725g1jx4yx1dfv

In [503]:
Matches_2019DF = pd.concat(Matches_2019.values())
Matches_2020DF = pd.concat(Matches_2020.values())
Matches_2021DF = pd.concat(Matches_2021.values())
Matches_2022DF = pd.concat(Matches_2022.values())
Matches_2023DF = pd.concat(Matches_2023.values())
Matches_2024DF = pd.concat(Matches_2024.values())
AllMatchesDF = pd.concat([Matches_2019DF,Matches_2020DF,Matches_2021DF,Matches_2022DF,Matches_2023DF,Matches_2024DF])


In [504]:
AllMatchesDF['Abs Margin'] = abs(AllMatchesDF['Margin'].astype(float)).round(2)
AllMatchesDF['Margin'] = round(AllMatchesDF['Margin'],2)

In [505]:
AllMatchesDF_Regular = AllMatchesDF[AllMatchesDF.Season=='Regular']
AllMatchesDF_Playoff = AllMatchesDF[AllMatchesDF.Season=='Playoff']

In [506]:
AllMatchesDF_Playoff

,Team,QB,RB1,RB2,WR1,WR2,TE,WRT,K,DEF,...,Opp,Margin,Week,Season,Week Index,Year,LeagueTotal,PercentTotal,Opp_team,Abs Margin
0,eegrady,{'Matt Ryan': 23.1},{'Austin Ekeler': 9.6},{'Joe Mixon': 17.1},{'Davante Adams': 19.8},{'Robert Woods': 3.7},{'Jason Witten': 11.6},{'Leonard Fournette': 9.8},{'Austin Seibert': 6.0},{'KC': 10.0},...,158.48,-47.78,15,Playoff,15,2019,1390.20,8.0,BillyRayGonnaGetcha,47.78
1,BillyRayGonnaGetcha,{'Baker Mayfield': 20.98},{'Christian McCaffrey': 33.5},{'Mark Ingram': 23.1},{'Keenan Allen': 14.4},{'A.J. Brown': 21.4},{'Austin Hooper': 3.5},{'Chris Godwin': 14.6},{'Matt Gay': 8.0},{'BUF': 19.0},...,110.70,47.78,15,Playoff,15,2019,1390.20,11.4,eegrady,47.78
2,jlglover,{'Russell Wilson': 23.34},{'Aaron Jones': 17.1},{'Derrick Henry': 8.6},{'Emmanuel Sanders': 1.9},{'Michael Gallup': 1.1},{'Travis Kelce': 19.7},{'Nick Chubb': 22.3},{'Wil Lutz': 10.0},{'PHI': 8.0},...,98.98,13.06,15,Playoff,15,2019,1390.20,8.1,BMoreBallers88,13.06
3,BMoreBallers88,{'Lamar Jackson': 46.08},{'James Conner': 13.1},{'Patrick Laird': 6.4},{'Julian Edelman': 1.9},{'Stefon Diggs': 10.0},{'Hunter Henry': 0.9},{'Odell Beckham': 10.6},{'Justin Tucker': 3.0},{'SF': 7.0},...,112.04,-13.06,15,Playoff,15,2019,1390.20,7.1,jlglover,13.06
4,GurlyGirls,{'Ryan Tannehill': 28.16},{'Josh Jacobs': 11.9},{'Saquon Barkley': 28.3},{'Adam Thielen': 4.5},{'Cooper Kupp': 13.1},{'Darren Waller': 16.2},{'Jared Cook': 7.4},{'Robbie Gould': 11.0},{'BAL': 7.0},...,76.92,50.64,15,Playoff,15,2019,1390.20,9.2,JTizzzzle,50.64
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3,InfiniteJesse,{'Michael Penix': 11.22},{'Alexander Mattison': 4.3},{'De'Von Achane': 4.8},{'Cooper Kupp': 3.4},{'Keenan Allen': 5.0},{'Sam LaPorta': 15.4},{'Jayden Reed': 1.1},{'Jake Bates': 9.9},{'LV': 13.0},...,98.00,-29.88,17,Playoff,87,2024,1421.46,4.8,RossLikeSauce,29.88
4,bgmaddox,{'Joe Flacco': 19.3},{'Derrick Henry': 23.5},{'Bucky Irving': 21.0},{'Amon-Ra St. Brown': 16.0},{'Jameson Williams': 22.0},{'Trey McBride': 24.3},{'Darnell Mooney': 4.7},{'Tyler Bass': 8.9},{'SEA': 16.0},...,135.88,19.82,17,Playoff,87,2024,1421.46,11.0,RascalHazard,19.82
5,RascalHazard,{'Brock Purdy': 36.28},{'Najee Harris': 9.1},{'Raheem Blackshear': 2.0},{'DeVonta Smith': 27.0},{'Courtland Sutton': 14.0},{'Travis Kelce': 18.4},{'Garrett Wilson': 14.1},{'Brandon Aubrey': 1.0},{'BAL': 14.0},...,155.70,-19.82,17,Playoff,87,2024,1421.46,9.6,bgmaddox,19.82
6,jlglover,{'Josh Allen': 26.98},{'Travis Etienne': 8.0},{'Rico Dowdle': 9.3},{'DJ Moore': 9.2},{'Tank Dell': 0.0},{'Noah Gray': 1.1},{'Justin Jefferson': 13.2},{'Chris Boswell': 4.6},{'TB': 16.0},...,96.76,-8.38,17,Playoff,87,2024,1421.46,6.2,RReclam,8.38


In [507]:
TopPlayoffScores = AllMatchesDF_Playoff.sort_values('Total', ascending=False)[:10]
TopPlayoffScores['Names'] = TopPlayoffScores.Team + ' [W' + TopPlayoffScores.Week.astype(str) + ' ' + TopPlayoffScores.Year.astype(str)+ ']' + ' - ' + TopPlayoffScores.Total.round(1).astype(str) 
TopPlayoffScores['Year'] = TopPlayoffScores['Year'].astype(int)

In [508]:
figTopPlayoffScores = px.bar(TopPlayoffScores, y='Names', x='Total', template = 'plotly_dark',
                             color = 'Year', orientation='h', text = 'Names', title ='<b>Hightest Playoff Scores</b>',
                             #color_discrete_sequence=px.colors.qualitative.G10,
                             color_discrete_map={
                              2019:'magenta',
                              2020:'green',
                              2021:'blue',
                              2022:'red',
                              2023:'purple',
                              2024:'gold'   
                             }

                             )
figTopPlayoffScores.update_layout(height = 1200, width = 900,barcornerradius=13,)
figTopPlayoffScores.update_layout(yaxis={'categoryorder': 'total ascending'})
figTopPlayoffScores.update_coloraxes(showscale=False)
figTopPlayoffScores.update_layout(font_family="Courier New", title_font = dict(size=40))   
figTopPlayoffScores.update_layout(
    font=dict(
        family="Courier New, monospace",
        size=18,  # Set the font size here
        color="white"
    )
)

figTopPlayoffScores.update_layout(yaxis={'visible': False, 'showticklabels': False})

figTopPlayoffScores.show()

## Opponent Matchup Data

In [94]:
def OppWinPercentage(All_df, team, opp):
    OppTable = pd.pivot_table(All_df, values='Won',index='Team',columns='Opp_team',aggfunc='mean').round(2).fillna('')
    result = OppTable.loc[team,opp]
    return result

In [95]:
def OppWinPercentageTable(All_df):
    result = pd.pivot_table(All_df, values='Won',index='Team',columns='Opp_team',aggfunc='mean').round(2).fillna('')
    result = result[result.columns.intersection(roster_ids_2023.values())].reset_index()
    result = result[result['Team'].isin(roster_ids_2023.values())].set_index('Team')
    return result

In [96]:
roster_ids_2023.values()

dict_values(['bgmaddox', 'jlglover', 'JTizzzzle', 'RascalHazard', 'BMoreBallers88', 'eegrady', 'DirtyCommie', 'jhuntmadd', 'RReclam', 'sgmaddox', 'RossLikeSauce', 'InfiniteJesse'])

In [97]:
OppWinPercentageTable_Current = OppWinPercentageTable(AllMatchesDF)

In [98]:
OppWinPercentageTable_Current

Opp_team,BMoreBallers88,DirtyCommie,InfiniteJesse,JTizzzzle,RReclam,RascalHazard,RossLikeSauce,bgmaddox,eegrady,jhuntmadd,jlglover,sgmaddox
Team,,,,,,,,,,,,
BMoreBallers88,,0.67,0.5,0.43,0.67,0.57,0.86,0.43,0.29,0.33,0.78,0.75
DirtyCommie,0.33,,0.5,0.25,0.75,0.5,0.0,0.67,0.4,0.75,0.33,0.5
InfiniteJesse,0.5,0.5,,0.67,0.0,0.33,0.0,0.0,1.0,0.67,1.0,0.5
JTizzzzle,0.57,0.75,0.33,,0.14,0.5,0.5,0.43,0.25,0.71,0.43,0.33
RReclam,0.33,0.25,1.0,0.86,,0.5,0.5,0.4,0.5,0.43,0.4,0.4
RascalHazard,0.43,0.5,0.67,0.5,0.5,,0.33,0.62,0.86,0.29,0.67,0.0
RossLikeSauce,0.14,1.0,1.0,0.5,0.5,0.67,,0.43,0.33,0.25,0.5,0.33
bgmaddox,0.57,0.33,1.0,0.57,0.6,0.38,0.57,,0.62,0.0,0.56,0.0
eegrady,0.71,0.6,0.0,0.75,0.5,0.14,0.67,0.38,,0.67,0.22,0.6


In [99]:
figHeat = px.imshow(OppWinPercentageTable(AllMatchesDF), text_auto=True)
figHeat.show()

In [100]:
# Replace 'team1' and 'team2' with the actual team names you want to compare
team1 = 'bgmaddox'
team2 = 'BMoreBallers88'

# Pulling the values for the two teams from the pivot table
win_rate_team1 = OppWinPercentageTable_Current.loc[team1, team2]
win_rate_team2 = OppWinPercentageTable_Current.loc[team2, team1]

# Create a simple line graph
figLine = go.Figure()

# Add a line representing the win rates between the two teams
figLine.add_trace(go.Scatter(x=[team1, team2], 
                         y=[win_rate_team1, win_rate_team2],
                         mode='lines+markers',
                         name=f"{team1} vs {team2}"))

# Add title and labels
figLine.update_layout(title=f"Win Rates Between {team1} and {team2}",
                  xaxis_title="Teams",
                  yaxis_title="Mean Wins")

figLine.show()


# Weekly Graphs

## Weekly Matchup Graph

In [536]:
def WeeklyGraph(MatchDF, Week):
    
    #points = MatchDF.drop(axis = 1,columns = ['Total','Won','Week','Opp','Margin']).applymap(lambda if isinstance(x,dict): float(list(x.values())[0]))
    points = MatchDF.applymap(lambda x: float(list(x.values())[0]) if isinstance(x, dict) else x)
    points = points.round(2).reset_index()
    
    fig1 = px.bar(points, y='Team',x=position_list,template = 'plotly_dark',color = "Matchup", barmode='group',text_auto=True, title = f'Week {Week} Matchups', orientation='h')
    
    #Update the layout to hide the legend:
    fig1.update(layout_coloraxis_showscale=False)
    
    fig1.update_layout(barcornerradius=13)
    # Adjust the figure size
    fig1.update_layout(width=800, height=1200)
    fig1.update_layout(title=dict(
            font=dict(
                size=50,
                family="Courier New")))  # Set the width and height in pixels
    fig1.update_traces(textfont_size=12, textangle=0, cliponaxis=True, textposition = 'inside', textfont=dict(weight='bold',family='Courier New',  # Font family
            size=12   # Font color
        ))
    
    # Customize the x-axis labels
    fig1.update_xaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=16,         # Font size
            color='white'    # Font color
        ),
        title = None
    )

    # Customize the y-axis labels
    fig1.update_yaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=18,         # Font size
            color='white'    # Font color
        ),
        title=None
    )

    fig1.add_annotation(
    text="VS.",
    xref="paper", yref="paper",
    x=-.1, y=.93, # Position relative to figure (right side, middle)
    showarrow=False,
    font=dict(
        family="Courier New, monospace",
        size=15,
        color="magenta",
        weight ='bold'
    ))

    fig1.add_annotation(
    text="VS.",
    xref="paper", yref="paper",
    x=-.1, y=.76, # Position relative to figure (right side, middle)
    showarrow=False,
    font=dict(
        family="Courier New, monospace",
        size=15,
        color="magenta"
    )
    )

    fig1.add_annotation(
    text="VS.",
    xref="paper", yref="paper",
    x=-.1, y=.58, # Position relative to figure (right side, middle)
    showarrow=False,
    font=dict(
        family="Courier New, monospace",
        size=15,
        color="magenta"
    )
    )

    fig1.add_annotation(
    text="VS.",
    xref="paper", yref="paper",
    x=-.1, y=.41, # Position relative to figure (right side, middle)
    showarrow=False,
    font=dict(
        family="Courier New, monospace",
        size=15,
        color="magenta"
    )
    )

    fig1.add_annotation(
    text="VS.",
    xref="paper", yref="paper",
    x=-.1, y=.24, # Position relative to figure (right side, middle)
    showarrow=False,
    font=dict(
        family="Courier New, monospace",
        size=15,
        color="magenta"
    )
    )

    fig1.add_annotation(
    text="VS.",
    xref="paper", yref="paper",
    x=-.1, y=.07, # Position relative to figure (right side, middle)
    showarrow=False,
    font=dict(
        family="Courier New, monospace",
        size=15,
        color="magenta"
    )
    )

    fig1.add_annotation(
    text="Bar Order: QB ------------> DEF",
    xref="paper", yref="paper",
    x=0.05, y=-.06, # Position relative to figure (right side, middle)
    showarrow=False,
    font=dict(
        family="Courier New, monospace",
        size=15,
        color="white"
    )
    )
    fig1.update_traces(marker_line_width=2,marker_line_color='black')
    
    fig1.update_traces(insidetextanchor= 'middle')
    fig1.update_layout(
        uniformtext_minsize=12,
        uniformtext_mode='hide'
        )
    
    fig1.show()
    return fig1

In [537]:
Week12Graph = WeeklyGraph(Week16NoMatches_2024, 17)

## WeeklyWins() & WeeklyWinsGraph()

In [103]:
def WeeklyWins(df_dict, week_num):
    # Filter out the dataframes corresponding to months up to and including the current month
    dataframes_to_concat = [df_dict[week] for week in range(1, week_num + 1)]
    
    # Concatenate the selected dataframes
    Weeks= pd.concat(dataframes_to_concat, axis=0)
    Weeks['Total Wins'] = Weeks.groupby('Team')['Won'].cumsum()

    Weeks['Score YTD'] = Weeks.groupby('Team')['Total'].cumsum()

    Weeks['Opp YTD'] = Weeks.groupby('Team')['Opp'].cumsum()
    
    ConcatinatedWeeks = Weeks
    
    Week_Pivot = Weeks.pivot(index = 'Team', columns = 'Week',values = 'Total Wins')
    
    return Week_Pivot, ConcatinatedWeeks

In [538]:
WeekPivot,WeekWins = WeeklyWins(Matches_2024,17)

In [539]:
WonGraph = Matches_2024DF.pivot(columns = 'Week',index='Team',values = 'Won')

In [540]:
WonGraph = WonGraph.replace(0,'L').replace(1,'W')

In [541]:
last_five_cols = WonGraph.columns[-5:]
WonGraph['Trend'] = WonGraph[last_five_cols].apply(lambda row: ' '.join(row.astype(str)), axis=1)



In [108]:
WonGraph = WonGraph.reset_index()

In [109]:
RecentTrend = dict(zip(WonGraph.Team, WonGraph.Trend))

In [110]:
RecentTrend

{'BMoreBallers88': 'L L W L W',
 'DirtyCommie': 'W W W W W',
 'InfiniteJesse': 'W W W L W',
 'JTizzzzle': 'L W L W L',
 'RReclam': 'L L L L L',
 'RascalHazard': 'W W W L W',
 'RossLikeSauce': 'W L W W W',
 'bgmaddox': 'L L W L L',
 'eegrady': 'W L L W W',
 'jhuntmadd': 'W W L W L',
 'jlglover': 'L W L L L',
 'sgmaddox': 'L L L W L'}

In [111]:
WeekPivot

Week,1,2,3,4,5,6,7,8,9,10,11,12
Team,,,,,,,,,,,,
BMoreBallers88,1,1,2,2,2,3,4,4,4,5,5,6
DirtyCommie,1,1,2,2,3,3,4,5,6,7,8,9
InfiniteJesse,1,1,1,2,2,2,2,3,4,5,5,6
JTizzzzle,0,0,1,1,2,3,3,3,4,4,5,5
RReclam,0,1,2,2,3,3,3,3,3,3,3,3
RascalHazard,1,2,2,2,3,3,3,4,5,6,6,7
RossLikeSauce,0,0,0,1,2,3,4,5,5,6,7,8
bgmaddox,0,1,1,2,3,4,5,5,5,6,6,6
eegrady,0,1,2,3,3,3,4,5,5,5,6,7


In [112]:
WeekPivot.sort_values(1)

Week,1,2,3,4,5,6,7,8,9,10,11,12
Team,,,,,,,,,,,,
JTizzzzle,0,0,1,1,2,3,3,3,4,4,5,5
RReclam,0,1,2,2,3,3,3,3,3,3,3,3
RossLikeSauce,0,0,0,1,2,3,4,5,5,6,7,8
bgmaddox,0,1,1,2,3,4,5,5,5,6,6,6
eegrady,0,1,2,3,3,3,4,5,5,5,6,7
jhuntmadd,0,0,0,1,1,2,3,4,5,5,6,6
BMoreBallers88,1,1,2,2,2,3,4,4,4,5,5,6
DirtyCommie,1,1,2,2,3,3,4,5,6,7,8,9
InfiniteJesse,1,1,1,2,2,2,2,3,4,5,5,6


In [113]:
WeekWins

,Team,QB,RB1,RB2,WR1,WR2,TE,WRT,K,DEF,...,Margin,Week,Week Index,Year,LeagueTotal,PercentTotal,Opp_team,Total Wins,Score YTD,Opp YTD
0,bgmaddox,{'Jalen Hurts': 19.42},{'Derrick Henry': 10.6},{'Alvin Kamara': 19.5},{'Amon-Ra St. Brown': 2.8},{'Calvin Ridley': 6.5},{'Trey McBride': 5.5},{'D'Andre Swift': 5.0},{'Younghoe Koo': 3.4},{'DAL': 19.0},...,-27.30,1,71,2024,1241.44,7.4,BMoreBallers88,0,91.72,119.02
1,BMoreBallers88,{'Tua Tagovailoa': 20.62},{'Jonathan Taylor': 10.8},{'Travis Etienne': 10.9},{'Deebo Samuel': 16.2},{'Amari Cooper': 2.6},{'Mark Andrews': 2.4},{'Aaron Jones': 17.9},{'Jake Moody': 26.6},{'SEA': 11.0},...,27.30,1,71,2024,1241.44,9.6,bgmaddox,1,119.02,91.72
2,jlglover,{'Josh Allen': 35.18},{'Tyjae Spears': 5.2},{'Ezekiel Elliott': 11.9},{'Justin Jefferson': 13.9},{'Rashee Rice': 13.8},{'Cole Kmet': 0.9},{'DJ Moore': 7.5},{'Cameron Dicker': 11.2},{'CHI': 26.0},...,3.94,1,71,2024,1241.44,10.1,RossLikeSauce,1,125.58,121.64
3,RossLikeSauce,{'Patrick Mahomes': 16.64},{'Joe Mixon': 25.3},{'James Conner': 17.8},{'Ja'Marr Chase': 9.2},{'Puka Nacua': 6.2},{'Jake Ferguson': 3.0},{'Chris Godwin': 18.3},{'Ka'imi Fairbairn': 17.2},{'SF': 8.0},...,-3.94,1,71,2024,1241.44,9.8,jlglover,0,121.64,125.58
4,JTizzzzle,{'Joe Burrow': 7.06},{'Bijan Robinson': 13.6},{'Kyren Williams': 12.9},{'Stefon Diggs': 18.9},{'DK Metcalf': 4.4},{'Luke Musgrave': 0.0},{'Zay Flowers': 8.1},{'Jake Elliott': 9.9},{'PHI': 0.0},...,-40.80,1,71,2024,1241.44,6.0,InfiniteJesse,0,74.86,115.66
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7,RReclam,{'Bo Nix': 23.42},{'Christian McCaffrey': 6.3},{'Ameer Abdullah': 15.0},{'Nico Collins': 17.7},{'Josh Downs': 4.36},{'Hunter Henry': 6.9},{'Calvin Ridley': 11.8},{'Chase McLaughlin': 4.3},{'DEN': 9.0},...,-19.14,12,82,2024,1298.88,7.6,RascalHazard,3,1263.44,1461.68
8,DirtyCommie,{'Lamar Jackson': 26.58},{'D'Andre Swift': 8.0},{'Rachaad White': 11.2},{'Terry McLaurin': 18.7},{'Tyreek Hill': 7.3},{'David Njoku': 3.4},{'Jerry Jeudy': 11.5},{'Austin Seibert': 7.2},{'KC': 2.0},...,7.62,12,82,2024,1298.88,7.4,jhuntmadd,9,1358.36,1129.34
9,jhuntmadd,{'Jared Goff': 9.76},{'Trey Benson': 1.8},{'Kenneth Walker': 11.3},{'Mike Evans': 9.3},{'DeAndre Hopkins': 12.0},{'Brock Bowers': 5.8},{'Jaxon Smith-Njigba': 16.7},{'Cameron Dicker': 16.6},{'MIN': 5.0},...,-7.62,12,82,2024,1298.88,6.8,DirtyCommie,6,1311.40,1352.32
10,eegrady,{'Jayden Daniels': 34.4},{'Jahmyr Gibbs': 23.4},{'Saquon Barkley': 44.2},{'Jakobi Meyers': 17.1},{'Michael Pittman': 12.6},{'T.J. Hockenson': 14.9},{'George Kittle': 17.2},{'Jason Sanders': 11.2},{'Missing': 0.0},...,65.98,12,82,2024,1298.88,13.5,sgmaddox,7,1319.60,1240.36


## Seasonal Snake Graph

In [114]:
def WeeklyWinsGraph(df,week):
    fig2 = px.line(df,x='Week',y='Total Wins', color = 'Team',template='plotly_dark',line_shape = 'spline')
    fig2.update_xaxes(dtick=1,
                    tickfont=dict(
            family='Courier New',  # Font family
            size=18,         # Font size
            color='white'    # Font color
        ))
    fig2.update_yaxes(dtick=1)
    fig2.update_layout(width=900, height=600)
    # Adjust the thickness of the lines
    fig2.update_traces(line=dict(width=4))  # Set the line width (e.g., 3 pixels)
    fig2.update_layout(legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
        title = ''
    ))

    
    # Customize the y-axis labels
    fig2.update_yaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=20,         # Font size
            color='white'    # Font color
        )
    )

    fig2.update_layout(uniformtext_minsize=10, uniformtext_mode='hide')

    # Update x-axis and y-axis titles with font customization
    fig2.update_layout(
        xaxis_title="Week",  # Set x-axis title
        yaxis_title="Wins",   # Set y-axis title
        xaxis=dict(
            title_font=dict(
                family="Courier New",  # Set font family for x-axis title
                size=20          # Set font size for x-axis title
            )
        ),
        yaxis=dict(
            title_font=dict(
                family="Courier New",  # Set font family for y-axis title
                size=20          # Set font size for y-axis title
            )
        )
    )
    
    # Determine final wins and sort by them
    final_scores = [(d.name, d.x[-1], d.y[-1], d.line.color) for d in fig2.data]
    final_scores.sort(key=lambda x: -x[2])  # Sort by final win count, descending

    # Define a list of potential text positions to avoid overlap
    text_positions = ['middle right','top right', 'bottom right', 'top left', 'bottom left', 'middle left','top center','bottom center']

    previous_score = None
    position_index = 0

    for team_name, x_final, y_final, color in final_scores:
        if y_final == previous_score:
            # Cycle through text positions to avoid overlap if scores are the same
            position_index = (position_index + 1) % len(text_positions)
        else:
            position_index = 0  # Reset position index when score changes

        text_position = text_positions[position_index]
        previous_score = y_final

        fig2.add_scatter(
            x=[x_final], y=[y_final],
            mode='markers+text',
            text=[team_name],
            textfont=dict(family="Courier New",color=color, size=14, weight = 'bold'),
            textposition=text_position,
            marker=dict(color=color, size=12),
            showlegend=False
            )    
        fig2.update_layout(
        margin=dict(l=50, r=100, t=50, b=50)  # Set left, right, top, bottom padding within the plot area
            )
    fig2.update_layout(xaxis=dict(range=[0, week + 1])) 
    fig2.update_layout(font_family="Courier New")
    return fig2

In [115]:
Season2024Graph = WeeklyWinsGraph(WeekWins,13)

In [116]:
Season2024Graph.show()

In [117]:
def WeeklyWinsGraphBreakout(df,week):
    fig2 = px.line(df,x='Week',y='Total Wins', color = 'Team',template='plotly_dark',line_shape = 'spline', facet_col='Team',facet_col_wrap=3, title = 'Weekly Wins: Breakout')
    fig2.update_xaxes(dtick=1,
                    tickfont=dict(
            family='Courier New',  # Font family
            size=12,         # Font size
            #rotation = 0,
            color='white'    # Font color
        ))
    fig2.update_yaxes(dtick=1)
    fig2.update_layout(width=900, height=900, showlegend=False)
    # Adjust the thickness of the lines
    fig2.update_traces(line=dict(width=4))  # Set the line width (e.g., 3 pixels)
    fig2.update_layout(legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
        title = ''
    ))

    
    # Customize the y-axis labels
    fig2.update_yaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=12,         # Font size
            color='white'    # Font color
        )
    )

    fig2.update_layout(uniformtext_minsize=10, uniformtext_mode='hide')

    # Update x-axis and y-axis titles with font customization
    fig2.update_layout(
        xaxis_title="",  
        yaxis_title="",  
        xaxis=dict(
            title_font=dict(
                family="Courier New",  # Set font family for x-axis title
                size=20          # Set font size for x-axis title
            )
        ),
        yaxis=dict(
            title_font=dict(
                family="Courier New",  # Set font family for y-axis title
                size=20          # Set font size for y-axis title
            )
        )
    )
    fig2.update_layout(
        xaxis2_title="",  
        yaxis2_title="",  
        xaxis2=dict(
            title_font=dict(
                family="Courier New",  # Set font family for x-axis title
                size=20          # Set font size for x-axis title
            )
        ),
        yaxis2=dict(
            title_font=dict(
                family="Courier New",  # Set font family for y-axis title
                size=20          # Set font size for y-axis title
            )
        )
    )
    fig2.update_layout(
        xaxis3_title="",  
        yaxis3_title="",  
        xaxis3=dict(
            title_font=dict(
                family="Courier New",  # Set font family for x-axis title
                size=20          # Set font size for x-axis title
            )
        ),
        yaxis3=dict(
            title_font=dict(
                family="Courier New",  # Set font family for y-axis title
                size=20          # Set font size for y-axis title
            )
        )
    )
    
    # Determine final wins and sort by them
    final_scores = [(d.name, d.x[-1], d.y[-1], d.line.color) for d in fig2.data]
    final_scores.sort(key=lambda x: -x[2])  # Sort by final win count, descending

    # Define a list of potential text positions to avoid overlap
    text_positions = ['middle right','top right', 'bottom right', 'top left', 'bottom left', 'middle left','top center','bottom center']

    previous_score = None
    position_index = 0
    fig2.for_each_xaxis(lambda xaxis: xaxis.update(showticklabels=True,showgrid=False, dtick = 5))
    fig2.for_each_yaxis(lambda yaxis: yaxis.update(showticklabels=True,showgrid=True, dtick=4))

    fig2.update_layout(xaxis=dict(range=[0, week + 1])) 
    fig2.update_layout(yaxis=dict(range=[0, 9])) 
    fig2.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    fig2.update_layout(
    font=dict(
        size=18,  # Set the font size here
    )
)
    fig2.update_layout(font_family="Courier New")
    fig2.update_layout( title_font = dict(size=35, weight = 'bold'))
    fig2.update_yaxes(title="", col =1)  # Set a new title for all y-axes
    '''
    fig2.update_layout(
    annotations=[
        dict(
            x=0.5,
            y=-0.07,
            xref="paper",
            yref="paper",
            text="Week",
            showarrow=False,
            font=dict(size=30),
        ),
        dict(
            y=0.5,
            x=-0.06,
            xref="paper",
            yref="paper",
            text="Wins",
            textangle = 270,
            showarrow=False,
            font=dict(size=30),
        )
    ]
    )
    '''
    return fig2

In [118]:
LatestWeek = WeekWins[WeekWins['Week'] == 11]
LatestWeek_dict = dict(zip(LatestWeek['Team'],LatestWeek['Total Wins']))
WeekWins['LatestWin'] = WeekWins['Team'].map(LatestWeek_dict)

In [119]:
Season2024GraphBreakout = WeeklyWinsGraphBreakout(WeekWins,12)
Season2024GraphBreakout.show()

In [120]:
WeekWins

,Team,QB,RB1,RB2,WR1,WR2,TE,WRT,K,DEF,...,Week,Week Index,Year,LeagueTotal,PercentTotal,Opp_team,Total Wins,Score YTD,Opp YTD,LatestWin
0,bgmaddox,{'Jalen Hurts': 19.42},{'Derrick Henry': 10.6},{'Alvin Kamara': 19.5},{'Amon-Ra St. Brown': 2.8},{'Calvin Ridley': 6.5},{'Trey McBride': 5.5},{'D'Andre Swift': 5.0},{'Younghoe Koo': 3.4},{'DAL': 19.0},...,1,71,2024,1241.44,7.4,BMoreBallers88,0,91.72,119.02,6
1,BMoreBallers88,{'Tua Tagovailoa': 20.62},{'Jonathan Taylor': 10.8},{'Travis Etienne': 10.9},{'Deebo Samuel': 16.2},{'Amari Cooper': 2.6},{'Mark Andrews': 2.4},{'Aaron Jones': 17.9},{'Jake Moody': 26.6},{'SEA': 11.0},...,1,71,2024,1241.44,9.6,bgmaddox,1,119.02,91.72,5
2,jlglover,{'Josh Allen': 35.18},{'Tyjae Spears': 5.2},{'Ezekiel Elliott': 11.9},{'Justin Jefferson': 13.9},{'Rashee Rice': 13.8},{'Cole Kmet': 0.9},{'DJ Moore': 7.5},{'Cameron Dicker': 11.2},{'CHI': 26.0},...,1,71,2024,1241.44,10.1,RossLikeSauce,1,125.58,121.64,4
3,RossLikeSauce,{'Patrick Mahomes': 16.64},{'Joe Mixon': 25.3},{'James Conner': 17.8},{'Ja'Marr Chase': 9.2},{'Puka Nacua': 6.2},{'Jake Ferguson': 3.0},{'Chris Godwin': 18.3},{'Ka'imi Fairbairn': 17.2},{'SF': 8.0},...,1,71,2024,1241.44,9.8,jlglover,0,121.64,125.58,7
4,JTizzzzle,{'Joe Burrow': 7.06},{'Bijan Robinson': 13.6},{'Kyren Williams': 12.9},{'Stefon Diggs': 18.9},{'DK Metcalf': 4.4},{'Luke Musgrave': 0.0},{'Zay Flowers': 8.1},{'Jake Elliott': 9.9},{'PHI': 0.0},...,1,71,2024,1241.44,6.0,InfiniteJesse,0,74.86,115.66,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7,RReclam,{'Bo Nix': 23.42},{'Christian McCaffrey': 6.3},{'Ameer Abdullah': 15.0},{'Nico Collins': 17.7},{'Josh Downs': 4.36},{'Hunter Henry': 6.9},{'Calvin Ridley': 11.8},{'Chase McLaughlin': 4.3},{'DEN': 9.0},...,12,82,2024,1298.88,7.6,RascalHazard,3,1263.44,1461.68,3
8,DirtyCommie,{'Lamar Jackson': 26.58},{'D'Andre Swift': 8.0},{'Rachaad White': 11.2},{'Terry McLaurin': 18.7},{'Tyreek Hill': 7.3},{'David Njoku': 3.4},{'Jerry Jeudy': 11.5},{'Austin Seibert': 7.2},{'KC': 2.0},...,12,82,2024,1298.88,7.4,jhuntmadd,9,1358.36,1129.34,8
9,jhuntmadd,{'Jared Goff': 9.76},{'Trey Benson': 1.8},{'Kenneth Walker': 11.3},{'Mike Evans': 9.3},{'DeAndre Hopkins': 12.0},{'Brock Bowers': 5.8},{'Jaxon Smith-Njigba': 16.7},{'Cameron Dicker': 16.6},{'MIN': 5.0},...,12,82,2024,1298.88,6.8,DirtyCommie,6,1311.40,1352.32,6
10,eegrady,{'Jayden Daniels': 34.4},{'Jahmyr Gibbs': 23.4},{'Saquon Barkley': 44.2},{'Jakobi Meyers': 17.1},{'Michael Pittman': 12.6},{'T.J. Hockenson': 14.9},{'George Kittle': 17.2},{'Jason Sanders': 11.2},{'Missing': 0.0},...,12,82,2024,1298.88,13.5,sgmaddox,7,1319.60,1240.36,6


In [121]:
AllMatchesDF['Year-Week'] = AllMatchesDF['Year'].astype(str) + ' - ' + AllMatchesDF['Week'].astype(str).str.zfill(2)
    
AllMatchesDF['Total Wins'] = AllMatchesDF.sort_values(['Year','Week']).groupby('Team')['Won'].cumsum()

AllMatchesDF['Score YTD'] = AllMatchesDF.sort_values(['Year','Week']).groupby('Team')['Total'].cumsum()

AllMatchesDF['Opp YTD'] = AllMatchesDF.sort_values(['Year','Week']).groupby('Team')['Opp'].cumsum()
    
AllMatchesDF['Week Count'] = AllMatchesDF.sort_values(['Year','Week']).groupby('Team').cumcount()
    
    
    
    
        
    

In [122]:
AllMatchesDF_Pivot = AllMatchesDF.pivot(index = 'Team', columns = 'Year-Week',values = 'Total Wins')

In [123]:
AllMatchesDF_Pivot

Year-Week,2019 - 01,2019 - 02,2019 - 03,2019 - 04,2019 - 05,2019 - 06,2019 - 07,2019 - 08,2019 - 09,2019 - 10,...,2024 - 03,2024 - 04,2024 - 05,2024 - 06,2024 - 07,2024 - 08,2024 - 09,2024 - 10,2024 - 11,2024 - 12
Team,,,,,,,,,,,,,,,,,,,,,
BMoreBallers88,0.0,1.0,1.0,2.0,2.0,3.0,4.0,5.0,5.0,6.0,...,45.0,45.0,45.0,46.0,47.0,47.0,47.0,48.0,48.0,49.0
BillyRayGonnaGetcha,1.0,2.0,3.0,4.0,5.0,5.0,5.0,6.0,7.0,7.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DirtyCommie,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,12.0,12.0,13.0,13.0,14.0,15.0,16.0,17.0,18.0,19.0
GurlyGirls,0.0,1.0,1.0,1.0,2.0,2.0,3.0,3.0,4.0,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
InfiniteJesse,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,7.0,8.0,8.0,8.0,8.0,9.0,10.0,11.0,11.0,12.0
JTizzzzle,1.0,1.0,2.0,3.0,3.0,3.0,4.0,5.0,6.0,6.0,...,31.0,31.0,32.0,33.0,33.0,33.0,34.0,34.0,35.0,35.0
Just_Here_For_The_Snacks,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
RReclam,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,34.0,34.0,35.0,35.0,35.0,35.0,35.0,35.0,35.0,35.0
RascalHazard,0.0,0.0,1.0,1.0,1.0,2.0,2.0,2.0,2.0,3.0,...,39.0,39.0,40.0,40.0,40.0,41.0,42.0,43.0,43.0,44.0


In [124]:
AllMatchesDF.head()

,Team,QB,RB1,RB2,WR1,WR2,TE,WRT,K,DEF,...,Year,LeagueTotal,PercentTotal,Opp_team,Abs Margin,Year-Week,Total Wins,Score YTD,Opp YTD,Week Count
0,bgmaddox,{'Drew Brees': 24.8},{'David Johnson': 22.7},{'Sony Michel': 1.4},{'JuJu Smith-Schuster': 10.8},{'DJ Moore': 8.1},{'David Njoku': 11.7},{'Calvin Ridley': 14.4},{'Harrison Butker': 17.0},{'BAL': 15.0},...,2019,1350.32,9.3,jlglover,12.50,2019 - 01,1,125.90,113.40,0
1,jlglover,{'Russell Wilson': 20.6},{'Nick Chubb': 10.0},{'Aaron Jones': 3.4},{'T.Y. Hilton': 24.7},{'Marvin Jones': 8.0},{'Travis Kelce': 10.3},{'Derrick Henry': 28.4},{'Mason Crosby': 4.0},{'NO': 4.0},...,2019,1350.32,8.4,bgmaddox,12.50,2019 - 01,0,113.40,125.90,0
2,BMoreBallers88,{'Lamar Jackson': 43.56},{'James Conner': 8.5},{'Tevin Coleman': 6.6},{'Odell Beckham': 10.6},{'Tyler Boyd': 10.3},{'Hunter Henry': 8.0},{'Stefon Diggs': 4.7},{'Robbie Gould': 11.0},{'LAC': 1.0},...,2019,1350.32,7.7,YouthPastor,10.36,2019 - 01,0,104.26,114.62,0
3,YouthPastor,{'Carson Wentz': 31.02},{'Damien Williams': 15.5},{'Devonta Freeman': 1.6},{'DeAndre Hopkins': 27.1},{'Tyler Lockett': 10.9},{'Darren Waller': 10.5},{'Duke Johnson': 11.0},{'Matt Bryant': 0.0},{'NE': 7.0},...,2019,1350.32,8.5,BMoreBallers88,10.36,2019 - 01,1,114.62,104.26,0
4,GurlyGirls,{'Cam Newton': 4.36},{'Saquon Barkley': 14.9},{'James White': 10.7},{'Adam Thielen': 11.8},{'Cooper Kupp': 8.1},{'Vance McDonald': 5.0},{'Josh Jacobs': 23.8},{'Brett Maher': 5.0},{'SEA': 11.0},...,2019,1350.32,7.0,SweetDizzzzzle,30.86,2019 - 01,0,94.66,125.52,0


## Schedule Graph

In [125]:
time_dict = {'Monday 7 PM': 'MON', 'Sunday 9 AM':'SUN 9AM', 'Sunday 1 PM':'SUN 1PM', 'Sunday 4 PM': 'SUN 4PM', 'Sunday 8 PM':'SUN 8PM',
              'Thursday 8 PM':'THU 8PM', 'Monday 8 PM':'MON 8PM','Thursday 4 PM':'THU 4PM','Thursday 12 PM':'THU 12PM','Friday 3 PM':'FRI 3PM'}
time_order = ['THU', 'SUN 9AM','SUN 1PM', 'SUN 4PM', 'SUN 8PM', 'MON']
teamcolors = {'JTizzzzle': 'lightsalmon', 'bgmaddox':'mediumpurple', 'jlglover':'cyan', 'RascalHazard':'pink', 'BMoreBallers88':'gold', 
              'eegrady':'teal', 'DirtyCommie':'lime', 'jhuntmadd':'orange', 'RReclam':'green', 'sgmaddox':'darkseagreen', 'RossLikeSauce':'red', 'InfiniteJesse':'magenta'}

In [126]:
def MatchupPreview_WinPercentage(breakoutDF, week, Alternate=None):
    teamcolors = {'JTizzzzle': 'lightsalmon', 'bgmaddox':'mediumpurple', 'jlglover':'cyan', 'RascalHazard':'pink', 
                'BMoreBallers88':'gold', 'eegrady':'teal', 'DirtyCommie':'lime', 'jhuntmadd':'orange', 'RReclam':'green', 
                'sgmaddox':'darkseagreen', 'RossLikeSauce':'red', 'InfiniteJesse':'magenta'}
    if Alternate == 'SundayMorning':
        time_order = ['THU 8PM', 'SUN 9AM','SUN 1PM', 'SUN 4PM', 'SUN 8PM', 'MON 8PM']
    elif Alternate == 'Thanksgiving':
        time_order = ['THU 12PM', 'THU 4PM','THU 8PM','FRI 3PM','SUN 1PM', 'SUN 4PM', 'SUN 8PM', 'MON 8PM']
    if Alternate == None:
        time_order = ['THU 8PM','SUN 1PM', 'SUN 4PM', 'SUN 8PM', 'MON 8PM']
    
    breakoutDF['color'] = breakoutDF['team'].map(teamcolors)

    breakoutDF_group = breakoutDF.groupby(['team','Game_date_time','matchup']).size().reset_index(name='count')
    breakoutDF_group['GameTime-Simp'] = breakoutDF_group['Game_date_time'].map(time_dict)
    grouped = breakoutDF_group.groupby('matchup')
        # Create a subplot with 1 row and 2 columns (for the bar chart and the pie chart)
    figCombo = make_subplots(
            rows=6, cols=2, 
            shared_xaxes=True,
            column_widths=[0.85, 0.15],  # Adjust the width of each subplot
            specs=[[{"type": "bar"}, {"type": "pie"}],
                [{"type": "bar"}, {"type": "pie"}],
                [{"type": "bar"}, {"type": "pie"}],
                [{"type": "bar"}, {"type": "pie"}],
                [{"type": "bar"}, {"type": "pie"}],
                [{"type": "bar"}, {"type": "pie"}]]
            #subplot_titles=['Matchup Schedule','Win History']# Specify the chart types
        )
    row_ys = {1: 1, 2: 0.83, 3: 0.66, 4: 0.49, 5: 0.32, 6: 0.15}  # Adjust as needed
    for i in range(1,7):
        
        testgroup = grouped.get_group(i)

        teams = testgroup['team'].unique()
        win_mean = OppWinPercentageTable_Current.loc[teams[0], teams[1]]




        # Add the bar chart to the first column
        figCombo.add_trace(
            go.Bar(
                y=testgroup['count'], 
                x=testgroup['GameTime-Simp'], 
                marker=dict(
                    color = [teamcolors[team] for team in testgroup['team']], 
                    cornerradius = 10
                ),
                text=testgroup['count'],
                textangle = 0,
                textposition='auto',
                showlegend=False,
            ),
            row=i, col=1
        )

        # Add the pie chart to the second column
        figCombo.add_trace(
            go.Pie(values=[win_mean, 1-win_mean] ,labels=[f'{teams[0]}',f'{teams[1]}'],
                marker = dict(colors = [teamcolors[teams[0]],teamcolors[teams[1]]]), showlegend=False),
            row=i, col=2
        )
            # Create a custom title with colored team names
        title_html = f'<span style="color:{teamcolors[teams[0]]}">{teams[0]}</span> vs <span style="color:{teamcolors[teams[1]]}">{teams[1]}</span>'

        # Add the custom title as an annotation at the top of each subplot
        figCombo.add_annotation(
            text=title_html,  # The HTML title
            xref=f'x domain', yref=f'y domain',
            x=.5, y=1.2,  # Position it above the subplot (y > 1)
            xanchor='center',
            font=dict(size=15, family='Courier New', weight ='bold'),
            showarrow=False,
            row=i, col=1  # Apply to the i-th row and first column (bar chart)
        )
        # Update the layout with dark theme and grouped bar mode
        figCombo.update_layout(barmode="group", template="plotly_dark")
        figCombo.update_xaxes(
            categoryorder="array",
            categoryarray=time_order,
            showticklabels = True, 
            row=i, col=1  # Apply to the bar chart in the i-th row and first column
        )
        

    # Show the figure
    figCombo.update_layout(width=800, height=1200, title_text = f'Week {week} Preview', title_x=0.1)
    figCombo.add_annotation(
            text='Game Time Schedule',  # The HTML title
            xref=f'paper', yref=f'paper',
            x=.4, y=1.06,  # Position it above the subplot (y > 1)
            xanchor='center',
            font=dict(size=14, family='Courier New'),
            showarrow=False,
        )
    figCombo.add_annotation(
            text='Win History',  # The HTML title
            xref=f'paper', yref=f'paper',
            x=.93, y=1.06,  # Position it above the subplot (y > 1)
            xanchor='center',
            font=dict(size=14, family='Courier New'),
            showarrow=False,
        )
    figCombo.update_annotations(font_size = 20)
    figCombo.update_xaxes(tickfont=dict(family="Courier New", size=12, color="White"))
    figCombo.update_yaxes(dtick=2)
    figCombo.update_layout(font_family="Courier New", title_font = dict(size=35, weight = 'bold'))
    figCombo.update_layout(uniformtext_minsize=10, uniformtext_mode='hide')
    figCombo.update_layout(title={
        'pad': {'t': 50, 'b': 50, 'l': 10, 'r': 10}  # Adjust these values as needed
    })
    figCombo.update_layout(
    margin=dict(t=150, b=40, l=60, r=60)  # Adjust these values as needed
    )
    #figCombo.update_layout(title_text='Week 12 Preview')
    figCombo.show()
    return figCombo

In [127]:
week13Breakout2024

,team,matchup,player,points,starter,position,week_x,player_name,recent_teams,week_id,...,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr,week_id_NFL,Game_date_time,color
0,bgmaddox,1,Cedric Tillman,0.00,0,WR,13,Cedric Tillman,CLE,CLE-13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Monday 8 PM,mediumpurple
1,bgmaddox,1,Bucky Irving,26.80,1,RB,13,Bucky Irving,TB,TB-13,...,-2.750000,0.088235,-0.052174,0.095831,0.0,24.50,27.50,TB-13,Sunday 4 PM,mediumpurple
2,bgmaddox,1,Derrick Henry,12.60,1,RB,13,Derrick Henry,BAL,BAL-13,...,9.666667,0.085714,0.008571,0.134571,0.0,11.10,14.10,BAL-13,Sunday 4 PM,mediumpurple
3,bgmaddox,1,Demarcus Robinson,11.90,0,WR,13,Demarcus Robinson,LA,LA-13,...,1.139535,0.130435,0.222798,0.351611,0.0,10.90,12.90,LA-13,Sunday 4 PM,mediumpurple
4,bgmaddox,1,Alvin Kamara,13.90,1,RB,13,Alvin Kamara,NO,NO-13,...,7.000000,0.176471,0.004348,0.267749,0.0,11.90,15.90,NO-13,Sunday 4 PM,mediumpurple
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168,InfiniteJesse,4,Jaylen Waddle,9.30,0,WR,13,Jaylen Waddle,MIA,MIA-13,...,1.152174,0.086957,0.147436,0.233640,0.0,7.30,11.30,MIA-13,Thursday 8 PM,magenta
169,InfiniteJesse,4,Javonte Williams,6.90,0,RB,13,Javonte Williams,DEN,DEN-13,...,0.000000,0.064516,0.000000,0.096774,0.0,6.40,7.40,DEN-13,Monday 8 PM,magenta
170,InfiniteJesse,4,De'Von Achane,17.00,1,RB,13,De'Von Achane,MIA,MIA-13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Thursday 8 PM,magenta
171,InfiniteJesse,4,C.J. Stroud,16.38,1,QB,13,C.J. Stroud,HOU,HOU-13,...,NaN,NaN,NaN,NaN,0.0,14.38,14.38,HOU-13,Sunday 1 PM,magenta


In [128]:
Week12MatchupPreview = MatchupPreview_WinPercentage(week13Breakout2024, 13, Alternate='Thanksgiving')

## Luck Graph

In [129]:
def LuckChart(df, current_week):
    
    df_pivot, df_wins = WeeklyWins(df, current_week)

    df_week = df_wins[df_wins['Week'] == current_week]
    topscore = df_week['Score YTD'].max()
    topopponent = df_week['Opp YTD'].max()
    minscore = df_week['Score YTD'].min()
    minopponent = df_week['Opp YTD'].min()

    figScat = px.scatter(df_week, x="Opp YTD",y = 'Score YTD',size ="Total Wins", template = 'plotly_dark', color = 'Team', text = 'Team',title=f'Lucky Squares after Week {current_week}')
    figScat.update_layout(width=800, height=800)

    figScat.update_layout(
    title_font=dict(
        family="Courier New",
        size=24,
        color="gold",
        weight='bold'
    ), title_x=.5
)

    # Calculate median values
    median_score = df_week['Score YTD'].median() #y value
    median_opp = df_week['Opp YTD'].median() #x value
    
    # Add horizontal and vertical lines to divide the plot into quadrants
    figScat.add_shape(type="line", x0=median_opp, x1=median_opp, 
                y0=df_week['Score YTD'].min()-15, y1=df_week['Score YTD'].max()+15
                , opacity=0.5,
                line=dict(color="gold", width=2, dash="dash"))

    figScat.add_shape(type="line", x0=df_week['Opp YTD'].min()-15, x1=df_week['Opp YTD'].max()+15, 
                y0=median_opp, y1=median_opp, opacity=0.5,
                line=dict(color="gold", width=2, dash="dash"))

    figScat.update_traces(textposition='top center')
    figScat.update_traces(textfont_size=15,textfont=dict(weight='bold',family='Courier New',  # Font family
            size=15   # Font color
        ))
    figScat.update_layout(showlegend=False)
    # Add the diagonal line
    #figScat.update_layout(shapes = [{'type': 'line', 'yref': 'paper', 'xref': 'paper', 'y0': 0, 'y1': 1, 'x0': 0, 'x1': 1}])

    # Add text to each corner
    figScat.add_annotation(x=0, y=0, text="Bad but Lucky", showarrow=False, xref="paper", yref="paper",font=dict(
                family="Courier New, monospace",
                size=20,
                color='#17becf' 
                ))
    figScat.add_annotation(x=1, y=0, text="Bad & Unlucky", showarrow=False, xref="paper", yref="paper",font=dict(
                family="Courier New, monospace",
                size=20,
                color='#17becf' 
                ))
    figScat.add_annotation(x=0, y=1, text="Good & Lucky", showarrow=False, xref="paper", yref="paper",font=dict(
                family="Courier New, monospace",
                size=20,
                color='#17becf' 
                ))
    figScat.add_annotation(x=1, y=1, text="Good & Tested", showarrow=False, xref="paper", yref="paper",font=dict(
                family="Courier New, monospace",
                size=20,
                color='#17becf' 
                ))
    figScat.update_layout(
        xaxis_title="Points Against",  # Set x-axis title
        yaxis_title="Points For",   # Set y-axis title
        xaxis=dict(
            title_font=dict(
                family="Courier New",  # Set font family for x-axis title
                size=30,          # Set font size for x-axis title
                color ='red',
                weight = 'bold'
            )
        ),
        yaxis=dict(
            title_font=dict(
                family="Courier New",  # Set font family for y-axis title
                size=30,          # Set font size for y-axis title
                color = 'green',
                weight = 'bold'
            )
        )
    )

    figScat.update_yaxes(dtick=100,
                    tickfont=dict(
            family='Courier New',  # Font family
            size=16,         # Font size
            color='white'    # Font color
        ))
    figScat.update_xaxes(dtick=100,
                    tickfont=dict(
            family='Courier New',  # Font family
            size=16,         # Font size
            color='white'    # Font color
        ))
    figScat.add_annotation(
    text="O Size = Wins | --- = Avg",
    xref="paper", yref="paper",
    x=0.5, y=1.06, # Position relative to figure (right side, middle)
    showarrow=False,
    font=dict(
        family="Courier New, monospace",
        size=12,
        color="white"
    )
    )
    
    figScat.update_layout(
    #margin=dict(l=110, r=50, t=100, b=110)  # Adjust values as needed
)
    figScat.update_layout(xaxis=dict(range=[minopponent - 30, topopponent + 40])) 
    figScat.update_layout(yaxis=dict(range=[minscore - 25, topscore + 25])) 
    return figScat



In [542]:
WeekLuckTest = LuckChart(Matches_2024,17)

In [543]:
WeekLuckTest

## Weekly Score Graph

In [417]:
figScoreLine = px.line(WeekWins, x='Week',y='Total', template='plotly_dark', facet_col='Team',markers=True,facet_col_wrap=4, color='Team', line_shape='spline')
figScoreLine.update_layout(xaxis_title="", yaxis_title="")
figScoreLine.update_layout(width=1200, height=1000, showlegend=False)
figScoreLine.update_layout(title="2024 Score Summary", title_font=dict(size=40))
figScoreLine.update_annotations(font_size=20)
figScoreLine.update_layout(
    title={
        'x': 0.5,  # 0: left, 0.5: center, 1: right
        'xanchor': 'center',  # 'left', 'center', 'right',
        'pad': {'t': 100, 'b': 100, 'l': 10, 'r': 10}
    }
)
# Customize facet titles
figScoreLine.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
figScoreLine.update_layout(
    font=dict(
        family="Courier New",  # Specify the font family you want
        size=15,  # Optional: Set the font size
        color="white"  # Optional: Set the font color
    )
)
figScoreLine.update_xaxes(title_text="", col=1)
figScoreLine.update_xaxes(title_text="", col=2)
figScoreLine.update_xaxes(title_text="", col=3)
figScoreLine.update_xaxes(title_text="", col=4)
figScoreLine.update_yaxes(title_text="", col=1)
figScoreLine.update_xaxes(matches=None)
figScoreLine.for_each_xaxis(lambda xaxis: xaxis.update(showticklabels=True))
'''
# Update facet titles and colors
for i, category in enumerate(WeekWins['Team'].unique()):
    figScoreLine.update_layout(
        annotations=[
            dict(
                #text=category, 
                #x=0.5, 
                #y=1.05, 
                #xref='paper', 
                #yref='paper', 
                showarrow=False, 
                font=dict(color=figScoreLine.data[i].line.color)
            )
        ]
    )
'''
figScoreLine.show()

## Time Area Graph

In [133]:
TimeAreaDF = week10Breakout2024[week10Breakout2024['starter'] == 1]
SortNumbers = {'Sunday 1 PM':3, 'Thursday 8 PM':1, 'Monday 8 PM':6, 'Sunday 4 PM':4,
       'Sunday 8 PM':5, 'Sunday 9 AM':2}
TimeAreaDF['TimeSort'] = TimeAreaDF.Game_date_time.map(SortNumbers)
TimeAreaDFGraph = TimeAreaDF.groupby(['matchup','team','Game_date_time'],sort='TimeSort').points.sum().reset_index()
TimeAreaDFGraph['TimeSort'] = TimeAreaDFGraph.Game_date_time.map(SortNumbers)
TimeAreaDFGraph['ScoreTally'] = TimeAreaDFGraph.sort_values('TimeSort',ascending=True).groupby('team').points.cumsum()
TimeAreaDFGraph = TimeAreaDFGraph.sort_values(['team','TimeSort'])
TeamNames = TimeAreaDFGraph.groupby('matchup').team.unique().reset_index()

TeamNames['MatchupTitle'] = TeamNames.team.str.join(' vs ')
MatchupNames = dict(zip(TeamNames.matchup, TeamNames.MatchupTitle))
TimeAreaDFGraph['MatchupTitle'] = TimeAreaDFGraph.matchup.map(MatchupNames)

In [134]:
TimeswithLines = {'Thursday 8 PM':'Thursday<br>8 PM',
'Sunday 9 AM':'Sunday<br>9 AM',
'Sunday 1 PM':'Sunday<br>1 PM',
'Sunday 4 PM':'Sunday<br>4 PM',
'Sunday 8 PM':'Sunday<br>8 PM',
'Monday 8 PM': 'Monday<br>8 PM'}
tickvals = list(TimeswithLines.keys())
ticktext = list(TimeswithLines.values())

In [135]:
TimeTickLabels = ['Thursday<br>8 PM',
'Sunday<br>9 AM',
'Sunday<br>1 PM',
'Sunday<br>1 PM',
'Sunday<br>8 PM',
'Monday<br>8 PM']

In [136]:
TimeAreaDF = week10Breakout2024[week10Breakout2024['starter'] == 1]
TimeAreaDFGraph['TimeTitle'] = TimeAreaDFGraph['Game_date_time'].map(TimeswithLines)

In [137]:
week11Breakout2024

,team,matchup,player,points,starter,position,week_x,player_name,recent_teams,week_id,...,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr,week_id_NFL,Game_date_time,color
0,bgmaddox,1,Cedric Tillman,6.20,1,WR,11,Cedric Tillman,CLE,CLE-11,...,0.394958,0.177778,0.264444,0.451778,0.0,4.70,7.700000,CLE-11,Sunday 1 PM,mediumpurple
1,bgmaddox,1,Bucky Irving,0.00,0,RB,11,Bucky Irving,TB,TB-11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,mediumpurple
2,bgmaddox,1,Derrick Henry,10.50,1,RB,11,Derrick Henry,BAL,BAL-11,...,NaN,NaN,NaN,NaN,0.0,10.50,10.500000,BAL-11,Sunday 1 PM,mediumpurple
3,bgmaddox,1,Demarcus Robinson,2.90,0,WR,11,Demarcus Robinson,LA,LA-11,...,0.593750,0.148148,0.110727,0.299731,0.0,1.90,3.900000,LA-11,Sunday 1 PM,mediumpurple
4,bgmaddox,1,Alvin Kamara,10.90,1,RB,11,Alvin Kamara,NO,NO-11,...,-1.222222,0.142857,-0.082192,0.156751,0.0,8.90,12.900000,NO-11,Sunday 1 PM,mediumpurple
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
164,InfiniteJesse,3,Tutu Atwell,2.60,0,WR,11,Tutu Atwell,LA,LA-11,...,0.750000,0.074074,0.096886,0.178931,0.0,2.10,3.100000,LA-11,Sunday 1 PM,magenta
165,InfiniteJesse,3,Javonte Williams,16.70,0,RB,11,Javonte Williams,DEN,DEN-11,...,-4.666667,0.156250,-0.045455,0.202557,0.0,14.70,18.700001,DEN-11,Sunday 4 PM,magenta
166,InfiniteJesse,3,De'Von Achane,18.50,1,RB,11,De'Von Achane,MIA,MIA-11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sunday 1 PM,magenta
167,InfiniteJesse,3,C.J. Stroud,10.88,1,QB,11,C.J. Stroud,HOU,HOU-11,...,NaN,NaN,NaN,NaN,0.0,10.88,10.880000,HOU-11,Monday 8 PM,magenta


In [569]:
def PointsOverTheWeekend(breakoutDF, week_num, Alternate=None):    
    
    title_string = f'Points Timeline: Week {week_num}'
    TimeAreaDF = breakoutDF[breakoutDF['starter'] == 1]
    SortNumbers = {'Wednesday 1 PM': 1, 'Wednesday 4 PM':2, 'Thursday 8 PM':3, 'Friday 3 PM': 4, 
                   'Saturday 1 PM': 5, 'Saturday 4 PM': 6, 'Saturday 8 PM': 7, 
                   'Sunday 9 AM':8, 'Sunday 1 PM':9, 'Sunday 4 PM':10, 'Sunday 8 PM':11, 'Monday 8 PM':12,}
    TimeAreaDF['TimeSort'] = TimeAreaDF.Game_date_time.map(SortNumbers)
    TimeAreaDFGraph = TimeAreaDF.groupby(['matchup','team','Game_date_time'],sort='TimeSort').points.sum().reset_index()
    TimeAreaDFGraph['TimeSort'] = TimeAreaDFGraph.Game_date_time.map(SortNumbers)
    TimeAreaDFGraph['ScoreTally'] = TimeAreaDFGraph.sort_values('TimeSort',ascending=True).groupby('team').points.cumsum()
    TimeAreaDFGraph = TimeAreaDFGraph.sort_values(['team','TimeSort'])

    TeamNames = TimeAreaDFGraph.groupby('matchup').team.unique().reset_index()
    TeamNames['MatchupTitle'] = TeamNames.team.str.join(' vs ')
    MatchupNames = dict(zip(TeamNames.matchup, TeamNames.MatchupTitle))
    TimeAreaDFGraph['MatchupTitle'] = TimeAreaDFGraph.matchup.map(MatchupNames)

    if Alternate == 'SundayMorning':
        TimeswithLines = {'Thursday 8 PM':'Thursday<br>8 PM',
        'Sunday 9 AM':'Sunday<br>9 AM',
        'Sunday 1 PM':'Sunday<br>1 PM',
        'Sunday 4 PM':'Sunday<br>4 PM',
        'Sunday 8 PM':'Sunday<br>8 PM',
        'Monday 8 PM': 'Monday<br>8 PM'}
        
    elif Alternate == 'Thanksgiving':
        TimeswithLines = {
        'Thursday 12 PM':'Thursday<br>12 PM',
        'Thursday 4 PM':'Thursday<br>4 PM',        
        'Thursday 8 PM':'Thursday<br>8 PM',
        'Friday 3 PM':'Friday<br>3 PM',
        'Sunday 9 AM':'Sunday<br>9 AM',
        'Sunday 1 PM':'Sunday<br>1 PM',
        'Sunday 4 PM':'Sunday<br>4 PM',
        'Sunday 8 PM':'Sunday<br>8 PM',
        'Monday 8 PM': 'Monday<br>8 PM'}
    elif Alternate == 'Saturday':
        TimeswithLines = {'Thursday 8 PM':'THU<br>8 PM',
        'Saturday 1 PM':'SAT<br>1 PM',
        'Saturday 4 PM':'SAT<br>4 PM',
        'Sunday 1 PM':'SUN<br>1 PM',
        'Sunday 4 PM':'SUN<br>4 PM',
        'Sunday 8 PM':'SUN<br>8 PM',
        'Monday 8 PM': 'MON<br>8 PM'}
    elif Alternate == 'NewYears':
        TimeswithLines = {
        'Wendesday 1 PM': 'WED<br>1 PM',
        'Wednesday 4 PM': 'WED<br>4 PM',
        'Thursday 8 PM':'THU<br>8 PM',
        'Saturday 1 PM':'SAT<br>1 PM',
        'Saturday 4 PM':'SAT<br>4 PM',
        'Saturday 8 PM':'SAT<br>8 PM',
        'Sunday 1 PM':'SUN<br>1 PM',
        'Sunday 4 PM':'SUN<br>4 PM',
        'Sunday 8 PM':'SUN<br>8 PM',
        'Monday 8 PM': 'MON<br>8 PM'}
    elif Alternate == None:
        TimeswithLines = {'Thursday 8 PM':'Thursday<br>8 PM',
        'Sunday 1 PM':'Sunday<br>1 PM',
        'Sunday 4 PM':'Sunday<br>4 PM',
        'Sunday 8 PM':'Sunday<br>8 PM',
        'Monday 8 PM': 'Monday<br>8 PM'}
        
    tickvals = list(TimeswithLines.keys())
    ticktext = list(TimeswithLines.values())
    teamcolors = {'JTizzzzle': 'lightsalmon', 'bgmaddox':'mediumpurple', 'jlglover':'cyan', 'RascalHazard':'pink', 
                    'BMoreBallers88':'gold', 'eegrady':'teal', 'DirtyCommie':'lime', 'jhuntmadd':'orange', 'RReclam':'tomato', 
                    'sgmaddox':'darkseagreen', 'RossLikeSauce':'red', 'InfiniteJesse':'magenta'}
    TimeAreaDFGraph['color'] = TimeAreaDFGraph['team'].map(teamcolors)

    figWeekLine = px.area(TimeAreaDFGraph.sort_values(['matchup','TimeSort']), x='Game_date_time', y = 'ScoreTally',color = 'team',color_discrete_map=teamcolors, template = 'plotly_dark', 
                        facet_col='MatchupTitle', facet_col_wrap=2,  facet_col_spacing=0.1, title=title_string, markers=True)
    figWeekLine.update_layout(height=1200, width = 1000)
    #'''
    if Alternate == 'SundayMorning':
        figWeekLine.update_layout(
            xaxis={'categoryorder': 'array', 'categoryarray': ['Thursday 8 PM','Sunday 9 AM','Sunday 1 PM', 'Sunday 4 PM',
            'Sunday 8 PM','Monday 8 PM']})
    elif Alternate == 'Thanksgiving':
        figWeekLine.update_layout(
            xaxis={'categoryorder': 'array', 'categoryarray': ['Thursday 12 PM','Thursday 4 PM','Thursday 8 PM','Friday 3 PM','Sunday 9 AM','Sunday 1 PM', 'Sunday 4 PM',
            'Sunday 8 PM','Monday 8 PM']})
    elif Alternate == 'Saturday':
        figWeekLine.update_layout(
            xaxis={'categoryorder': 'array', 'categoryarray': ['Thursday 8 PM','Saturday 1 PM', 'Saturday 4 PM''Sunday 1 PM', 'Sunday 4 PM',
            'Sunday 8 PM','Monday 8 PM']})
    elif Alternate == 'NewYears':
        figWeekLine.update_layout(
            xaxis={'categoryorder': 'array', 'categoryarray': ['Wednesday 1 PM','Wednesday 4 PM','Thursday 8 PM','Saturday 1 PM', 'Saturday 4 PM','Saturday 8 PM'
                                                               ,'Sunday 1 PM', 'Sunday 4 PM','Sunday 8 PM','Monday 8 PM']})
    elif Alternate == None:
        figWeekLine.update_layout(
            xaxis={'categoryorder': 'array', 'categoryarray': ['Thursday 8 PM','Sunday 1 PM', 'Sunday 4 PM',
            'Sunday 8 PM','Monday 8 PM']})    
    #'''
    figWeekLine.update_traces(stackgroup=None,fill='tozeroy', line_shape = 'spline')
    figWeekLine.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    figWeekLine.update_layout(font_family="Courier New", title_font = dict(size=35), showlegend=False)
    figWeekLine.update_xaxes(title_text="")
    # Update facet titles to include team colors
    for annotation in figWeekLine.layout.annotations:
        title_text = annotation.text  # e.g., "Team A vs Team B"
        teams = title_text.split(" vs ")  # Split into individual team names
        # Create a styled title with team names in respective colors
        annotation.text = f"<span style='color:{teamcolors[teams[0]]}'>{teams[0]}</span> vs " \
                        f"<span style='color:{teamcolors[teams[1]]}'>{teams[1]}</span>"
        annotation.font.size = 15  # Optional: Adjust font size for clarity

    # hide subplot y-axis titles and x-axis titles
    for axis in figWeekLine.layout:
        if type(figWeekLine.layout[axis]) == go.layout.YAxis:
            figWeekLine.layout[axis].title.text = ''
        if type(figWeekLine.layout[axis]) == go.layout.XAxis:
            figWeekLine.layout[axis].title.text = ''
    figWeekLine.update_yaxes( showticklabels=True, visible=True)
    figWeekLine.update_xaxes( showticklabels=True, visible=True,ticktext=TimeTickLabels)
    figWeekLine.update_layout(
        xaxis=dict(ticktext=ticktext, tickvals = tickvals),
        xaxis1=dict(ticktext=ticktext, tickvals = tickvals),
        xaxis2=dict(ticktext=ticktext, tickvals = tickvals),
        xaxis3=dict(ticktext=ticktext, tickvals = tickvals),
        xaxis4=dict(ticktext=ticktext, tickvals = tickvals),
        xaxis5=dict(ticktext=ticktext, tickvals = tickvals),
        xaxis6=dict(ticktext=ticktext, tickvals = tickvals),
        )

    figWeekLine.show()
    return figWeekLine, TimeAreaDFGraph

In [551]:
week17Breakout2024

,team,matchup,player,points,starter,position,week_x,player_name,recent_teams,week_id,...,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr,week_id_NFL,Game_date_time,color
0,bgmaddox,4.0,Bucky Irving,21.0,1,RB,17,Bucky Irving,TB,TB-17,...,-5.923077,0.125000,-0.049808,0.152634,0.0,19.0,23.0,TB-17,Sunday 1 PM,mediumpurple
1,bgmaddox,4.0,Jalen McMillan,20.5,0,WR,17,Jalen McMillan,TB,TB-17,...,1.593750,0.156250,0.122605,0.320199,0.0,18.0,23.0,TB-17,Sunday 1 PM,mediumpurple
2,bgmaddox,4.0,Joe Flacco,19.3,1,QB,17,Joe Flacco,IND,IND-17,...,NaN,NaN,NaN,NaN,0.0,15.3,15.3,IND-17,Sunday 1 PM,mediumpurple
3,bgmaddox,4.0,Brandin Cooks,7.2,0,WR,17,Brandin Cooks,DAL,DAL-17,...,0.356164,0.275862,0.553030,0.800914,0.0,5.2,9.2,DAL-17,Sunday 1 PM,mediumpurple
4,bgmaddox,4.0,Derrick Henry,23.5,1,RB,17,Derrick Henry,BAL,BAL-17,...,18.000000,0.125000,0.005291,0.191204,0.0,22.5,24.5,BAL-17,Wednesday 4 PM,mediumpurple
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166,InfiniteJesse,2.0,Javonte Williams,1.0,0,RB,17,Javonte Williams,DEN,DEN-17,...,NaN,0.064516,0.000000,0.096774,0.0,0.0,2.0,DEN-17,Saturday 4 PM,magenta
167,InfiniteJesse,2.0,Christian Watson,0.0,0,WR,17,Christian Watson,GB,GB-17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sunday 4 PM,magenta
168,InfiniteJesse,2.0,De'Von Achane,4.8,1,RB,17,De'Von Achane,MIA,MIA-17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sunday 4 PM,magenta
169,InfiniteJesse,2.0,C.J. Stroud,6.1,0,QB,17,C.J. Stroud,HOU,HOU-17,...,NaN,NaN,NaN,NaN,0.0,6.1,6.1,HOU-17,Wednesday 4 PM,magenta


In [570]:
Week12Weekend, TimeAreaDFGraph = PointsOverTheWeekend(week17Breakout2024, 17, Alternate='NewYears')

In [141]:
TimeAreaDFGraph

,matchup,team,Game_date_time,points,TimeSort,ScoreTally,MatchupTitle,color
0,1,BMoreBallers88,Sunday 1 PM,120.38,3,120.38,BMoreBallers88 vs bgmaddox,gold
1,1,BMoreBallers88,Sunday 4 PM,10.48,4,130.86,BMoreBallers88 vs bgmaddox,gold
31,5,DirtyCommie,Thursday 8 PM,14.90,1,14.90,DirtyCommie vs jhuntmadd,lime
30,5,DirtyCommie,Sunday 1 PM,54.40,3,69.30,DirtyCommie vs jhuntmadd,lime
29,5,DirtyCommie,Monday 8 PM,26.58,6,95.88,DirtyCommie vs jhuntmadd,lime
15,3,InfiniteJesse,Sunday 1 PM,76.38,3,76.38,InfiniteJesse vs JTizzzzle,magenta
16,3,InfiniteJesse,Sunday 4 PM,4.90,4,81.28,InfiniteJesse vs JTizzzzle,magenta
17,3,InfiniteJesse,Sunday 8 PM,15.00,5,96.28,InfiniteJesse vs JTizzzzle,magenta
19,3,JTizzzzle,Sunday 1 PM,41.00,3,41.00,InfiniteJesse vs JTizzzzle,lightsalmon
20,3,JTizzzzle,Sunday 4 PM,10.10,4,51.10,InfiniteJesse vs JTizzzzle,lightsalmon


## Weekly Score Trends

In [142]:
ScoreTrends = round(Matches_2024DF.groupby('Week')['Total'].agg(['min','mean','max']),2)

In [143]:
ScoreTrends

,min,mean,max
Week,,,
1,74.86,103.45,125.58
2,79.24,105.05,161.12
3,70.08,100.02,145.96
4,80.10,105.24,143.80
5,56.26,107.03,157.52
6,87.14,112.02,135.52
7,55.98,100.52,130.34
8,83.92,113.09,139.00
9,76.20,118.39,157.28


In [419]:
figTrends = px.line(ScoreTrends.reset_index(), x='Week', y = ['min','max','mean'], template = 'plotly_dark',line_shape = 'spline', title = 'Scoring Trends', markers=True )
figTrends.update_layout(
    yaxis=dict(
        range=[0, 180]  # Set the y-axis range from 0 to 10
    )
)
figTrends.update_layout(width=1200, height=800)
figTrends.update_layout(font_family="Courier New", title_font = dict(size=45))
figTrends.update_yaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=15,         # Font size
            color='white'    # Font color
        ),
        title = None,
        
    )
figTrends.update_xaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=15,         # Font size
            color='white'    # Font color
        ),
        title = 'Week',
        
    )
figTrends.update_layout(legend=dict(orientation="h",font=dict(size= 18)))
# Set legend location
figTrends.update_layout(
    legend=dict(
        x=0.45,  # x-coordinate of the legend (0 to 1, where 0 is left and 1 is right)
        y=1.02,   # y-coordinate of the legend (0 to 1, where 0 is bottom and 1 is top)
        xanchor="center",  # horizontal anchor point ('left', 'center', 'right')
        yanchor="top",   # vertical anchor point ('top', 'middle', 'bottom')
        title = None
    )
)
figTrends.show()

# Season Graphs

## All-Time Graph

In [145]:
AllTimeGraph = WeeklyWinsGraph(AllMatchesDF,70)

In [146]:
def AllTimeGraphing(df,week):
    fig2 = px.line(df,x='Week Index',y='Total Wins', color = 'Team',template='plotly_dark',line_shape = 'spline', title = 'All-Time Wins')
    fig2.update_xaxes(
                    tickfont=dict(
            family='Courier New',  # Font family
            size=18,         # Font size
            color='white'    # Font color
        ))
    fig2.update_yaxes(dtick=10)
    fig2.update_layout(width=1400, height=900)
    # Adjust the thickness of the lines
    fig2.update_traces(line=dict(width=4))  # Set the line width (e.g., 3 pixels)
    fig2.update_layout(legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
        title = ''
    ))

    #fig2.update_layout(showlegend = False)
    # Customize the y-axis labels
    fig2.update_yaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=20,         # Font size
            color='white'    # Font color
        )
    )

    fig2.update_layout(uniformtext_minsize=10, uniformtext_mode='hide')

    # Update x-axis and y-axis titles with font customization
    fig2.update_layout(
        xaxis_title="Week",  # Set x-axis title
        yaxis_title="Wins",   # Set y-axis title
        xaxis=dict(dtick = 10,
            title_font=dict(
                family="Courier New",  # Set font family for x-axis title
                size=20          # Set font size for x-axis title
            )
        ),
        yaxis=dict(
            title_font=dict(
                family="Courier New",  # Set font family for y-axis title
                size=20          # Set font size for y-axis title
            )
        )
    )
    
    # Determine final wins and sort by them
    final_scores = [(d.name, d.x[-1], d.y[-1], d.line.color) for d in fig2.data]
    final_scores.sort(key=lambda x: -x[2])  # Sort by final win count, descending

    # Define a list of potential text positions to avoid overlap
    text_positions = ['middle right','top right', 'bottom right', 'top left', 'bottom left', 'middle left']

    previous_score = None
    position_index = 0
    '''
    for team_name, x_final, y_final, color in final_scores:
        if y_final == previous_score:
            # Cycle through text positions to avoid overlap if scores are the same
            position_index = (position_index + 1) % len(text_positions)
        else:
            position_index = 0  # Reset position index when score changes

        text_position = text_positions[position_index]
        previous_score = y_final

        fig2.add_scatter(
            x=[x_final], y=[y_final],
            mode='markers+text',
            text=[team_name],
            textfont=dict(family="Courier New",color=color, size=14, weight = 'bold'),
            textposition=text_position,
            marker=dict(color=color, size=12),
            showlegend=False
            )    
        fig2.update_layout(
        margin=dict(l=50, r=100, t=50, b=50)  # Set left, right, top, bottom padding within the plot area
            )
        
        '''
    fig2.update_layout(xaxis=dict(range=[0, week + 3])) 
    fig2.update_layout(title=dict(
            font=dict(
                size=50,
                family="Courier New"))) 
    
    # Find the last data point for each host
    last_points = AllMatchesDF.groupby('Team').apply(lambda d: d.nlargest(1, 'Total Wins','last')).reset_index(drop=True)

    # Define the subset of hosts to style differently
    special_hosts = ['SleeperGawd69']
    special_hosts2 = ['sgmaddox', 'RReclam', 'BillyRayGonnaGetcha','jhuntmadd']
    special_hosts3 = ['eegrady', 'Just_Here_For_The_Snacks']
    special_hosts4 = []
    special_hosts5 = ['GurlyGirls', 'SweetDizzzzzle', 'YouthPastor']
    normal_teams = ['bgmaddox', 'jlglover', 'BMoreBallers88', 'RascalHazard', 'InfiniteJess', 'DirtyCommie', 'JTizzzzle', 'RossLikeSauce','akbrown29']

    for i, row in last_points.iterrows():
        if row['Team'] in special_hosts:
            # Apply special styling for the subset
            text_position = 'top left'
            marker_size = 12
            text_color = 'lightgrey'
            # Default offset
            x_offset = -20
            y_offset = -50
            show_arrow = True
        elif row['Team'] in special_hosts2:
            x_offset = 30
            y_offset = 25
            text_position = 'top left'
            marker_size = 12
            text_color = 'lightgrey'
            show_arrow = True
        elif row['Team'] in special_hosts3:
            x_offset = 50
            y_offset = -20
            text_position = 'top left'
            marker_size = 12
            text_color = 'lightgrey'
            show_arrow = True    
        elif row['Team'] in special_hosts4:
            x_offset = -75
            y_offset = -75
            text_position = 'middle right'
            marker_size = 12
            text_color = 'lightgrey'
            show_arrow = True   
        elif row['Team'] in special_hosts5:
            x_offset = -75
            y_offset = -75
            text_position = 'middle right'
            marker_size = 12
            text_color = 'lightgrey'
            show_arrow = True   
        elif row['Team'] in normal_teams:
            x_offset = 75
            y_offset = 0
            text_position = 'middle right'
            marker_size = 12
            text_color = 'lightgrey'
            show_arrow = True   
        else:
            # Default styling
            text_position = 'top right'
            marker_size = 12
            #text_color = 'lightgrey'
            # Default offset
            x_offset = -100
            y_offset = -0
            show_arrow = False
        fig2.add_annotation(
            x=row['Week Index'],
            y=row['Total Wins'],
            text=row['Team'],
            showarrow=show_arrow,
            arrowhead=2,
            ax=x_offset,
            ay=y_offset,
            font=dict(
                size=18,
                #color="white",
                family = 'Courier New',
                weight = 'bold'
            ),
            align="left"
        )
    fig2.update_layout(legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-.30,
        xanchor="center",
        x=.5,
        title = '',
        font_size=18,
        font=dict(family="Courier New")
    ))
    return fig2

In [147]:
AllMatchesDF['Team'].unique()

array(['bgmaddox', 'jlglover', 'BMoreBallers88', 'YouthPastor',
       'GurlyGirls', 'SweetDizzzzzle', 'RascalHazard', 'JTizzzzle',
       'akbrown29', 'eegrady', 'BillyRayGonnaGetcha', 'RossLikeSauce',
       'jhuntmadd', 'RReclam', 'DirtyCommie', 'Just_Here_For_The_Snacks',
       'sgmaddox', 'InfiniteJesse'], dtype=object)

In [148]:
AllMatchesDF.sort_values('Week Index')
AllTimeGraphTest = AllTimeGraphing(AllMatchesDF,80)
AllTimeGraphTest.show()

## All Game Points Graphs

### Points by Week

In [149]:
AllMatchesDF.head()

,Team,QB,RB1,RB2,WR1,WR2,TE,WRT,K,DEF,...,Year,LeagueTotal,PercentTotal,Opp_team,Abs Margin,Year-Week,Total Wins,Score YTD,Opp YTD,Week Count
0,bgmaddox,{'Drew Brees': 24.8},{'David Johnson': 22.7},{'Sony Michel': 1.4},{'JuJu Smith-Schuster': 10.8},{'DJ Moore': 8.1},{'David Njoku': 11.7},{'Calvin Ridley': 14.4},{'Harrison Butker': 17.0},{'BAL': 15.0},...,2019,1350.32,9.3,jlglover,12.50,2019 - 01,1,125.90,113.40,0
1,jlglover,{'Russell Wilson': 20.6},{'Nick Chubb': 10.0},{'Aaron Jones': 3.4},{'T.Y. Hilton': 24.7},{'Marvin Jones': 8.0},{'Travis Kelce': 10.3},{'Derrick Henry': 28.4},{'Mason Crosby': 4.0},{'NO': 4.0},...,2019,1350.32,8.4,bgmaddox,12.50,2019 - 01,0,113.40,125.90,0
2,BMoreBallers88,{'Lamar Jackson': 43.56},{'James Conner': 8.5},{'Tevin Coleman': 6.6},{'Odell Beckham': 10.6},{'Tyler Boyd': 10.3},{'Hunter Henry': 8.0},{'Stefon Diggs': 4.7},{'Robbie Gould': 11.0},{'LAC': 1.0},...,2019,1350.32,7.7,YouthPastor,10.36,2019 - 01,0,104.26,114.62,0
3,YouthPastor,{'Carson Wentz': 31.02},{'Damien Williams': 15.5},{'Devonta Freeman': 1.6},{'DeAndre Hopkins': 27.1},{'Tyler Lockett': 10.9},{'Darren Waller': 10.5},{'Duke Johnson': 11.0},{'Matt Bryant': 0.0},{'NE': 7.0},...,2019,1350.32,8.5,BMoreBallers88,10.36,2019 - 01,1,114.62,104.26,0
4,GurlyGirls,{'Cam Newton': 4.36},{'Saquon Barkley': 14.9},{'James White': 10.7},{'Adam Thielen': 11.8},{'Cooper Kupp': 8.1},{'Vance McDonald': 5.0},{'Josh Jacobs': 23.8},{'Brett Maher': 5.0},{'SEA': 11.0},...,2019,1350.32,7.0,SweetDizzzzzle,30.86,2019 - 01,0,94.66,125.52,0


In [150]:


#figBarWeek = px.bar(AllMatchesDFTop.sort_values('Week Index'), x=AllMatchesDFTop.index,y='Total',template = 'plotly_dark',color = "Team", orientation='h')
figBarWeek = px.bar(AllMatchesDF.sort_values('Week Index', ascending=True), y=AllMatchesDF.index,x='Total',template = 'plotly_dark',color = "Team", orientation='h')


#figBarWeek.update_layout(barmode='group', xaxis={'categoryorder':'total descending'})
figBarWeek.update_layout(legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
        title = ''
    ))

# want different color for first bar... change marker color to a list for appropriate trace
figBarWeek.for_each_trace(
    lambda t: t.update(
        {
            "marker": {
                "color": [
                    "green" if i == 0 else t.marker.color for i, _ in enumerate(t.x)
                ]
            }
        }
    )
    if t.name == 'bgmaddox'
    else t
)
#Update the layout to hide the legend:
figBarWeek.update(layout_coloraxis_showscale=True)
figBarWeek.add_vline(x=162)
figBarWeek.update_layout(width=800, height=1200)
figBarWeek.show()

### Whole Season Bar Graph

In [151]:
figBarWeek = px.bar(Matches_2024DF.reset_index().sort_values(['Week','Total']), x='Team',y='Total',template = 'plotly_dark',color = "Week", barmode='stack')
figBarWeek.update_layout(barmode='group', xaxis={'categoryorder':'total descending'})
figBarWeek.update_layout(legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
        title = ''
    ))
#Update the layout to hide the legend:
figBarWeek.update(layout_coloraxis_showscale=True)
figBarWeek.add_vline(x=5.5)
figBarWeek.show()

### Top League Players per Team

In [434]:
year = 2024
figBar = px.bar(Best_2024,x='player',y='points',template = 'plotly_dark',color = "position", barmode='stack', facet_col ='team',facet_col_wrap=3, 
                title = f'Top League Players per Team: {year}')
figBar.update_layout(barmode='stack')#, xaxis={'categoryorder':'total descending'})
figBar.update_layout(legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
        title = ''
    ))
figBar.update_layout(title=dict(
            font=dict(
                size=28,
                family="Courier New")),
            title_x =0.5
                     )  
figBar.update_layout(width=1200, height=1000,
                     xaxis=dict(
            title_font=dict(
                family="Courier New",  # Set font family for x-axis title
                size=30,          # Set font size for x-axis title
                color ='red',
                weight = 'bold'
            )
        ),
        yaxis=dict(
            title_font=dict(
                family="Courier New",  # Set font family for y-axis title
                size=30,          # Set font size for y-axis title
                color = 'green',
                weight = 'bold'
            )
        )
                     )
figBar.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
figBar.update_xaxes(showticklabels=False, title = None)
figBar.update_yaxes( title = None)
figBar.update_yaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=18,         # Font size
            color='white'    # Font color
        ),
        title=None
    )
figBar.update_annotations(font_size=16,font=dict(family="Courier New"))

figBar.update_layout(legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=.5,
        title = '',
        font_size=22,
        font=dict(family="Courier New")
    ))
figBar.show()

figBar = make_subplots(rows=2, cols=1)

figBar.append_trace(go.Bar(x=first_half['player'],y=first_half['points'],template = 'plotly_dark',color = first_half["position"], barmode='stack',row=1,col=1))
figBar.update_layout(barmode='stack', xaxis={'categoryorder':'total descending'})
figBar.update_layout(legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=1,
        title = ''
    ))
figBar.append_trace(go.bar(second_half, x='player',y='points',template = 'plotly_dark',color = "position", barmode='stack',row=2,col=1))
figBar.update_layout(barmode='stack', xaxis={'categoryorder':'total descending'})
figBar.update_layout(legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
        title = ''
    ))
figBar.show()

## WeekYTDTotals Graph

In [153]:
def WeekYTDTotalsPercents(dfNoMatch,week):
    percent = df.groupby('Week')['Total'].sum()
    percent = percent.reset_index()
    WeekTotal = dict(zip(percent['Week'],percent['Total']))
    
    df['LeagueTotal'] = df['Week'].map(WeekTotal)
    df['PercentTotal'] = ((df['Total'] / df['LeagueTotal']) * 100).round(1)
    
    return

In [154]:
percent = Week5NoMatches_2024.groupby('Week')['Total'].sum()

In [155]:
percent = percent.reset_index()
WeekTotal = dict(zip(percent['Week'],percent['Total']))

In [156]:
WeekWins = WeekWins.reset_index()
WeekWins['BarName'] = WeekWins['Team'] + ' [' + WeekWins['PercentTotal'].astype(str) + '%]'
WeekWins.head()

,index,Team,QB,RB1,RB2,WR1,WR2,TE,WRT,K,...,Week Index,Year,LeagueTotal,PercentTotal,Opp_team,Total Wins,Score YTD,Opp YTD,LatestWin,BarName
0,0,bgmaddox,{'Jalen Hurts': 19.42},{'Derrick Henry': 10.6},{'Alvin Kamara': 19.5},{'Amon-Ra St. Brown': 2.8},{'Calvin Ridley': 6.5},{'Trey McBride': 5.5},{'D'Andre Swift': 5.0},{'Younghoe Koo': 3.4},...,71,2024,1241.44,7.4,BMoreBallers88,0,91.72,119.02,6,bgmaddox [7.4%]
1,1,BMoreBallers88,{'Tua Tagovailoa': 20.62},{'Jonathan Taylor': 10.8},{'Travis Etienne': 10.9},{'Deebo Samuel': 16.2},{'Amari Cooper': 2.6},{'Mark Andrews': 2.4},{'Aaron Jones': 17.9},{'Jake Moody': 26.6},...,71,2024,1241.44,9.6,bgmaddox,1,119.02,91.72,5,BMoreBallers88 [9.6%]
2,2,jlglover,{'Josh Allen': 35.18},{'Tyjae Spears': 5.2},{'Ezekiel Elliott': 11.9},{'Justin Jefferson': 13.9},{'Rashee Rice': 13.8},{'Cole Kmet': 0.9},{'DJ Moore': 7.5},{'Cameron Dicker': 11.2},...,71,2024,1241.44,10.1,RossLikeSauce,1,125.58,121.64,4,jlglover [10.1%]
3,3,RossLikeSauce,{'Patrick Mahomes': 16.64},{'Joe Mixon': 25.3},{'James Conner': 17.8},{'Ja'Marr Chase': 9.2},{'Puka Nacua': 6.2},{'Jake Ferguson': 3.0},{'Chris Godwin': 18.3},{'Ka'imi Fairbairn': 17.2},...,71,2024,1241.44,9.8,jlglover,0,121.64,125.58,7,RossLikeSauce [9.8%]
4,4,JTizzzzle,{'Joe Burrow': 7.06},{'Bijan Robinson': 13.6},{'Kyren Williams': 12.9},{'Stefon Diggs': 18.9},{'DK Metcalf': 4.4},{'Luke Musgrave': 0.0},{'Zay Flowers': 8.1},{'Jake Elliott': 9.9},...,71,2024,1241.44,6.0,InfiniteJesse,0,74.86,115.66,5,JTizzzzle [6.0%]


In [157]:
WeekWins.groupby('Week')['Total'].sum()

Week
1     1241.44
2     1260.56
3     1200.24
4     1262.82
5     1284.32
6     1344.28
7     1206.18
8     1357.12
9     1420.64
10    1196.94
11    1389.98
12    1298.88
Name: Total, dtype: float64

In [158]:
year = 2024
figBar = px.bar(WeekWins.sort_values('Total'), x='Week',y='Total',template = 'plotly_dark',color = "Team", barmode='stack',text='BarName', title = f'Week-by-Week Domination: {year}')
figBar.update_layout(barmode='stack', xaxis={'categoryorder':'total descending'})
figBar.update_layout(barcornerradius=5)
figBar.update_layout(legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
        title = ''
    ))
figBar.update_traces(textfont_size=10, textangle=0, cliponaxis=True,insidetextanchor="middle", textposition = 'inside', textfont=dict(family='Courier New',  # Font family
            size=12   # Font color
        ))
figBar.update_layout(width=1600, height=700)
figBar.update_layout(showlegend=False)

    # Customize the x-axis labels
figBar.update_xaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=16,         # Font size
            color='white'    # Font color
        ),
        title = None
    )

    # Customize the y-axis labels
figBar.update_yaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=18,         # Font size
            color='white'    # Font color
        ),
        title=None
    )
figBar.update_xaxes(dtick=1)
figBar.update_yaxes(dtick=200)
figBar.update_traces(width=.96)
figBar.update_traces(marker_line_width=2,marker_line_color='black')

figBar.show()

## Scoring Frequency

In [159]:
figScoring = px.histogram(Matches_2024DF, x='Total', template='plotly_dark',title = 'Scoring Frequency', marginal='rug', color = 'Team')
figScoring.update_layout(width=1200, height=800)
figScoring.update_layout(font_family="Courier New", title_font = dict(size=45))
figScoring.update_yaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=15,         # Font size
            color='white'    # Font color
        ),
        title = None,
        
    )
figScoring.update_xaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=15,         # Font size
            color='white'    # Font color
        ),
        title = None,
        dtick=10
    )
figScoring.update_traces(marker_line_width=2,marker_line_color='black')
figScoring.update_layout(legend=dict(font=dict(size= 20)
))
figScoring.update_layout(
       legend=dict(
           x=1.01,  # x-coordinate of the legend (0 to 1)
           y=0.5   # y-coordinate of the legend (0 to 1)
       )
   )
figScoring.show()

## Position Points YTD

In [160]:
Starters = Breakout_2024[Breakout_2024['starter'] == 1]
PositionPoints = Starters.groupby(['team','position'])['points'].sum()
PositionPoints = PositionPoints.reset_index()
PositionPoints = round(PositionPoints,1)

In [161]:
Starters

,team,matchup,player,points,starter,position,week_x,player_name,recent_teams,week_id,...,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr,week_id_NFL,Game_date_time,color,Score YTD,TotalYTD,Year
2,bgmaddox,1,Derrick Henry,10.60,1,RB,1,Derrick Henry,BAL,BAL-1,...,0.048783,0.0,10.60,10.600000,BAL-1,Thursday 8 PM,mediumpurple,10.60,233.10,2024
3,bgmaddox,1,Alvin Kamara,19.50,1,RB,1,Alvin Kamara,NO,NO-1,...,0.255288,0.0,17.00,22.000000,NO-1,Sunday 1 PM,mediumpurple,19.50,191.00,2024
4,bgmaddox,1,Younghoe Koo,3.40,1,K,1,Younghoe Koo,ATL,ATL-1,...,NaN,NaN,NaN,NaN,NaN,Sunday 1 PM,mediumpurple,3.40,99.90,2024
5,bgmaddox,1,Calvin Ridley,6.50,1,WR,1,Calvin Ridley,TEN,TEN-1,...,0.877972,0.0,5.00,8.000000,TEN-1,Sunday 1 PM,mediumpurple,6.50,68.60,2024
8,bgmaddox,1,D'Andre Swift,5.00,1,RB,1,D'Andre Swift,CHI,CHI-1,...,0.054995,0.0,5.00,5.000000,CHI-1,Sunday 1 PM,mediumpurple,5.00,139.80,2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
162,InfiniteJesse,3,David Montgomery,14.80,1,RB,12,David Montgomery,DET,DET-12,...,0.139846,0.0,13.30,16.299999,DET-12,Sunday 1 PM,magenta,168.42,168.42,2024
163,InfiniteJesse,3,CeeDee Lamb,11.80,1,WR,12,CeeDee Lamb,DAL,DAL-12,...,0.711205,0.0,6.80,16.799999,DAL-12,Sunday 1 PM,magenta,153.50,153.50,2024
168,InfiniteJesse,3,De'Von Achane,19.10,1,RB,12,De'Von Achane,MIA,MIA-12,...,NaN,NaN,NaN,NaN,NaN,Sunday 1 PM,magenta,167.28,167.28,2024
169,InfiniteJesse,3,C.J. Stroud,19.68,1,QB,12,C.J. Stroud,HOU,HOU-12,...,NaN,0.0,15.68,15.680000,HOU-12,Sunday 1 PM,magenta,192.20,192.20,2024


In [162]:
figPosTotals = px.bar(PositionPoints, x='points',y='team', color = 'position', template = 'plotly_dark', orientation='h', 
                      text='points',title = 'Points by Position',
                      category_orders={'position':['QB','RB','WR','TE','K','DEF']})
figPosTotals.update_yaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=15,         # Font size
            color='white'    # Font color
        ),
        title = None,
        categoryorder="total ascending",
        categoryarray = Week8GraphInfo['Difference']
    )
figPosTotals.update_layout(width=800, height=1200)
figPosTotals.update_layout(barcornerradius=13)
figPosTotals.update_layout(font_family="Courier New", title_font = dict(size=45))
figPosTotals.update_traces(textfont_size=18, textangle=0, cliponaxis=True, textposition = 'inside', textfont=dict(weight='bold',family='Courier New',  # Font family
            size=18   # Font color
        ))
figPosTotals.update_traces(marker_line_width=2,marker_line_color='black')
    
figPosTotals.update_traces(insidetextanchor= 'middle')
figPosTotals.update_layout(
        uniformtext_minsize=14,
        uniformtext_mode='hide'
        )
# Update the legend title
figPosTotals.update_layout(
    legend=dict(
        title=dict(
            text=None,
            font=dict(
                size=25,
                
            )
        )
    )
)  
figPosTotals.update_layout(legend=dict(title_font_family="Courier New",
                              font=dict(size= 20)
))  
figPosTotals.update_xaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=20,         # Font size
            color='white'    # Font color
        ),
        title = None
    )
figPosTotals.update_yaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=20,         # Font size
            color='white'    # Font color
        ),
        title=None
    )
figPosTotals.update_layout(legend=dict(orientation="h"))
# Set legend location
figPosTotals.update_layout(
    legend=dict(
        x=0.45,  # x-coordinate of the legend (0 to 1, where 0 is left and 1 is right)
        y=1.03,   # y-coordinate of the legend (0 to 1, where 0 is bottom and 1 is top)
        xanchor="center",  # horizontal anchor point ('left', 'center', 'right')
        yanchor="top"    # vertical anchor point ('top', 'middle', 'bottom')
    )
)

figPosTotals.show()


NameError: name 'Week8GraphInfo' is not defined

## Violin Plots

In [163]:
RBOnly = Breakout_2024[Breakout_2024['position'] == 'RB']

In [164]:
RBViolin = RBOnly.groupby(['team','week_x'])['points'].sum().reset_index()

In [165]:
RBViolin

,team,week_x,points
0,BMoreBallers88,1,39.6
1,BMoreBallers88,2,34.1
2,BMoreBallers88,3,49.3
3,BMoreBallers88,4,39.3
4,BMoreBallers88,5,14.9
...,...,...,...
139,sgmaddox,8,63.2
140,sgmaddox,9,51.9
141,sgmaddox,10,31.1
142,sgmaddox,11,45.0


In [166]:
figViolin = px.violin(Breakout_2024,x='points', y='position',facet_col='team',facet_col_wrap=3, color = 'position', template='plotly_dark')
figViolin.update_traces(orientation='h', side='positive', width=3, points=False, spanmode = 'hard')
figViolin.update_layout(width=800, height=1200)
figViolin.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
figViolin.for_each_xaxis(lambda xaxis: xaxis.update(showticklabels=True))
figViolin.update_layout(font_family="Courier New", title_font = dict(size=45))
figViolin.show()


In [167]:
figViolin2 = px.violin(Breakout_2024[Breakout_2024['starter'] == 1],x='points', y='team',facet_col='position',facet_col_wrap=2, color = 'team', 
                       template='plotly_dark',category_orders={
                           "position": ["QB", "RB", "WR", "TE", "K", "DEF"],
                              })
figViolin2.update_traces(orientation='h', side='positive', width=3, points=False, spanmode='hard')
figViolin2.update_layout(width=800, height=1200, showlegend=False,)
figViolin2.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
figViolin2.for_each_xaxis(lambda xaxis: xaxis.update(showticklabels=True))
figViolin2.update_layout(font_family="Courier New", title_font = dict(size=55))
figViolin2.update_yaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=13,         # Font size
            color='white'    # Font color
        ),
        title=None
    )
figViolin2.update_xaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=13,         # Font size
            color='white'    # Font color
        ),
        title=None
    )
figViolin2.update_annotations(font_size=20)
figViolin2.update_layout(
    title=dict(text="<b>Positional Points Distribution</b>", font=dict(size=25), automargin=False)
)

figViolin2.show()

## Score Area Graph

In [556]:
Week142021Pivot,Week142021Wins = WeeklyWins(Matches_2024,17)

In [557]:
AreaOrder = WeekWins.groupby('Team')['Won'].sum()
AreaOrder = AreaOrder.reset_index().sort_values('Won', ascending=True)
AreaOrder_rev = AreaOrder.reset_index().sort_values('Won', ascending=False)

AreaOrder_List = list(AreaOrder['Team'])
AreaOrder_List_rev = list(AreaOrder_rev['Team'])


In [558]:
df = Week142021Wins
year = 2024
fig = px.area(df, x="Week", y="PercentTotal", color="Team",template = 'plotly_dark', category_orders={'Team':AreaOrder_List}, title = f'Week-by-Week Domination: {year}')
fig.update_layout(width=1200, height=800)
fig.update_xaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=16,         # Font size
            color='white'    # Font color
        ),
        title = None
    )

    # Customize the y-axis labels
fig.update_yaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=18,         # Font size
            color='white'    # Font color
        ),
        title=None
    )
df_lastweek = df[df['Week'] == 14]
fig.update_xaxes(dtick=1)
fig.update_layout(
    legend=dict(
        x=1.05,
        y=.5,
        traceorder="reversed",
        title_font_family="Courier New",
        title = 'Team',
        font=dict(
            family="Courier New",
            size=25,
            color="white"
        )#,
        #bgcolor="LightSteelBlue",
        #bordercolor="Black",
        #borderwidth=2
    )
)
fig.update_layout(title=dict(
            font=dict(
                size=50,
                family="Courier New")))  
fig.update_layout(yaxis_ticksuffix = '%')
fig.show()

## Power Rankings

In [171]:
Matches_2024DF['Week Rank'] = Matches_2024DF.groupby('Week')['Total'].rank() - 1

In [172]:
Matches_2024DF['Power YTD'] = Matches_2024DF.groupby('Team')['Week Rank'].cumsum().astype(int)

In [173]:
Matches_2024DF['Power Loss YTD'] = (Matches_2024DF['Week'] * 11 - Matches_2024DF['Power YTD']).astype(int)

In [174]:
Matches_2024DF['Win Percentage'] = round(((Matches_2024DF['Power YTD']/(Matches_2024DF['Week'] * 11)) * 100),2)

In [175]:
Matches_2024DF[Matches_2024DF['Week'] == 11]

,Team,QB,RB1,RB2,WR1,WR2,TE,WRT,K,DEF,...,Week,Week Index,Year,LeagueTotal,PercentTotal,Opp_team,Week Rank,Power YTD,Power Loss YTD,Win Percentage
0,bgmaddox,{'Jalen Hurts': 18.74},{'Derrick Henry': 10.5},{'Alvin Kamara': 10.9},{'Amon-Ra St. Brown': 33.2},{'Cedric Tillman': 6.2},{'Dawson Knox': 6.0},{'Nick Chubb': 5.0},{'Younghoe Koo': 9.2},{'DET': 10.0},...,11,81,2024,1389.98,7.9,sgmaddox,5.0,77,44,63.64
1,sgmaddox,{'Jordan Love': 22.24},{'Josh Jacobs': 21.4},{'Rhamondre Stevenson': 9.9},{'A.J. Brown': 9.0},{'Khalil Shakir': 11.0},{'Will Dissly': 16.0},{'Brian Robinson': 13.7},{'Justin Tucker': 6.4},{'MIA': 6.0},...,11,81,2024,1389.98,8.3,bgmaddox,6.0,55,66,45.45
2,BMoreBallers88,{'Tua Tagovailoa': 28.52},{'Jonathan Taylor': 6.5},{'Aaron Jones': 3.8},{'Deebo Samuel': 4.1},{'Amari Cooper': 6.5},{'Mark Andrews': 3.2},{'Jordan Addison': 13.6},{'Jake Moody': 5.3},{'HOU': 17.0},...,11,81,2024,1389.98,6.4,eegrady,0.0,45,76,37.19
3,eegrady,{'Jayden Daniels': 15.44},{'Jahmyr Gibbs': 18.8},{'Saquon Barkley': 32.8},{'Jakobi Meyers': 6.8},{'Michael Pittman': 7.1},{'T.J. Hockenson': 2.3},{'George Kittle': 0.0},{'Jason Sanders': 11.7},{'BUF': 8.0},...,11,81,2024,1389.98,7.4,BMoreBallers88,3.0,52,69,42.98
4,RossLikeSauce,{'Patrick Mahomes': 21.84},{'Joe Mixon': 34.3},{'Audric Estime': 4.0},{'Ja'Marr Chase': 23.0},{'Puka Nacua': 21.8},{'Tucker Kraft': 0.0},{'Ladd McConkey': 15.3},{'Ka'imi Fairbairn': 10.2},{'SF': 9.0},...,11,81,2024,1389.98,10.0,InfiniteJesse,10.0,87,34,71.90
5,InfiniteJesse,{'C.J. Stroud': 10.88},{'David Montgomery': 23.0},{'De'Von Achane': 18.5},{'Cooper Kupp': 25.6},{'CeeDee Lamb': 14.6},{'Cole Kmet': 5.7},{'Jayden Reed': 9.44},{'Jake Bates': 12.4},{'NYJ': 3.0},...,11,81,2024,1389.98,8.9,RossLikeSauce,7.0,54,67,44.63
6,jlglover,{'Josh Allen': 25.98},{'Travis Etienne': 4.8},{'Kareem Hunt': 6.0},{'George Pickens': 12.9},{'Justin Jefferson': 11.1},{'Dallas Goedert': 7.6},{'Tank Dell': 7.8},{'Chris Boswell': 25.0},{'LAC': -2.0},...,11,81,2024,1389.98,7.1,JTizzzzle,2.0,44,77,36.36
7,JTizzzzle,{'Joe Burrow': 35.04},{'Bijan Robinson': 8.3},{'Kyren Williams': 8.6},{'DK Metcalf': 10.5},{'Xavier Worthy': 14.8},{'Dalton Schultz': 5.8},{'Zay Flowers': 10.9},{'Jake Elliott': 6.2},{'PHI': 9.0},...,11,81,2024,1389.98,7.9,jlglover,4.0,60,61,49.59
8,jhuntmadd,{'Jared Goff': 42.58},{'Breece Hall': 27.6},{'Kenneth Walker': 12.9},{'Davante Adams': 10.2},{'DeAndre Hopkins': 4.4},{'Brock Bowers': 24.8},{'Jaxon Smith-Njigba': 16.8},{'Evan McPherson': 8.3},{'MIN': 12.0},...,11,81,2024,1389.98,11.5,RReclam,11.0,68,53,56.20
9,RReclam,{'Bo Nix': 36.78},{'Christian McCaffrey': 12.6},{'James Cook': 17.2},{'Jauan Jennings': 20.1},{'Calvin Ridley': 7.8},{'Evan Engram': 5.3},{'Nico Collins': 7.4},{'John Parker Romo': 5.0},{'DEN': 12.0},...,11,81,2024,1389.98,8.9,jhuntmadd,8.0,53,68,43.80


# Weekly Side Bets

## Week 1

In [176]:
def Week1Graph(df, Week,top):
    
    
    
    df = df.sort_values('Total', ascending = False)
    
    
    #points = df.drop(axis = 1,columns = ['Total','Won','Week','Opp','Matchup']).applymap(lambda x: float(list(x.values())[0]))
    points = df.applymap(lambda x: float(list(x.values())[0]) if isinstance(x, dict) else x)
    points = points.round(2).reset_index()
    
    default_color = "blue"
    colors = {top: "red"}
    
    #points = df.sort_values('Total', ascending = False)
    team_list = df.sort_values('Total', ascending = True).index.tolist()
    
    #Ranked = df.index.sort_values('Total', ascending = True).unique()
    SizeZip = dict(zip(team_list,range(12,30)))
    print(SizeZip)

    color_discrete_map = {
        c: colors.get(c, default_color) 
        for c in team_list}


    fig1 = px.bar(points, y='Team',x=position_list,template = 'plotly_dark',color = "Team", text_auto=True, title = f'Week {Week} Side Bet', orientation='h',color_discrete_map=color_discrete_map)
    
    #Update the layout to hide the legend:
    fig1.update(layout_coloraxis_showscale=False)
    
    fig1.update_layout(barcornerradius=13)
    # Adjust the figure size
    fig1.update_layout(width=800, height=1200)
    fig1.update_layout(title=dict(
            font=dict(
                size=50,
                family="Courier New")))  # Set the width and height in pixels
    fig1.update_traces(textfont_size=12, textangle=0, cliponaxis=True, textposition = 'auto', textfont=dict(weight='bold',family='Courier New',  # Font family
            size=12   # Font color
        ))
    
    # Customize the x-axis labels
    fig1.update_xaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=16,         # Font size
            color='white'    # Font color
        ),
        title = None
    )

    # Customize the y-axis labels
    fig1.update_yaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=18,         # Font size
            color='white'    # Font color
        ),
        title=None
    )

    fig1.update(layout_coloraxis_showscale=False)
    fig1.update_layout(showlegend=False)
    fig1.update_traces(insidetextanchor= 'middle')
    
    # Add image
    fig1.add_layout_image(
        dict(
            source=r"C:\Users\Brett\Downloads\CROWN.png",
            xref="paper", yref="paper",
            x=.5, y=.5,
            sizex=0.2, sizey=0.2
            #xanchor="right", yanchor="bottom"
            )
                        )
    # Add annotations
    annotations = []
    for i, val in enumerate(points.Total):
        annotations.append(
            dict(
                x=val+10, 
                y=points.Team[i], 
                text=str(val), 
                xanchor='left', 
                yanchor='middle', 
                showarrow=False, 
                font=dict(size=SizeZip[points.Team[i]],family='Courier New',)
                )   
                        )

    fig1.update_layout(annotations=annotations)

    fig1.update_traces(marker_line_width=1.5,marker_line_color='black')
    
    fig1.update_traces(insidetextanchor= 'middle')
    fig1.update_layout(
        uniformtext_minsize=12,
        uniformtext_mode='hide'
        )
    fig1.add_annotation(
    text="Bar Order: QB ------------> DEF",
    xref="paper", yref="paper",
    x=0.05, y=-.06, # Position relative to figure (right side, middle)
    showarrow=False,
    font=dict(
        family="Courier New, monospace",
        size=15,
        color="white"
    )
    )
    fig1.show()
    return fig1

In [177]:
Week1Graph(Week1NoMatches_2024,1,'jlglover')

{'JTizzzzle': 12, 'eegrady': 13, 'RReclam': 14, 'bgmaddox': 15, 'sgmaddox': 16, 'jhuntmadd': 17, 'RascalHazard': 18, 'InfiniteJesse': 19, 'BMoreBallers88': 20, 'RossLikeSauce': 21, 'DirtyCommie': 22, 'jlglover': 23}


In [178]:
Week1NoMatches_2024 = Week1NoMatches_2024.sort_values('Total')
team_list = Week1NoMatches_2024.index.tolist()
team_list

['JTizzzzle',
 'eegrady',
 'RReclam',
 'bgmaddox',
 'sgmaddox',
 'jhuntmadd',
 'RascalHazard',
 'InfiniteJesse',
 'BMoreBallers88',
 'RossLikeSauce',
 'DirtyCommie',
 'jlglover']

## Week 2

In [ ]:
Week2 = weekly[weekly['week'] == 2]
Week2 = WeeklyNFLData_24[WeeklyNFLData_24['week'] == 2]
td_cols = [col for col in Week2.columns if 'tds' in col.lower()]
td_cols.append('player_display_name')

Week2filtered= week2Breakout2024[td_cols]
#Week2Data = week2Breakout2024.merge(Week2filtered,left_on='player', right_on='player_display_name')
Week2Data.loc[9,'rushing_tds'] = 1
Week2Data = Week2Data[Week2Data['starter']==1]

Week2Data['Total']= Week2Data['passing_tds'] + Week2Data['rushing_tds'] + Week2Data['receiving_tds'] + Week2Data['special_teams_tds']

Week2Totals = Week2Data.groupby('team').agg('sum')

Week2Groups = Week2Data.groupby('team')
Week2Groups.get_group('bgmaddox').sort_values('position')

Week2Totals.sort_values('Total', ascending=False)
Week2Totals.sort_values('Total', ascending=False)
Week2Totals = Week2Totals.sort_values('Total', ascending=False).reset_index()
Week2Data = Week2Data.sort_values(['player'], ascending=False)

### Graph

In [ ]:
fig = px.bar(Week2Data,y='team',x=['passing_tds_x','rushing_tds_x','receiving_tds_x','special_teams_tds_x'],color='position' ,template = 'plotly_dark',title = f'Week 2 Side Bet: TD Party!', orientation='h',text='Total')
fig.update_layout(width=800, height=1200)
fig.update_traces(insidetextanchor= 'middle',textfont=dict(
        family="Courier New",
        size=25))
fig.update_layout(title=dict(
            font=dict(
                size=35,
                family="Courier New")))
fig.update_layout(barcornerradius=13)
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.update_layout(
    legend=dict(
        #x=0,
        #y=1,
        #traceorder="reversed",
        title_font_family="Courier New",
        title = 'Position',
        font=dict(
            family="Courier New",
            size=25,
            color="white"
        )#,
        #bgcolor="LightSteelBlue",
        #bordercolor="Black",
        #borderwidth=2
    )
)
fig.update_layout(
        xaxis_title="",  # Set x-axis title
        yaxis_title="")
fig.update_xaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=16,         # Font size
            color='white'    # Font color
        ),
        title = None
    )

    # Customize the y-axis labels
fig.update_yaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=18,         # Font size
            color='white'    # Font color
        ),
        title=None
    )
fig.update_traces(marker_line_width=2,marker_line_color='black')
fig.show()


In [ ]:
def Week2Graph(df, Week,top):
    
    fig = px.bar(df,x)
    return fig

## Week 3

In [184]:
def Week3Graph(df, Week,top):
    
    df = df.sort_values('points', ascending = True)
    #points = df.drop(axis = 1,columns = ['Total','Won','Week','Opp','Matchup']).applymap(lambda x: float(list(x.values())[0]))
    #points = points.round(2).reset_index()
    
    default_color = "blue"
    colors = {top: "red"}

    team_list = df['team'].unique()

    color_discrete_map = {
        c: colors.get(c, default_color) 
        for c in team_list}


    fig1 = px.bar(df, y='team',x='points',template = 'plotly_dark',color = "team",text ='player', title = f'Week {Week} Side Bet', orientation='h',color_discrete_map=color_discrete_map)
    
    #Update the layout to hide the legend:
    fig1.update(layout_coloraxis_showscale=False)
    
    fig1.update_layout(barcornerradius=13)
    # Adjust the figure size
    fig1.update_layout(width=800, height=1200)
    fig1.update_layout(title=dict(
            font=dict(
                size=50,
                family="Courier New")))  # Set the width and height in pixels
    fig1.update_traces(textfont_size=12, textangle=0, cliponaxis=True, textposition = 'auto', textfont=dict(weight='bold',family='Courier New',  # Font family
            size=12   # Font color
        ))
    
    # Customize the x-axis labels
    fig1.update_xaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=16,         # Font size
            color='white'    # Font color
        ),
        title = None
    )

    # Customize the y-axis labels
    fig1.update_yaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=18,         # Font size
            color='white'    # Font color
        ),
        title=None
    )
    fig1.add_annotation(
    text="JAX >",
    xref="paper", yref="paper",
    x=0.1, y=.885, # Position relative to figure (right side, middle)
    showarrow=False,
    font=dict(
        family="Courier New, monospace",
        size=12,
        color="white"
    )
    )
    
    fig1.add_annotation(
    text="PHI >",
    xref="paper", yref="paper",
    x=0.1, y=.8, # Position relative to figure (right side, middle)
    showarrow=False,
    font=dict(
        family="Courier New, monospace",
        size=12,
        color="white"
    )
    )
    
    fig1.update(layout_coloraxis_showscale=False)
    fig1.update_layout(showlegend=False)
    
    


    fig1.show()
    return fig1

In [185]:
week4Breakout2024

,team,matchup,player,points,starter,position,week_x,player_name,recent_teams,week_id,...,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr,week_id_NFL,Game_date_time,color
0,bgmaddox,1,Carson Steele,-0.9,0,RB,4,Carson Steele,KC,KC-4,...,NaN,0.107143,0.000000,0.160714,0.0,-1.400000,-0.400000,KC-4,Sunday 4 PM,mediumpurple
1,bgmaddox,1,Bucky Irving,12.0,0,RB,4,Bucky Irving,TB,TB-4,...,-1.500000,0.042553,-0.013245,0.054558,0.0,11.500000,12.500000,TB-4,Sunday 1 PM,mediumpurple
2,bgmaddox,1,Derrick Henry,33.4,1,RB,4,Derrick Henry,BAL,BAL-4,...,10.000000,0.166667,0.007812,0.255469,0.0,32.900002,35.900002,BAL-4,Sunday 8 PM,mediumpurple
3,bgmaddox,1,Demarcus Robinson,4.7,0,WR,4,Demarcus Robinson,LA,LA-4,...,0.822222,0.103448,0.223881,0.311889,0.0,3.700000,5.700000,LA-4,Sunday 1 PM,mediumpurple
4,bgmaddox,1,Alvin Kamara,21.4,1,RB,4,Alvin Kamara,NO,NO-4,...,3.230769,0.250000,0.067358,0.422150,0.0,17.900000,24.900000,NO-4,Sunday 1 PM,mediumpurple
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
172,InfiniteJesse,2,Javonte Williams,9.0,0,RB,4,Javonte Williams,DEN,DEN-4,...,0.428571,0.120000,0.027668,0.199368,0.0,8.000000,10.000000,DEN-4,Sunday 1 PM,magenta
173,InfiniteJesse,2,De'Von Achane,4.4,1,RB,4,De'Von Achane,MIA,MIA-4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Monday 7 PM,magenta
174,InfiniteJesse,2,C.J. Stroud,27.5,1,QB,4,C.J. Stroud,HOU,HOU-4,...,NaN,NaN,NaN,NaN,0.0,23.500000,23.500000,HOU-4,Sunday 1 PM,magenta
175,InfiniteJesse,2,BAL,10.0,0,DEF,4,BAL,BAL,BAL-4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sunday 8 PM,magenta


In [186]:
Week3SideBet = week4Breakout2024[week4Breakout2024['position']=='DEF']
Week3SideBet = Week3SideBet.sort_values('points')

In [187]:
Week3SideBet

,team,matchup,player,points,starter,position,week_x,player_name,recent_teams,week_id,...,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr,week_id_NFL,Game_date_time,color
115,jhuntmadd,4,BUF,-4.0,1,DEF,4,BUF,BUF,BUF-4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sunday 8 PM,orange
102,DirtyCommie,3,JAX,0.0,0,DEF,4,JAX,JAX,JAX-4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sunday 1 PM,lime
43,JTizzzzle,1,PHI,0.0,1,DEF,4,PHI,PHI,PHI-4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sunday 1 PM,lightsalmon
132,RReclam,5,PIT,1.0,1,DEF,4,PIT,PIT,PIT-4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sunday 1 PM,green
28,jlglover,4,HOU,2.0,1,DEF,4,HOU,HOU,HOU-4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sunday 1 PM,cyan
29,jlglover,4,NE,2.0,0,DEF,4,NE,NE,NE-4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sunday 4 PM,cyan
116,jhuntmadd,4,MIN,3.0,0,DEF,4,MIN,MIN,MIN-4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sunday 1 PM,orange
57,RascalHazard,6,MIA,4.0,1,DEF,4,MIA,MIA,MIA-4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Monday 7 PM,pink
14,bgmaddox,1,DAL,5.0,1,DEF,4,DAL,DAL,DAL-4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Thursday 8 PM,mediumpurple
87,eegrady,5,KC,7.0,1,DEF,4,KC,KC,KC-4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sunday 4 PM,teal


In [188]:
Week3Graph(Week3SideBet,4,'jhuntmadd')

## Week 4

In [189]:
Week4SideBet = week3Breakout2024[week3Breakout2024['points'] <= 21]
Week4SideBet = Week4SideBet[Week4SideBet['points'] >= 17]
Week4SideBet = Week4SideBet.sort_values('points', ascending = False)

In [190]:
Week4SideBet=Week4SideBet.head(15).sort_values('points', ascending = False)

### Graph

In [191]:
def Week4Graph(df, Week,top):
    df['string_pts'] = df['points'].astype(str)
    df['team-points'] = df['team'] + " (" + df['string_pts'] + ')'
    df = df.sort_values('points', ascending = True)
    #points = df.drop(axis = 1,columns = ['Total','Won','Week','Opp','Matchup']).applymap(lambda x: float(list(x.values())[0]))
    #points = points.round(2).reset_index()
    
    default_color = "blue"
    colors = {top: "red"}

    team_list = df['team'].unique()

    color_discrete_map = {
        c: colors.get(c, default_color) 
        for c in team_list}


    fig1 = px.bar(df, y='player',x='points',template = 'plotly_dark', color = 'team', text='team-points', #text_auto=True, 
                    title = f'Week {Week} Side Bet: Blackjack', orientation='h' #,color_discrete_map=color_discrete_map
                   )
    
    #Update the layout to hide the legend:
    fig1.update(layout_coloraxis_showscale=False)
    
    fig1.update_layout(barcornerradius=13)
    # Adjust the figure size
    fig1.update_layout(width=800, height=1200)
    fig1.update_layout(title=dict(
            font=dict(
                size=35,
                family="Courier New")))  # Set the width and height in pixels
    fig1.update_traces(textfont_size=12, textangle=0, cliponaxis=True, textposition = 'inside', textfont=dict(weight='bold',family='Courier New',  # Font family
            size=12   # Font color
        ))
    fig1.update_layout(yaxis={'categoryorder':'total ascending'})
    # Customize the x-axis labels
    fig1.update_xaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=16,         # Font size
            color='white'    # Font color
        ),
        title = None
    )

    # Customize the y-axis labels
    fig1.update_yaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=18,         # Font size
            color='white'    # Font color
        ),
        title=None
    )

    #fig1.update(layout_coloraxis_showscale=False)
    fig1.update_layout(showlegend=False)
    
    fig1.update_layout(legend=dict(
    orientation="h",
    #yanchor="bottom",
    #y=-.1,
    xanchor="center",
    x=.5,
    title = ""
))
    fig1.add_vline(x=21)


    fig1.show()
    return fig1

In [192]:
Week4Graph(Week4SideBet,3,'bgmaddox')

## Week 5

In [193]:
week5Breakout2024.groupby(['team','starter'])['points'].count()

team            starter
BMoreBallers88  0          6
                1          9
DirtyCommie     0          7
                1          9
InfiniteJesse   0          5
                1          9
JTizzzzle       0          5
                1          9
RReclam         0          6
                1          9
RascalHazard    0          6
                1          9
RossLikeSauce   0          6
                1          9
bgmaddox        0          6
                1          9
eegrady         0          6
                1          9
jhuntmadd       0          6
                1          9
jlglover        0          6
                1          9
sgmaddox        0          6
                1          9
Name: points, dtype: int64

In [194]:
Week5SideBetdf = week5Breakout2024[week5Breakout2024['starter']==0]
Week5List = Week5SideBetdf.groupby('team')['points'].sum().reset_index()
Week5List.sort_values('points')
figWeek5 = px.bar(Week5SideBetdf, x='points',y='team',template = 'plotly_dark',title='Week 5: The Replacements', color = 'position', orientation='h' )
figWeek5.update_layout(barmode='stack', yaxis={'categoryorder':'total ascending'})
figWeek5.update_layout(barcornerradius=13)
    # Adjust the figure size
figWeek5.update_layout(width=1200, height=800)
figWeek5.update_layout(title=dict(
            font=dict(
                size=35,
                family="Courier New")))  # Set the width and height in pixels
figWeek5.update_traces(textfont_size=12, textangle=0, cliponaxis=True, textposition = 'inside', textfont=dict(weight='bold',family='Courier New',  # Font family
            size=12   # Font color
        ))
# Update the line thickness
figWeek5.update_traces(marker_line_width=2.5,marker_line_color='black')
figWeek5.update_layout(
        xaxis_title="Points",  # Set x-axis title
        yaxis_title="Teams",   # Set y-axis title
        xaxis=dict(
            title_font=dict(
                family="Courier New",  # Set font family for x-axis title
                size=30,          # Set font size for x-axis title
                color ='red',
                weight = 'bold'
            )
        ),
        yaxis=dict(
            title_font=dict(
                family="Courier New",  # Set font family for y-axis title
                size=30,          # Set font size for y-axis title
                color = 'green',
                weight = 'bold'
            )
        )
    )
figWeek5.update_layout(
    legend=dict(
        #x=0,
        #y=1,
        #traceorder="reversed",
        title_font_family="Courier New",
        title = 'Position',
        font=dict(
            family="Courier New",
            size=25,
            color="white"
        )#,
        #bgcolor="LightSteelBlue",
        #bordercolor="Black",
        #borderwidth=2
    )
)

figWeek5.update_xaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=16,         # Font size
            color='white'    # Font color
        ),
        title = None
    )

    # Customize the y-axis labels
figWeek5.update_yaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=18,         # Font size
            color='white'    # Font color
        ),
        title=None
    )
figWeek5.show()

In [198]:
def Week5Graph(df, Week,top):
    
    df = df.sort_values('Total', ascending = False)
    points = df.drop(axis = 1,columns = ['Total','Won','Week','Opp','Matchup']).applymap(lambda x: float(list(x.values())[0]))
    points = points.round(2).reset_index()
    
    default_color = "blue"
    colors = {top: "red"}

    team_list = points['Team'].unique()

    color_discrete_map = {
        c: colors.get(c, default_color) 
        for c in team_list}


    fig1 = px.bar(points, y='Team',x=position_list,template = 'plotly_dark',color = "Team", text_auto=True, title = f'Week {Week} Side Bet', orientation='h',color_discrete_map=color_discrete_map)
    
    #Update the layout to hide the legend:
    fig1.update(layout_coloraxis_showscale=False)
    
    fig1.update_layout(barcornerradius=13)
    # Adjust the figure size
    fig1.update_layout(width=800, height=1200)
    fig1.update_layout(title=dict(
            font=dict(
                size=50,
                family="Courier New")))  # Set the width and height in pixels
    fig1.update_traces(textfont_size=12, textangle=0, cliponaxis=True, textposition = 'auto', textfont=dict(weight='bold',family='Courier New',  # Font family
            size=12   # Font color
        ))
    
    # Customize the x-axis labels
    fig1.update_xaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=16,         # Font size
            color='white'    # Font color
        ),
        title = None
    )

    # Customize the y-axis labels
    fig1.update_yaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=18,         # Font size
            color='white'    # Font color
        ),
        title=None
    )

    fig1.update(layout_coloraxis_showscale=False)
    fig1.update_layout(showlegend=False)
    fig1.update_layout(barcornerradius=13)
    # Adjust the figure size
    fig1.update_layout(width=800, height=1200)
    fig1.update_layout(title=dict(
            font=dict(
                size=35,
                family="Courier New")))  # Set the width and height in pixels
    fig1.update_traces(textfont_size=12, textangle=0, cliponaxis=True, textposition = 'inside', textfont=dict(weight='bold',family='Courier New',  # Font family
            size=12   # Font color
        ))
    


    fig1.show()
    return fig1

## Week 6

In [199]:
Week6df = Week6NoMatches_2024.reset_index()

In [200]:
figWeek6 = px.bar(Week6df, x='Margin',y='Team',template = 'plotly_dark',title='Week 6: Biggest Margin', color = 'Matchup', orientation='h', text='Team' )
figWeek6.update_layout(barmode='stack', yaxis={'categoryorder':'total ascending'})
figWeek6.update_layout(barcornerradius=13)
figWeek6.update_layout(width=1200, height=800)
figWeek6.update_layout(title=dict(
            font=dict(
                size=35,
                family="Courier New")))  # Set the width and height in pixels
figWeek6.update_traces(textfont_size=20, textangle=0, cliponaxis=True, textposition = 'auto', textfont=dict(weight='bold',family='Courier New',  # Font family
            size=15   # Font color
        ))
figWeek6.update_layout(
        xaxis_title="Margin",  # Set x-axis title
        yaxis_title="",   # Set y-axis title
        xaxis=dict(
            title_font=dict(
                family="Courier New",  # Set font family for x-axis title
                size=30,          # Set font size for x-axis title
                color ='red',
                weight = 'bold'
            )
        ),
        yaxis=dict(
            title_font=dict(
                family="Courier New",  # Set font family for y-axis title
                size=30,          # Set font size for y-axis title
                color = 'green',
                weight = 'bold'
            )
        )
    )
figWeek6.update_layout(yaxis=dict(showticklabels=False))
#Update the layout to hide the legend:
figWeek6.update(layout_coloraxis_showscale=False)
figWeek6.update_xaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=22,         # Font size
            color='white'    # Font color
        ),
        title = None
    )
figWeek6.show()

## Week 7

In [201]:
WeeklyNFLData_24

,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,...,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr,week_id,player_week_id
0,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,NYJ,2024,1,REG,...,0,NaN,NaN,NaN,NaN,0.0,8.580000,8.580000,NYJ-1,Aaron Rodgers - 1
1,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,NYJ,2024,2,REG,...,0,NaN,NaN,NaN,NaN,0.0,15.140000,15.140000,NYJ-2,Aaron Rodgers - 2
2,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,NYJ,2024,3,REG,...,0,NaN,NaN,NaN,NaN,0.0,21.040001,21.040001,NYJ-3,Aaron Rodgers - 3
3,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,NYJ,2024,4,REG,...,0,NaN,NaN,NaN,NaN,0.0,11.600000,11.600000,NYJ-4,Aaron Rodgers - 4
4,00-0023459,A.Rodgers,Aaron Rodgers,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,NYJ,2024,5,REG,...,0,NaN,NaN,NaN,NaN,0.0,11.760000,11.760000,NYJ-5,Aaron Rodgers - 5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4696,00-0039921,T.Benson,Trey Benson,RB,RB,https://static.www.nfl.com/image/upload/f_auto...,ARI,2024,10,REG,...,0,-3.125,0.083333,-0.072072,0.074550,0.0,8.700000,10.700000,ARI-10,Trey Benson - 10
4697,00-0039921,T.Benson,Trey Benson,RB,RB,https://static.www.nfl.com/image/upload/f_auto...,ARI,2024,12,REG,...,0,NaN,NaN,NaN,NaN,0.0,1.800000,1.800000,ARI-12,Trey Benson - 12
4698,00-0039921,T.Benson,Trey Benson,RB,RB,https://static.www.nfl.com/image/upload/f_auto...,ARI,2024,13,REG,...,0,NaN,NaN,NaN,NaN,0.0,2.000000,2.000000,ARI-13,Trey Benson - 13
4699,00-0039921,T.Benson,Trey Benson,RB,RB,https://static.www.nfl.com/image/upload/f_auto...,ARI,2024,14,REG,...,0,-0.800,0.026316,-0.031847,0.017181,0.0,1.900000,2.900000,ARI-14,Trey Benson - 14


In [203]:
#Week7Data = week7Breakout2024.merge(Week7,left_on='player', right_on='player_display_name')

In [204]:
Week7Data

,team,matchup,player,points,starter,position,week_x,player_name,recent_teams,week_id,...,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr,week_id_NFL,Game_date_time,color
0,bgmaddox,1,Bucky Irving,15.20,0,RB,7,Bucky Irving,TB,TB-7,...,10.800000,0.066667,0.012755,0.108929,0.0,13.700000,16.700001,TB-7,Monday 8 PM,mediumpurple
1,bgmaddox,1,Derrick Henry,24.70,1,RB,7,Derrick Henry,BAL,BAL-7,...,3.250000,0.047619,0.022727,0.087338,0.0,24.200001,25.200001,BAL-7,Monday 8 PM,mediumpurple
2,bgmaddox,1,Demarcus Robinson,1.40,0,WR,7,Demarcus Robinson,LA,LA-7,...,0.169811,0.136364,0.375887,0.467666,0.0,0.900000,1.900000,LA-7,Sunday 4 PM,mediumpurple
3,bgmaddox,1,Alvin Kamara,5.40,1,RB,7,Alvin Kamara,NO,NO-7,...,-14.000000,0.179487,-0.003650,0.266676,0.0,2.400000,8.400000,NO-7,Thursday 8 PM,mediumpurple
4,bgmaddox,1,Younghoe Koo,2.00,1,K,7,Younghoe Koo,ATL,ATL-7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sunday 1 PM,mediumpurple
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168,InfiniteJesse,6,Tutu Atwell,8.10,1,WR,7,Tutu Atwell,LA,LA-7,...,1.821429,0.409091,0.198582,0.752643,0.0,5.100000,11.100000,LA-7,Sunday 4 PM,magenta
169,InfiniteJesse,6,Javonte Williams,24.60,1,RB,7,Javonte Williams,DEN,DEN-7,...,-2.300000,0.120000,-0.057803,0.139538,0.0,23.100000,26.100000,DEN-7,Thursday 8 PM,magenta
170,InfiniteJesse,6,De'Von Achane,9.50,1,RB,7,De'Von Achane,MIA,MIA-7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sunday 1 PM,magenta
171,InfiniteJesse,6,C.J. Stroud,5.34,1,QB,7,C.J. Stroud,HOU,HOU-7,...,NaN,NaN,NaN,NaN,0.0,5.340000,5.340000,HOU-7,Sunday 1 PM,magenta


In [202]:
Week7Data = week7Breakout2024
Week7Groups = Week7Data.groupby('team')['rushing_yards'].sum()
Week7Groups = Week7Groups.reset_index()
figWeek7 = px.bar(Week7Data,y='team',x='rushing_yards',color='position' ,template = 'plotly_dark',title = f'Week 7 Side Bet: Rush Week', orientation='h', text='player_display_name')
figWeek7.update_layout(width=800, height=1200)
figWeek7.update_traces(insidetextanchor= 'middle',textfont=dict(
        family="Courier New",
        size=25))
figWeek7.update_layout(title=dict(
            font=dict(
                size=35,
                family="Courier New")))
figWeek7.update_layout(barcornerradius=13)
figWeek7.update_layout(yaxis={'categoryorder':'total ascending'})
figWeek7.update_layout(
    legend=dict(
        #x=0,
        #y=1,
        #traceorder="reversed",
        title_font_family="Courier New",
        title = 'Position',
        font=dict(
            family="Courier New",
            size=25,
            color="white"
        )#,
        #bgcolor="LightSteelBlue",
        #bordercolor="Black",
        #borderwidth=2
    )
)
figWeek7.update_layout(
        xaxis_title="",  # Set x-axis title
        yaxis_title="")
figWeek7.update_xaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=16,         # Font size
            color='white'    # Font color
        ),
        title = None
    )

    # Customize the y-axis labels
figWeek7.update_yaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=18,         # Font size
            color='white'    # Font color
        ),
        title=None
    )
figWeek7.update_traces(marker_line_width=2,marker_line_color='black')
figWeek7.update_traces(insidetextanchor= 'middle', textposition='inside', textangle=0)
figWeek7.update_layout(
        uniformtext_minsize=13,
        uniformtext_mode='hide'
        )
# Add annotations for rushing yards
for i, row in enumerate(Week7Groups['team']):
    figWeek7.add_annotation(
        x=Week7Groups['rushing_yards'][i].sum()+5,  # Rushing yards for x position
        y=Week7Groups['team'][i],  # Team name for y position
        text=f"{Week7Groups['rushing_yards'][i]} yards",  # Annotation text
        showarrow=False,  # No arrow
        font=dict(
            family="Courier New",
            size=13,
            color="white"
        ),
        xanchor='left',  # Align to the left of the bar
        yanchor='middle'
    )
figWeek7.show()

## Week 8

In [208]:
PredictedScores = {
    'bgmaddox':110.56, 'DirtyCommie':110.09, 'BMoreBallers88':104.72, 'jhuntmadd':110.76, 'RReclam':96.77,
       'RossLikeSauce':104.24, 'jlglover':95.53, 'RascalHazard':103.65, 'JTizzzzle':94.33, 'eegrady':104.07,
       'sgmaddox':112.30, 'InfiniteJesse':105.93
}

In [209]:
Week8NoMatches_2024['Predicted'] = Week8NoMatches_2024.index.map(PredictedScores)
Week8NoMatches_2024['Difference'] = Week8NoMatches_2024['Total'] - Week8NoMatches_2024['Predicted']
Week8NoMatches_2024['Difference'] = abs(Week8NoMatches_2024['Difference'])

In [210]:
Week8NoMatches_2024.sort_values('Difference')

,QB,RB1,RB2,WR1,WR2,TE,WRT,K,DEF,Matchup,...,Opp,Margin,Week,Week Index,Year,LeagueTotal,PercentTotal,Opp_team,Predicted,Difference
Team,,,,,,,,,,,,,,,,,,,,,
sgmaddox,{'Jordan Love': 5.74},{'Josh Jacobs': 25.0},{'Brian Robinson': 8.1},{'A.J. Brown': 10.9},{'Malik Nabers': 10.6},{'Dalton Kincaid': 11.1},{'Brian Thomas': 13.5},{'Justin Tucker': 7.9},{'DET': 14.0},6,...,119.02,-12.18,8,78,2024,1357.12,7.9,InfiniteJesse,112.30,5.46
jlglover,{'Josh Allen': 22.82},{'Rico Dowdle': 0.0},{'Kareem Hunt': 12.8},{'DJ Moore': 4.4},{'Justin Jefferson': 15.5},{'Hunter Henry': 7.0},{'George Pickens': 9.4},{'Wil Lutz': 4.0},{'LAC': 8.0},4,...,139.00,-55.08,8,78,2024,1357.12,6.2,RascalHazard,95.53,11.61
InfiniteJesse,{'C.J. Stroud': 14.4},{'Javonte Williams': 7.2},{'De'Von Achane': 23.7},{'Cooper Kupp': 13.6},{'Jayden Reed': 7.02},{'Cole Kmet': 3.9},{'CeeDee Lamb': 33.1},{'Jake Bates': 12.1},{'NYJ': 4.0},6,...,106.84,12.18,8,78,2024,1357.12,8.8,sgmaddox,105.93,13.09
jhuntmadd,{'Jared Goff': 19.5},{'Breece Hall': 8.4},{'Kenneth Walker': 6.5},{'Jaxon Smith-Njigba': 9.9},{'Davante Adams': 7.4},{'Brock Bowers': 8.3},{'Kyle Pitts': 23.1},{'Evan McPherson': 4.7},{'DEN': 9.0},2,...,86.10,10.70,8,78,2024,1357.12,7.1,BMoreBallers88,110.76,13.96
bgmaddox,{'Jalen Hurts': 37.14},{'Derrick Henry': 14.2},{'Alvin Kamara': 15.2},{'Amon-Ra St. Brown': 7.7},{'Darnell Mooney': 16.6},{'Trey McBride': 16.9},{'Nick Chubb': 5.2},{'Younghoe Koo': 7.9},{'GB': 4.0},1,...,127.16,-2.32,8,78,2024,1357.12,9.2,DirtyCommie,110.56,14.28
JTizzzzle,{'Joe Burrow': 14.86},{'Bijan Robinson': 20.1},{'Kyren Williams': 20.1},{'Stefon Diggs': 10.2},{'DK Metcalf': 0.0},{'Colby Parkinson': 2.7},{'Zay Flowers': 15.0},{'Jake Elliott': 17.5},{'PHI': 9.0},5,...,122.24,-12.78,8,78,2024,1357.12,8.1,eegrady,94.33,15.13
DirtyCommie,{'Lamar Jackson': 28.16},{'D'Andre Swift': 18.9},{'Chase Brown': 10.4},{'Terry McLaurin': 15.0},{'Tyreek Hill': 10.2},{'David Njoku': 14.6},{'Rachaad White': 13.2},{'Tyler Bass': 6.7},{'KC': 10.0},1,...,124.84,2.32,8,78,2024,1357.12,9.4,bgmaddox,110.09,17.07
eegrady,{'Jayden Daniels': 24.24},{'Jahmyr Gibbs': 19.8},{'Saquon Barkley': 11.6},{'Chris Olave': 14.7},{'Michael Pittman': 2.1},{'George Kittle': 21.8},{'Jakobi Meyers': 14.2},{'Jason Myers': 4.8},{'BUF': 9.0},5,...,109.46,12.78,8,78,2024,1357.12,9.0,JTizzzzle,104.07,18.17
BMoreBallers88,{'Sam Darnold': 22.2},{'Jonathan Taylor': 18.2},{'Aaron Jones': 10.5},{'Amari Cooper': 0.8},{'Jordan Addison': 3.2},{'Mark Andrews': 12.1},{'Deebo Samuel': 10.6},{'Cameron Dicker': 8.5},{'BAL': 0.0},2,...,96.80,-10.70,8,78,2024,1357.12,6.3,jhuntmadd,104.72,18.62


In [211]:
Week8GraphInfo = Week8NoMatches_2024.reset_index()[['Team', 'Total','Predicted','Difference']]
Week8GraphInfo['Difference'] = round(Week8GraphInfo['Difference'],2)
Week8Teams = Week8GraphInfo['Team'].unique()

### Graph

In [212]:
figWeek8 = go.Figure(
    data=[
        go.Scatter(
            x=Week8GraphInfo["Predicted"],
            y=Week8Teams,
            mode="markers",
            name="Predicted",
            marker=dict(
                color="green",
                size=15
            )
            
        ),
        go.Scatter(
            x=Week8GraphInfo["Total"],
            y=Week8Teams,
            mode="markers",
            name="Actual Score",
            marker=dict(
                color="blue",
                size=15
            )   
        ),
    ]
)

# Add a line with an arrow-like effect by creating a separate Scatter trace for each arrow
for i in range(len(Week8Teams)):
    # Line segment from Predicted to Actual
    figWeek8.add_trace(
        go.Scatter(
            x=[Week8GraphInfo["Predicted"][i], Week8GraphInfo["Total"][i]],
            y=[Week8Teams[i], Week8Teams[i]],
            mode="lines+markers",
            line=dict(color='rgba(255, 0, 0, 0.5)', width=2, dash="dashdot", backoff = .5),
            marker=dict(symbol="arrow", color="gray", size=10),
            showlegend=False  # Hide legend for each line
        )
    )
    
    # Add annotation for the difference above each line
    figWeek8.add_annotation(
        x=(Week8GraphInfo["Predicted"][i] + Week8GraphInfo["Total"][i]) / 2,  # Midpoint of Predicted and Actual
        y=Week8Teams[i],
        text=str(Week8GraphInfo["Difference"][i]),  # Display the difference value
        showarrow=False,
        font=dict(color="white"),
        yshift=10  # Shift the text slightly above the line
    )
figWeek8.update_layout(template="plotly_dark")
figWeek8.update_layout(width=800, height=1200)
figWeek8.update_layout(font_family="Courier New", title_font = dict(size=45))
figWeek8.update_yaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=20,         # Font size
            color='white'    # Font color
        ),
        title = None,
        categoryorder="total ascending",
        categoryarray = Week8GraphInfo['Difference']
    )
figWeek8.update_layout(
    xaxis=dict(
        dtick=20  # Set the tick interval to 2
    )
)
figWeek8.update_layout(
    yaxis=dict(
        showgrid=False,
        zeroline=False
    )
)
figWeek8.add_shape(type="rect",
    xref="paper", yref="paper",
    x0=-.250, y0=.43,
    x1=1.1, y1=.49,
    line=dict(
        color="RoyalBlue",
        width=3,
    ),
    fillcolor='#39FF14',
    opacity=.3
)
figWeek8.update_layout(legend=dict(orientation="h",font=dict(size= 18)))
# Set legend location
figWeek8.update_layout(
    legend=dict(
        x=0.45,  # x-coordinate of the legend (0 to 1, where 0 is left and 1 is right)
        y=1.02,   # y-coordinate of the legend (0 to 1, where 0 is bottom and 1 is top)
        xanchor="center",  # horizontal anchor point ('left', 'center', 'right')
        yanchor="top"    # vertical anchor point ('top', 'middle', 'bottom')
    )
)
figWeek8.update_layout(title="Week 8 Side Bet: Closest to Prediction",font=dict(
            size=20  # Set the font size here
        ))
figWeek8.add_layout_image(
    dict(
        source=Image.open(r'C:\Users\Brett\Documents\star1.png'),
        xref="paper",  # Use paper coordinates for x-axis
        yref="paper",  # Use paper coordinates for y-axis
        x=0.9,  # X-coordinate (0 is left, 1 is right)
        y=0.46,  # Y-coordinate (0 is bottom, 1 is top)
        sizex=.1,  # Width of the image (adjust for your image)
        sizey=.1,  # Height of the image
        xanchor="center", yanchor="middle"  # Center the image over the bar
    )
)
figWeek8.show()

FileNotFoundError: [Errno 2] No such file or directory: '/Users/brettmaddox/Library/CloudStorage/GoogleDrive-bgmaddox@gmail.com/Other computers/My Computer/Python Projects/Sleeper/C:\\Users\\Brett\\Documents\\star1.png'

## Week 9

In [213]:
Week9_TEs = week9Breakout2024[week9Breakout2024['position']=='TE']

In [214]:
Week9_TEs_10 = Week9_TEs.sort_values('points', ascending=False).head(10)

In [215]:
Week9_TEs_10['GraphName'] = Week9_TEs_10['team'] + ' - ' + Week9_TEs_10['points'].astype(str)

In [216]:
Week9_TEs_10.sort_values('points', ascending=True)
figTEs = px.bar(Week9_TEs_10, x = 'points',y='player', orientation='h', color = 'team', template='plotly_dark', text='GraphName')
figTEs.update_layout(width=800, height=1200)
figTEs.update_layout(yaxis={'categoryorder':'total ascending'})
figTEs.update_layout(
    legend=dict(
        y=0.5,  # x-coordinate of the legend (0 to 1, where 0 is left and 1 is right)
        x=1.02,   # y-coordinate of the legend (0 to 1, where 0 is bottom and 1 is top)
        xanchor="left",  # horizontal anchor point ('left', 'center', 'right')
        yanchor="middle",
        ))    # vertical anchor point ('top', 'middle', 'bottom')
figTEs.update_layout(font_family="Courier New", title_font = dict(size=40))   
figTEs.update_layout(legend=dict(font=dict(size= 18)))
figTEs.update_layout(title="Week 9 Side Bet: Best Performing TE",font=dict(
            size=17, color = 'white'  # Set the font size here
        )) 
figTEs.update_layout(barcornerradius=13)
figTEs.update_layout(
        xaxis_title="",  # Set x-axis title
        yaxis_title="")
figTEs.update_xaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=16,         # Font size
            color='white'    # Font color
        ),
        title = None
    )

    # Customize the y-axis labels
figTEs.update_yaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=18,         # Font size
            color='white'    # Font color
        ),
        title=None
    )
figTEs.update_layout(showlegend=False)
figTEs.add_layout_image(
    dict(
        source=Image.open(r'C:\Users\Brett\Documents\star1.png'),
        xref="paper",  # Use paper coordinates for x-axis
        yref="paper",  # Use paper coordinates for y-axis
        x=0.55,  # X-coordinate (0 is left, 1 is right)
        y=0.95,  # Y-coordinate (0 is bottom, 1 is top)
        sizex=.1,  # Width of the image (adjust for your image)
        sizey=.1,  # Height of the image
        xanchor="center", yanchor="middle"  # Center the image over the bar
    )
)
figTEs.show()

FileNotFoundError: [Errno 2] No such file or directory: '/Users/brettmaddox/Library/CloudStorage/GoogleDrive-bgmaddox@gmail.com/Other computers/My Computer/Python Projects/Sleeper/C:\\Users\\Brett\\Documents\\star1.png'

## Week 10

In [217]:
Week10SideBet = week10Breakout2024.groupby(['team'])[['player','points','recent_team']]
person = team_list[0]
CurrentGraph = Week10SideBet.get_group(person)
Week10SideBet2 = week10Breakout2024.groupby(['team','recent_team'])[['player','points']].sum(numeric_only=True).reset_index()
Week10SideBet2Data = Week10SideBet2.sort_values('points', ascending=False).head(10)
Week10SideBet2Data['Name'] = Week10SideBet2Data['team'] + ' - ' + Week10SideBet2Data['recent_team']
#breakoutDF_group = breakoutDF.groupby(['team','Game_date_time','matchup']).size().reset_index(name='count')
Week10SideBet = week10Breakout2024.groupby(['team'])[['player','points','recent_team']]
Week10SideBet2 = week10Breakout2024.groupby(['team','recent_team'])[['player','points']].sum(numeric_only=True).reset_index()
Week10SideBet2Data = Week10SideBet2.sort_values('points', ascending=False).head(10)
Week10SideBet2Data['Name'] = Week10SideBet2Data['team'] + ' - ' + Week10SideBet2Data['recent_team']

#grouped = breakoutDF_group.groupby('matchup')
        # Create a subplot with 1 row and 2 columns (for the bar chart and the pie chart)
figCombo2 = make_subplots(
            rows=7, cols=2, 
            shared_xaxes=False,
            horizontal_spacing=0.04, 
            vertical_spacing=0.05,
            shared_yaxes=True,
            column_widths=[0.5, 0.5],  # Adjust the width of each subplot
            specs=[[{"type": "bar"}, {"type": "bar"}],
                [{"type": "bar"}, {"type": "bar"}],
                [{"type": "bar"}, {"type": "bar"}],
                [{"type": "bar"}, {"type": "bar"}],
                [{"type": "bar"}, {"type": "bar"}],
                [{"type": "bar"}, {"type": "bar"}],
                [{"colspan":2,"type":"bar"},None]]
            #subplot_titles=['Matchup Schedule','Win History']# Specify the chart types
        )
for i in range(1,13):
        person = roster_ids_2023[i]
        CurrentGraph = Week10SideBet.get_group(person)
        rowlist = [1,1,2,2,3,3,4,4,5,5,6,6]
        collist = [1,2,1,2,1,2,1,2,1,2,1,2]
        rowdict = dict(enumerate(rowlist,1))
        coldict = dict(enumerate(collist,1))
        
        # Add the bar chart to the first column
        figCombo2.add_trace(
            go.Bar(
                y=CurrentGraph['points'], 
                x=CurrentGraph['recent_team'], 
                
                #marker=dict(
                #    color = [teamcolors[team] for team in CurrentGraph['team']], 
                #    cornerradius = 10
                #),
                
                text=CurrentGraph['points'],
                textangle = 0,
                textposition='auto',
                showlegend=False,
            ),
            row=rowdict[i], col=coldict[i]
        )

        
            # Create a custom title with colored team names
        #title_html = f'<span style="color:{teamcolors[teams[0]]}">{teams[0]}</span> vs <span style="color:{teamcolors[teams[1]]}">{teams[1]}</span>'

        # Add the custom title as an annotation at the top of each subplot
        figCombo2.add_annotation(
            text=roster_ids_2023[i], #,  # The HTML title
            xref=f'x domain', yref=f'y domain',
            x=.5, y=1.2,  # Position it above the subplot (y > 1)
            xanchor='center',
            font=dict(size=15, family='Courier New', weight ='bold'),
            showarrow=False,
            row=rowdict[i], col=coldict[i] # Apply to the i-th row and first column (bar chart)
        )
        # Update the layout with dark theme and grouped bar mode
        figCombo2.update_layout(barmode="group", template="plotly_dark",barcornerradius=7)
        figCombo2.update_xaxes(
            categoryorder="array",
            #categoryarray=time_order,
            showticklabels = True, 
            row=rowdict[i], col=coldict[i]  # Apply to the bar chart in the i-th row and first column
        )
        figCombo2.update_layout(yaxis1=dict(range=[0, 55]),yaxis2=dict(range=[0, 55]),yaxis3=dict(range=[0, 55]),yaxis4=dict(range=[0, 55]),
                                yaxis5=dict(range=[0, 55]),yaxis6=dict(range=[0, 55]),yaxis7=dict(range=[0, 55]),yaxis8=dict(range=[0, 55]),
                                yaxis9=dict(range=[0, 55]),yaxis10=dict(range=[0, 55]),yaxis11=dict(range=[0, 55]),yaxis12=dict(range=[0, 55])
                                )
        figCombo2.update_layout(xaxis1=dict(tickangle=90),xaxis2=dict(tickangle=90),xaxis3=dict(tickangle=90),xaxis4=dict(tickangle=90),
                                xaxis5=dict(tickangle=90),xaxis6=dict(tickangle=90),xaxis7=dict(tickangle=90),xaxis8=dict(tickangle=90),
                                xaxis9=dict(tickangle=90),xaxis10=dict(tickangle=90),xaxis11=dict(tickangle=90),xaxis12=dict(tickangle=90)
                                )
figCombo2.add_trace(
            go.Bar(
                y=Week10SideBet2Data['points'], 
                x=Week10SideBet2Data['Name'], 
                marker_color = ('teal','tomato','tomato','tomato','tomato','tomato','tomato','tomato','tomato','tomato'),
                
                #marker=dict(
                #    color = [teamcolors[team] for team in CurrentGraph['team']], 
                #    cornerradius = 10
                #),
                
                text=Week10SideBet2Data['recent_team'],
                textangle = 0,
                textposition='auto',
                showlegend=False,
            ),
            row=7, col=1
        )
figCombo2.update_layout(width=900, height=1200,title_text="Week 10 Side Bet - Franchise Week")
figCombo2.update_layout(font_family="Courier New", title_font = dict(size=35))   
figCombo2.show()

In [222]:
figFranchise = px.bar(CurrentGraph, x = 'points',y='recent_team', color = 'recent_team', template='plotly_dark', text='recent_team')
figFranchise.update_layout(width=800, height=1200)
figFranchise.show()

## Week 11

In [223]:
def heaviside(x):
    return np.where(x < 0, 0, 1)

# Define the parametric functions
t = np.linspace(0, 2*np.pi, 100)
t1 = np.linspace(0, 2*np.pi, 100)
x = -1/7*np.sin(7/12-124*t)-3/7*np.sin(22/15-123*t)-2/7*np.sin(9/7-119*t)-5/9*np.sin(7/9-118*t)-3/10*np.sin(7/9-117*t)-5/7*np.sin(4/3-111*t)-5/8*np.sin(4/3-104*t)-5/11*np.sin(16/13-95*t)-6/19*np.sin(10/9-92*t)-4/5*np.sin(4/11-88*t)-18/11*np.sin(17/11-72*t)-10/7*np.sin(1/5-71*t)-13/10*np.sin(1/4-70*t)-7/11*np.sin(4/3-69*t)-np.sin(1/21-68*t)-10/13*np.sin(11/8-67*t)-23/12*np.sin(1/3-63*t)-6/5*np.sin(3/2-62*t)-9/7*np.sin(7/8-61*t)-7/4*np.sin(1-60*t)-15/7*np.sin(5/7-58*t)-1/5*np.sin(8/7-56*t)-43/17*np.sin(14/9-50*t)-25/8*np.sin(16/17-45*t)-41/21*np.sin(4/9-44*t)-48/19*np.sin(1/5-42*t)-15/7*np.sin(1/18-41*t)-36/11*np.sin(1/7-36*t)-43/16*np.sin(13/11-32*t)-12/5*np.sin(1/6-31*t)-46/11*np.sin(3/11-27*t)-39/8*np.sin(2/11-24*t)-56/11*np.sin(2/9-23*t)-37/7*np.sin(14/9-16*t)-1168/5*np.sin(5/6-2*t)+28/5*np.sin(26*t)+41961/40*np.sin(t+41/20)+443/3*np.sin(3*t+29/10)+1311/13*np.sin(4*t+17/10)+268/7*np.sin(5*t+22/9)+86/9*np.sin(6*t+4/7)+97/6*np.sin(7*t+31/16)+42/5*np.sin(8*t+41/21)+240/11*np.sin(9*t+24/7)+45/4*np.sin(10*t+13/3)+108/11*np.sin(11*t+5/9)+531/28*np.sin(12*t+11/8)+76/11*np.sin(13*t+4/5)+19/2*np.sin(14*t+60/13)+235/18*np.sin(15*t+29/8)+89/8*np.sin(17*t+13/11)+39/7*np.sin(18*t+29/15)+122/13*np.sin(19*t+35/8)+128/15*np.sin(20*t+51/13)+66/13*np.sin(21*t+29/9)+52/15*np.sin(22*t+40/11)+14/3*np.sin(25*t+1/21)+3/2*np.sin(28*t+1/34)+19/14*np.sin(29*t+37/8)+53/9*np.sin(30*t+5/12)+54/11*np.sin(33*t+18/7)+37/10*np.sin(34*t+5/2)+27/7*np.sin(35*t+1/4)+19/3*np.sin(37*t+9/2)+16/5*np.sin(38*t+29/8)+17/13*np.sin(39*t+9/8)+20/13*np.sin(40*t+5/3)+26/11*np.sin(43*t+17/18)+7/4*np.sin(46*t+24/7)+15/8*np.sin(47*t+9/2)+8/7*np.sin(48*t+26/9)+31/15*np.sin(49*t+40/11)+4/3*np.sin(51*t+13/6)+4/3*np.sin(52*t+29/11)+1/2*np.sin(53*t+12/11)+5/4*np.sin(54*t+30/13)+9/8*np.sin(55*t+23/12)+17/18*np.sin(57*t+5/3)+15/11*np.sin(59*t+94/21)+np.sin(64*t+6/7)+9/5*np.sin(65*t+20/11)+18/13*np.sin(66*t+1/2)+7/9*np.sin(73*t+5/2)+7/6*np.sin(74*t+31/11)+4/7*np.sin(75*t+10/21)+5/7*np.sin(76*t+4/7)+6/13*np.sin(77*t+27/11)+7/8*np.sin(78*t+31/9)+3/5*np.sin(79*t+13/6)+4/5*np.sin(80*t+5/3)+2/3*np.sin(81*t+16/5)+6/5*np.sin(82*t+5/7)+1/5*np.sin(83*t+1/13)+4/5*np.sin(84*t+21/13)+4/5*np.sin(85*t+35/12)+4/7*np.sin(86*t+31/10)+1/7*np.sin(87*t+47/10)+1/7*np.sin(89*t+23/5)+7/10*np.sin(90*t+4/11)+1/8*np.sin(91*t+25/7)+3/7*np.sin(93*t+59/13)+3/5*np.sin(94*t+27/8)+1/4*np.sin(96*t+8/7)+3/5*np.sin(97*t+9/7)+17/18*np.sin(98*t+10/3)+3/7*np.sin(99*t+1/4)+1/22*np.sin(100*t+22/5)+2/3*np.sin(101*t+1)+1/3*np.sin(102*t+4/7)+7/11*np.sin(103*t+11/4)+1/11*np.sin(105*t+53/16)+1/9*np.sin(106*t+61/13)+5/9*np.sin(107*t+113/28)+4/11*np.sin(108*t+5/14)+1/2*np.sin(109*t+25/9)+5/14*np.sin(110*t+28/9)+3/10*np.sin(112*t+8/11)+1/9*np.sin(113*t+13/10)+2/5*np.sin(114*t+15/7)+11/16*np.sin(115*t+19/7)+2/7*np.sin(116*t+32/7)+1/4*np.sin(120*t+12/7)+3/4*np.sin(121*t+11/9)+10/19*np.sin(122*t+11/9)

y = -4/7*np.sin(4/7-123*t)-1/6*np.sin(8/7-121*t)-2/5*np.sin(12/11-117*t)-1/5*np.sin(4/7-116*t)-3/13*np.sin(23/24-112*t)-6/5*np.sin(5/8-104*t)-10/13*np.sin(8/7-103*t)-39/40*np.sin(7/6-96*t)-4/5*np.sin(4/13-95*t)-3/4*np.sin(7/8-92*t)-2/9*np.sin(9/8-91*t)-8/7*np.sin(26/17-79*t)-11/7*np.sin(1/17-76*t)-5/6*np.sin(7/8-72*t)-5/8*np.sin(7/5-70*t)-5/11*np.sin(14/9-68*t)-4/7*np.sin(4/7-65*t)-7/12*np.sin(8/7-64*t)-11/7*np.sin(9/14-62*t)-13/5*np.sin(1-57*t)-3/5*np.sin(2/3-55*t)-1/17*np.sin(3/2-53*t)-8/3*np.sin(1/2-46*t)-11/12*np.sin(19/13-41*t)-5/4*np.sin(28/19-23*t)+39/20*np.sin(43*t)+2/9*np.sin(108*t)+4415/7*np.sin(t+4/9)+4997/21*np.sin(2*t+3)+551/4*np.sin(3*t+11/6)+455/8*np.sin(4*t+23/6)+247/10*np.sin(5*t+19/11)+669/10*np.sin(6*t+23/24)+377/6*np.sin(7*t+14/5)+115/8*np.sin(8*t+28/9)+43/8*np.sin(9*t+31/12)+219/14*np.sin(10*t+57/28)+1057/33*np.sin(11*t+19/11)+43/9*np.sin(12*t+9/14)+183/7*np.sin(13*t+1/8)+71/5*np.sin(14*t+6/7)+211/15*np.sin(15*t+31/8)+23/3*np.sin(16*t+35/9)+96/7*np.sin(17*t+25/7)+13*np.sin(18*t+75/19)+93/11*np.sin(19*t+9/4)+90/13*np.sin(20*t+22/7)+99/8*np.sin(21*t+7/11)+68/9*np.sin(22*t+12/7)+74/11*np.sin(24*t+59/20)+39/7*np.sin(25*t+33/8)+49/9*np.sin(26*t+16/7)+19/8*np.sin(27*t+67/17)+62/13*np.sin(28*t+5/4)+26/7*np.sin(29*t+16/7)+19/12*np.sin(30*t+7/9)+17/8*np.sin(31*t+17/5)+23/6*np.sin(32*t+39/10)+73/8*np.sin(33*t+4)+32/9*np.sin(34*t+19/5)+3/5*np.sin(35*t+37/19)+7/9*np.sin(36*t+25/8)+1/8*np.sin(37*t+67/17)+17/6*np.sin(38*t+18/5)+10/3*np.sin(39*t+19/9)+40/9*np.sin(40*t+19/8)+3/2*np.sin(42*t+9/4)+16/7*np.sin(44*t+5/12)+26/11*np.sin(45*t+3/8)+15/8*np.sin(47*t+31/12)+15/8*np.sin(48*t+27/8)+19/14*np.sin(49*t+15/7)+13/5*np.sin(50*t+26/9)+5/3*np.sin(51*t+1/2)+17/16*np.sin(52*t+9/2)+9/7*np.sin(54*t+32/7)+11/8*np.sin(56*t+1/13)+16/9*np.sin(58*t+101/25)+68/67*np.sin(59*t+17/4)+12/13*np.sin(60*t+18/7)+9/5*np.sin(61*t+20/7)+8/9*np.sin(63*t+37/8)+4/3*np.sin(66*t+19/11)+11/7*np.sin(67*t+14/9)+8/15*np.sin(69*t+25/9)+11/9*np.sin(71*t+61/13)+1/7*np.sin(73*t+59/15)+5/7*np.sin(74*t+23/13)+3/2*np.sin(75*t+9/7)+1/2*np.sin(77*t+7/10)+2/9*np.sin(78*t+25/6)+6/7*np.sin(80*t+2/9)+2/3*np.sin(81*t+25/12)+1/4*np.sin(82*t+25/7)+8/7*np.sin(83*t+17/18)+7/10*np.sin(84*t+21/8)+11/9*np.sin(85*t+39/10)+4/7*np.sin(86*t+38/13)+3/8*np.sin(87*t+42/13)+3/8*np.sin(88*t+18/7)+11/21*np.sin(89*t+1/4)+24/25*np.sin(90*t+13/11)+2/5*np.sin(93*t+65/16)+3/7*np.sin(94*t+151/38)+8/11*np.sin(97*t+1/23)+20/19*np.sin(98*t+31/16)+3/4*np.sin(99*t+21/10)+1/2*np.sin(100*t+9/4)+7/12*np.sin(101*t+26/11)+11/16*np.sin(102*t+50/17)+4/7*np.sin(105*t+43/10)+1/6*np.sin(106*t+17/6)+3/10*np.sin(107*t+19/8)+1/3*np.sin(109*t+51/13)+3/13*np.sin(110*t+4)+5/9*np.sin(111*t+50/17)+3/7*np.sin(113*t+21/8)+7/15*np.sin(114*t+15/7)+2/3*np.sin(115*t+5/6)+7/9*np.sin(118*t+47/12)+5/6*np.sin(119*t+7/2)+1/7*np.sin(120*t+30/7)+1/14*np.sin(122*t+1/3)+1/3*np.sin(124*t+38/9)

x1 = ((-34/3*np.sin(1/2-t1)+8/3*np.sin(2*t1+5/3)-179/2)*heaviside((7*np.pi-t1))*heaviside((t1-3*np.pi))+(-1/6*np.sin(4/3-145*t1)-1/7*np.sin(6/5-142*t1)-1/7*np.sin(1-127*t1)-1/3*np.sin(1/5-121*t1)-1/4*np.sin(3/2-112*t1)-1/4*np.sin(2/3-103*t1)-1/3*np.sin(3/2-100*t1)-1/4*np.sin(3/2-93*t1)-1/4*np.sin(1/2-88*t1)-1/3*np.sin(3/2-74*t1)-2/3*np.sin(1-70*t1)-2/3*np.sin(1/5-67*t1)-1/6*np.sin(6/5-66*t1)-1/3*np.sin(4/3-64*t1)-1/2*np.sin(1/3-63*t1)-2/3*np.sin(3/2-62*t1)-np.sin(1/7-61*t1)-1/2*np.sin(1-54*t1)-1/2*np.sin(1-48*t1)-3/2*np.sin(3/4-38*t1)-13/3*np.sin(3/2-26*t1)-25/2*np.sin(1-20*t1)-25/2*np.sin(3/4-15*t1)-11*np.sin(2/3-14*t1)-23*np.sin(1-13*t1)-147/2*np.sin(3/2-4*t1)-187/2*np.sin(1/2-3*t1)-271*np.sin(6/5-2*t1)+7/2*np.sin(9*t1)+1/7*np.sin(110*t1)+1/5*np.sin(116*t1)+1/5*np.sin(125*t1)+1/4*np.sin(128*t1)+2041/3*np.sin(t1+1/2)+78*np.sin(5*t1+2)+286/5*np.sin(6*t1+5/3)+82/3*np.sin(7*t1+1/7)+13/3*np.sin(8*t1+3/4)+91/3*np.sin(10*t1+8/3)+63/2*np.sin(11*t1+7/2)+55/4*np.sin(12*t1+15/4)+15/2*np.sin(16*t1+1)+32/3*np.sin(17*t1+5/2)+19/2*np.sin(18*t1+5/3)+12*np.sin(19*t1+13/4)+5*np.sin(21*t1+5/3)+15/4*np.sin(22*t1+4)+26/5*np.sin(23*t1+2)+9/4*np.sin(24*t1+9/4)+3*np.sin(25*t1+5/2)+4/3*np.sin(27*t1+1/5)+1/4*np.sin(28*t1+5/2)+8/3*np.sin(29*t1+2)+4/3*np.sin(30*t1+1/2)+5*np.sin(31*t1+1)+2*np.sin(32*t1+5/2)+3/2*np.sin(33*t1+11/5)+5/2*np.sin(34*t1+3)+7/4*np.sin(35*t1+2)+3/2*np.sin(36*t1+5/2)+1/2*np.sin(37*t1+10/3)+3/4*np.sin(39*t1+3/2)+np.sin(40*t1+13/3)+3/2*np.sin(41*t1+5/3)+1/2*np.sin(42*t1+5/3)+5/3*np.sin(43*t1+2)+3/4*np.sin(44*t1+4)+3/4*np.sin(45*t1+19/6)+np.sin(46*t1+7/2)+2/3*np.sin(47*t1+7/2)+3/4*np.sin(49*t1+4/3)+3/4*np.sin(50*t1+10/3)+6/5*np.sin(51*t1+7/2)+3/2*np.sin(52*t1+17/4)+1/2*np.sin(53*t1+1/3)+3/2*np.sin(55*t1+2/3)+1/5*np.sin(56*t1+3)+np.sin(57*t1+4/3)+np.sin(58*t1+7/2)+1/2*np.sin(59*t1+1/4)+3/4*np.sin(60*t1+15/4)+1/2*np.sin(65*t1+1)+1/3*np.sin(68*t1+1/5)+1/6*np.sin(69*t1+5/3)+1/2*np.sin(71*t1+1)+1/5*np.sin(72*t1+4/3)+1/4*np.sin(76*t1+21/5)+1/6*np.sin(77*t1+4/3)+1/2*np.sin(80*t1+1/2)+1/3*np.sin(82*t1+4/3)+1/2*np.sin(83*t1+5/2)+1/3*np.sin(84*t1+11/4)+1/5*np.sin(85*t1+14/3)+1/4*np.sin(87*t1+9/2)+1/3*np.sin(89*t1+1)+1/3*np.sin(90*t1+7/3)+1/4*np.sin(95*t1+7/6)+1/6*np.sin(96*t1+7/3)+1/6*np.sin(97*t1+11/4)+1/2*np.sin(99*t1+7/2)+1/6*np.sin(101*t1+7/3)+1/3*np.sin(102*t1+13/3)+1/5*np.sin(104*t1+1/4)+1/2*np.sin(105*t1+11/4)+1/3*np.sin(106*t1+9/2)+1/6*np.sin(107*t1+5/3)+1/4*np.sin(108*t1+5/2)+1/5*np.sin(111*t1+7/2)+1/3*np.sin(122*t1+4/3)+1/3*np.sin(123*t1+8/3)+1/3*np.sin(124*t1+25/6)+1/7*np.sin(126*t1+3)+1/4*np.sin(129*t1+2/3)+1/5*np.sin(130*t1+11/3)+1/6*np.sin(131*t1+4/3)+1/7*np.sin(133*t1+2/3)+1/6*np.sin(135*t1+4/3)+1/4*np.sin(137*t1+3/2)+1/7*np.sin(138*t1+7/2)+1/6*np.sin(139*t1+1/2)+1/3*np.sin(143*t1+1)+1/5*np.sin(144*t1+7/3)-173/2)*heaviside((3*np.pi-t1))*heaviside(t1+np.pi))*heaviside(np.sqrt(np.maximum(0, np.sign(np.sin(t1 / 2)))))
y1 = ((112/3*np.sin(t1+14/3)+1/2*np.sin(2*t1+15/4)-1411/6)*heaviside(7*np.pi-t1)*heaviside(t1-3*np.pi)+(-1/3*np.sin(3/2-131*t1)-1/4*np.sin(4/3-125*t1)-1/7*np.sin(6/5-123*t1)-1/3*np.sin(3/2-119*t1)-1/4*np.sin(4/3-111*t1)-1/5*np.sin(7/6-107*t1)-1/4*np.sin(1/4-105*t1)-1/3*np.sin(8/7-95*t1)-1/3*np.sin(1-92*t1)-1/2*np.sin(1/3-89*t1)-1/3*np.sin(1/2-80*t1)-1/2*np.sin(3/2-68*t1)-2/3*np.sin(1/2-66*t1)-3/4*np.sin(1-53*t1)-6/5*np.sin(1/5-45*t1)-1/3*np.sin(1-43*t1)-np.sin(1/5-42*t1)-2*np.sin(1/2-41*t1)-3/2*np.sin(1-40*t1)-2*np.sin(3/4-38*t1)-5/2*np.sin(1/3-35*t1)-5/2*np.sin(1/2-34*t1)-5/2*np.sin(2/3-29*t1)-17/4*np.sin(3/2-27*t1)-8/3*np.sin(2/3-25*t1)-17/2*np.sin(1/2-18*t1)-32/3*np.sin(1/3-15*t1)-79/4*np.sin(1/2-11*t1)-23/2*np.sin(1/2-9*t1)-131/3*np.sin(1/5-7*t1)+189/2*np.sin(3*t1)+121/3*np.sin(6*t1)+14/3*np.sin(13*t1)+7/2*np.sin(31*t1)+1/2*np.sin(93*t1)+1/2*np.sin(96*t1)+1/2*np.sin(102*t1)+590*np.sin(t1+9/2)+487/2*np.sin(2*t1+2)+112*np.sin(4*t1+3/4)+191/4*np.sin(5*t1+3)+59/2*np.sin(8*t1+1)+9*np.sin(10*t1+4/3)+36/5*np.sin(12*t1+1/2)+19/2*np.sin(14*t1+13/4)+46/5*np.sin(16*t1+2)+23/3*np.sin(17*t1+4)+25/3*np.sin(19*t1+1)+7*np.sin(20*t1+3)+11/2*np.sin(21*t1+14/3)+14/3*np.sin(22*t1+1/2)+4*np.sin(23*t1+3/2)+8/3*np.sin(24*t1+7/2)+5*np.sin(26*t1+8/3)+5/3*np.sin(28*t1+1/2)+6/5*np.sin(30*t1+1/2)+3*np.sin(32*t1+2)+5/2*np.sin(33*t1+25/6)+np.sin(36*t1+2/3)+7/4*np.sin(37*t1+9/2)+1/3*np.sin(39*t1+3/4)+2/3*np.sin(44*t1+9/2)+np.sin(46*t1+3/2)+1/2*np.sin(47*t1+14/3)+1/5*np.sin(48*t1+1)+1/3*np.sin(49*t1+7/2)+np.sin(50*t1+3)+1/5*np.sin(51*t1+9/2)+2/3*np.sin(52*t1+15/7)+1/2*np.sin(54*t1+1)+2/3*np.sin(55*t1+3)+1/2*np.sin(56*t1+5/2)+1/3*np.sin(57*t1+1)+6/5*np.sin(58*t1+2)+np.sin(59*t1+11/4)+2/3*np.sin(61*t1+11/3)+1/3*np.sin(62*t1+4/3)+1/3*np.sin(63*t1+3/2)+1/2*np.sin(64*t1+2)+2/3*np.sin(65*t1+11/3)+1/2*np.sin(67*t1+11/4)+1/5*np.sin(69*t1+3)+1/4*np.sin(70*t1+3/4)+2/3*np.sin(71*t1+5/2)+2/3*np.sin(72*t1+13/3)+1/3*np.sin(73*t1+5/2)+1/3*np.sin(74*t1+7/2)+1/2*np.sin(75*t1+8/3)+1/4*np.sin(76*t1+13/3)+1/7*np.sin(77*t1+9/2)+1/3*np.sin(78*t1+3)+1/2*np.sin(79*t1+25/6)+1/2*np.sin(81*t1+2)+1/3*np.sin(82*t1+7/4)+1/3*np.sin(83*t1+11/3)+1/4*np.sin(85*t1+19/6)+1/5*np.sin(86*t1+1)+1/4*np.sin(87*t1+5/2)+1/6*np.sin(88*t1+5/3)+1/2*np.sin(90*t1+1)+1/3*np.sin(91*t1+3)+1/2*np.sin(94*t1+3/2)+1/3*np.sin(97*t1+5/3)+1/4*np.sin(98*t1+10/3)+1/7*np.sin(100*t1+2/3)+1/7*np.sin(101*t1+21/5)+1/3*np.sin(108*t1+1/2)+1/7*np.sin(109*t1+11/4)+1/4*np.sin(110*t1+8/3)+1/3*np.sin(112*t1+1)+1/6*np.sin(113*t1+4)+1/4*np.sin(114*t1+1)+1/4*np.sin(118*t1+3/2)+1/5*np.sin(120*t1+1/2)+1/5*np.sin(124*t1+5/3)+1/4*np.sin(127*t1+9/2)+1/6*np.sin(130*t1+5/3)+1/3*np.sin(133*t1+7/2)+1/3*np.sin(136*t1+2)+1/5*np.sin(137*t1+7/2)+1/6*np.sin(139*t1+4)+1/5*np.sin(142*t1+2)+1/4*np.sin(143*t1+17/4)+1/5*np.sin(144*t1+1/6)+3/4)*heaviside(3*np.pi-t1)*heaviside(t1+np.pi))*heaviside(np.sqrt(np.maximum(0, np.sign(np.sin(t1 / 2)))))

# Create the figure
figWeek11 = go.Figure()

# Add the parametric plot
figWeek11.add_trace(go.Scatter(x=x, y=y, mode='lines'))
figWeek11.add_trace(go.Scatter(x=x1, y=y1, mode='lines'))

# Update the layout (optional)
figWeek11.update_layout(title='Week 11 Side Bet: Most Trades',
                  xaxis_title='',
                  yaxis_title='')
figWeek11.update_layout(height = 800, width = 1200, template = 'plotly_dark')
figWeek11.update_layout(font_family="Courier New", title_font = dict(size=40), showlegend = False)   
figWeek11.update_xaxes(showticklabels=False)
figWeek11.update_yaxes(showticklabels=False)
# Show the plot
figWeek11.show()

## Week 12

In [224]:
Week12Graph = week12Breakout2024[week12Breakout2024['starter']==1]

In [225]:
Week12Graph = Week12Graph[Week12Graph.position == 'QB']

In [227]:
cols = ['team','player','completions', 'attempts', 'color', 'recent_teams']

In [228]:
Week12Simple = Week12Graph[cols]

In [229]:
Week12Simple['CompletionPercent'] = round(Week12Simple['completions'] / Week12Simple['attempts'] * 100,2)

In [230]:
Week12Simple = Week12Simple.sort_values('CompletionPercent', ascending=False)

In [231]:
Week12Simple['GraphText'] = Week12Simple.CompletionPercent.astype(str) + '% - ' + Week12Simple.player + ' (' + Week12Simple.recent_teams + ')'

In [232]:
figWeek12 = px.pie(Week12Simple, values = 'CompletionPercent', template = 'plotly_dark', color = 'team',
                   hole = .4)
figWeek12.show()

In [233]:

figQBComplete = go.Figure()
'''
px.bar(Week12Simple, y = 'team', x = ['attempts', 'completions'], template = 'plotly_dark', color = 'team',
                       barmode = 'overlay', orientation='h')
'''
figQBComplete.update_layout(height = 1000, width = 800)
figQBComplete.update_layout(barcornerradius=13)
figQBComplete.update_layout(title_text="<b>Week 12 Side Bet:</b> Best Completion %")


figQBComplete.add_trace(go.Bar(
    x=Week12Simple.attempts,
    y=Week12Simple.team,
    name='Trace 1', 
    marker_color = Week12Simple.color,
    orientation='h',
    opacity=.65,
    text=Week12Simple.attempts
    
))

figQBComplete.add_trace(go.Bar(
    x=Week12Simple.completions,
    y=Week12Simple.team,
    name='Trace 2',
    marker_color = Week12Simple.color,
    orientation='h',
    opacity=.5,
    text=Week12Simple.completions

    )
)

figQBComplete.add_trace(go.Bar(
    x=-Week12Simple.CompletionPercent,
    y=Week12Simple.team,
    name='Trace 3',
    marker_color = Week12Simple.color,
    orientation='h',
    opacity=.9,
    text=Week12Simple.GraphText

    )
)
figQBComplete.update_layout(yaxis3={'categoryorder': 'total ascending'})


figQBComplete.update_layout(barmode='overlay',template = 'plotly_dark', showlegend=False)
figQBComplete.update_layout(font_family="Courier New", title_font = dict(size=25))   
figQBComplete.update_layout(
    xaxis=dict(
        tickmode='array',
        tickvals=[-80, -60, -40, -20,0,20,40,60],
        ticktext=['80%', '60%', '40%', '20%', '0','20 ATT','40 ATT']
    )
)
figQBComplete.update_yaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=18,         # Font size
            color='white'    # Font color
        ),
        title=None
    )
figQBComplete.update_layout(
    xaxis=dict(
        side="top"
    )
)
figQBComplete.update_layout(
    font=dict(
        family="Courier New, monospace",
        size=14,  # Set the font size here
        color="white"
    )
)
figQBComplete.add_layout_image(
    dict(
        source=Image.open(r'/Users/brettmaddox/Library/CloudStorage/GoogleDrive-bgmaddox@gmail.com/Other computers/My Computer/Python Projects/Sleeper/star.png'),
        xref="paper",  # Use paper coordinates for x-axis
        yref="paper",  # Use paper coordinates for y-axis
        x=0.95,  # X-coordinate (0 is left, 1 is right)
        y=0.04,  # Y-coordinate (0 is bottom, 1 is top)
        sizex=.1,  # Width of the image (adjust for your image)
        sizey=.1,  # Height of the image
        xanchor="center", yanchor="middle"  # Center the image over the bar
    )
)

figQBComplete.update_traces(marker_line_width=1.5,marker_line_color='black')

figQBComplete.show()

In [234]:


figQBComplete2 = px.bar_polar(Week12Simple, r="CompletionPercent", theta="team", template='plotly_dark', color = 'team', range_r=[26, 90])
figQBComplete2.update_layout(font_family="Courier New", title_font = dict(size=35))   
figQBComplete2.update_layout(
    polar=dict(
        angularaxis=dict(
            
            ticks="outside",
        ),
        radialaxis=dict(
            tickmode="linear",
            dtick=25,
            ticksuffix="%",
            tickangle=30,
            tickfont=dict(
                weight='bold',
                size=20,
                color="red"
            )
            
        )
    )
)

                   
figQBComplete2.show()

In [ ]:
Week12Graph = week12Breakout2024[week12Breakout2024['starter']==1]
Week12Graph = Week12Graph[Week12Graph.position == 'QB']
cols = ['team','player','completions', 'attempts', 'color', 'recent_teams']
Week12Simple = Week12Graph[cols]
Week12Simple['CompletionPercent'] = round(Week12Simple['completions'] / Week12Simple['attempts'] * 100,2)
Week12Simple = Week12Simple.sort_values('CompletionPercent', ascending=False)
Week12Simple['GraphText'] = Week12Simple.CompletionPercent.astype(str) + '% - ' + Week12Simple.player + ' (' + Week12Simple.recent_teams + ')'
figWeek12 = px.pie(Week12Simple, values = 'CompletionPercent', template = 'plotly_dark', color = 'team',
                   hole = .4)
figWeek12.show()

figQBComplete = go.Figure()
'''
px.bar(Week12Simple, y = 'team', x = ['attempts', 'completions'], template = 'plotly_dark', color = 'team',
                       barmode = 'overlay', orientation='h')
'''
figQBComplete.update_layout(height = 1000, width = 800)
figQBComplete.update_layout(barcornerradius=13)
figQBComplete.update_layout(title_text="<b>Week 12 Side Bet:</b> Best Completion %")


figQBComplete.add_trace(go.Bar(
    x=Week12Simple.attempts,
    y=Week12Simple.team,
    name='Trace 1', 
    marker_color = Week12Simple.color,
    orientation='h',
    opacity=.65,
    text=Week12Simple.attempts
    
))

figQBComplete.add_trace(go.Bar(
    x=Week12Simple.completions,
    y=Week12Simple.team,
    name='Trace 2',
    marker_color = Week12Simple.color,
    orientation='h',
    opacity=.5,
    text=Week12Simple.completions

    )
)

figQBComplete.add_trace(go.Bar(
    x=-Week12Simple.CompletionPercent,
    y=Week12Simple.team,
    name='Trace 3',
    marker_color = Week12Simple.color,
    orientation='h',
    opacity=.9,
    text=Week12Simple.GraphText

    )
)
figQBComplete.update_layout(yaxis3={'categoryorder': 'total ascending'})


figQBComplete.update_layout(barmode='overlay',template = 'plotly_dark', showlegend=False)
figQBComplete.update_layout(font_family="Courier New", title_font = dict(size=25))   
figQBComplete.update_layout(
    xaxis=dict(
        tickmode='array',
        tickvals=[-80, -60, -40, -20,0,20,40,60],
        ticktext=['80%', '60%', '40%', '20%', '0','20 ATT','40 ATT']
    )
)
figQBComplete.update_yaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=18,         # Font size
            color='white'    # Font color
        ),
        title=None
    )
figQBComplete.update_layout(
    xaxis=dict(
        side="top"
    )
)
figQBComplete.update_layout(
    font=dict(
        family="Courier New, monospace",
        size=14,  # Set the font size here
        color="white"
    )
)
figQBComplete.add_layout_image(
    dict(
        source=Image.open(r'/Users/brettmaddox/Library/CloudStorage/GoogleDrive-bgmaddox@gmail.com/Other computers/My Computer/Python Projects/Sleeper/star.png'),
        xref="paper",  # Use paper coordinates for x-axis
        yref="paper",  # Use paper coordinates for y-axis
        x=0.95,  # X-coordinate (0 is left, 1 is right)
        y=0.04,  # Y-coordinate (0 is bottom, 1 is top)
        sizex=.1,  # Width of the image (adjust for your image)
        sizey=.1,  # Height of the image
        xanchor="center", yanchor="middle"  # Center the image over the bar
    )
)

figQBComplete.update_traces(marker_line_width=1.5,marker_line_color='black')

figQBComplete.show()

## Week 13

In [235]:
dfWeek13 = Week12NoMatches_2024.reset_index()
dfWeek13['Abs Margin'] = dfWeek13.Margin.abs().round(1)
dfWeek13['TeamName'] = dfWeek13.Team + ' [' + dfWeek13.Margin.round(1).astype(str) + ']'

figWeek13 = px.bar(dfWeek13.sort_values('Abs Margin', ascending=False), x='Margin',y='Team',template = 'plotly_dark',title='Week 6: Biggest Margin', color = 'Matchup', orientation='h', text='TeamName' )
#figWeek13.update_layout(barmode='stack', yaxis={'categoryorder':'mean ascending'})
figWeek13.update_layout(barcornerradius=13)
figWeek13.update_layout(width=1200, height=800)
figWeek13.update_layout(title=dict(
            font=dict(
                size=35,
                family="Courier New")))  # Set the width and height in pixels
figWeek13.update_traces(textfont_size=20, textangle=0, cliponaxis=True, textposition = 'auto', textfont=dict(weight='bold',family='Courier New',  # Font family
            size=15   # Font color
        ))
figWeek13.update_layout(
        xaxis_title="Margin",  # Set x-axis title
        yaxis_title="",   # Set y-axis title
        xaxis=dict(
            title_font=dict(
                family="Courier New",  # Set font family for x-axis title
                size=30,          # Set font size for x-axis title
                color ='red',
                weight = 'bold'
            )
        ),
        yaxis=dict(
            title_font=dict(
                family="Courier New",  # Set font family for y-axis title
                size=30,          # Set font size for y-axis title
                color = 'green',
                weight = 'bold'
            )
        )
    )
figWeek13.update_layout(yaxis=dict(showticklabels=False))
#Update the layout to hide the legend:
figWeek13.update(layout_coloraxis_showscale=False)
figWeek13.update_xaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=22,         # Font size
            color='white'    # Font color
        ),
        title = None
    )
figWeek13.show()

## Week 14

## Side Bet Tally

In [ ]:
tally_list = [['JTizzzzle',1],
 ['eegrady',1],
 ['cosmodromedary',1],
 ['bgmaddox',2],
 ['sgmaddox',1],
 ['jhuntmadd',2],
 ['RascalHazard',0],
 ['InfiniteJesse',0],
 ['BMoreBallers88',0],
 ['RossLikeSauce',0],
 ['DirtyCommie',2],
 ['jlglover',0]]

tallyDF = pd.DataFrame(tally_list, columns=['Team','Wins'])

In [ ]:
SideBetWeeklyWins2024_list = [["WEEK 1: I'm flying, Jack! - Team with the highest score (starters only)",'cosmodromedary'],
['WEEK 2: Look At These TDs - Team with the most offensive touchdowns scored','DirtyCommie'],
['WEEK 3: Big Helpers, too (just ask my mom): Most combined points with starting D/ST & Kicker','jhuntmadd'],
['WEEK 4: Blackjack - Team with a starter closest to 21 points without going over','sgmaddox & jhuntmadd'],
['WEEK 5: The Replacements - Team with the highest total points for their bench','DirtyCommie'],
['WEEK 6: The Boom & Bust: Team with the largest point differential between their single highest-scoring starter and their single lowest-scoring starter.','eegrady'],
['WEEK 7: Campus Rush Week - Total rush yards for team (active or bench)','bgmaddox'],
['WEEK 8: All Hands on Deck: Team with the most starting players who score over 15 points','bgmaddox'],
['WEEK 9: The Old Man & Young Buck: Best combined score from a starting player over 30 and a rookie','JTizzzzle'],
['WEEK 10: NFL Franchise Week - Team with the highest point total of players from the same franchise (active or bench)',''],
['WEEK 11: Please not the Jets (Trade Deadline Week) - Team with the most trades this seasons wins',''],
['WEEK 12: Go Long - Team with the Starting QB with the highest completion % (over 10 throws)',''],
["WEEK 13: Coffee's For Closers - Team that beats its opponent by the smallest margin of victory",''],
['WEEK 14: Breaking of the Tie (if needed) - Choose 3 non-QB players. Highest combined total wins.','']]

SideBetWeeklyWins2024 = pd.DataFrame(SideBetWeeklyWins2024_list, columns=['Side Bet','Winner'])

In [239]:
SideBetWeeklyWins2024

,Side Bet,Winner
0,"WEEK 1: I'm flying, Jack! - Team with the high...",BMoreBallers88
1,WEEK 2: Look At These TDs - Team with the most...,bgmaddox
2,WEEK 3: Blackjack - Team with a starter closes...,jhuntmadd
3,WEEK 4: Endzones that way --> - Team with the ...,JTizzzzle
4,WEEK 5: The Replacements - Team with the highe...,sgmaddox
5,WEEK 6: Like A Boss - Team with the biggest ma...,bgmaddox
6,WEEK 7: Campus Rush Week - Total rush yards fo...,eegrady
7,WEEK 8: Soothsayer - Team closest to projecte...,sgmaddox
8,WEEK 9: Keeping it Tight - Team with best perf...,JTizzzzle
9,WEEK 10: NFL Franchise Week - Team with the hi...,RossLikeSauce


In [240]:
headerColor = 'grey'
rowEvenColor = 'lightgrey'
rowOddColor = 'white'
df = SideBetWeeklyWins2024
figSideBetList = go.Figure(data=[go.Table(
    columnorder = [1,2],
    columnwidth = [50,25],
    header=dict(values=list(df.columns),
                #fill_color='paleturquoise',
                align='center',
                font=dict(color='white', size=20)),
    cells=dict(values=[df['Side Bet'], df.Winner],
               fill_color = [[rowOddColor,rowEvenColor,rowOddColor, rowEvenColor]*5],
               align=['left','center'],
               height = 20,
               font=dict(color='black', size=12)))
])
figSideBetList.update_layout(template='plotly_dark')
figSideBetList.update_layout(font_family="Courier New", title_font = dict(size=45))
figSideBetList.update_layout(width=800, height=1200)
figSideBetList.show()

In [241]:
tallyDF

,Team,Wins
0,JTizzzzle,2
1,eegrady,1
2,RReclam,0
3,bgmaddox,2
4,sgmaddox,2
5,jhuntmadd,1
6,RascalHazard,0
7,InfiniteJesse,0
8,BMoreBallers88,1
9,RossLikeSauce,1


In [242]:
figTally = px.bar_polar(tallyDF.sort_values('Wins'), r='Wins', theta='Team', color = 'Team',template='plotly_dark')
figTally.update_layout(polar=dict(
        radialaxis=dict(
            tickvals=[0,1,2,3,4],  # Specify the tick values
            ticktext=['', '1 Win','2 Wins']  # Customize tick labels
        )))
figTally.update_layout(font_family="Courier New", title_font = dict(size=45), showlegend=False)
figTally.show()

In [243]:
figSideBets = make_subplots(
            rows=1, cols=2, 
            shared_xaxes=False,
            horizontal_spacing=0.08, 
            vertical_spacing=0.05,
            shared_yaxes=True,
            column_widths=[0.4, 0.6],  # Adjust the width of each subplot
            specs=[[{"type": "polar"}, {"type": "table"}]]
                
            #subplot_titles=['Matchup Schedule','Win History']# Specify the chart types
        )
tallyDF = tallyDF.sort_values('Wins')
figSideBets.add_trace(go.Barpolar(r=tallyDF['Wins'], theta=tallyDF['Team'],marker_color = ['tomato', 'teal','gold']*12), row=1, col=1)
figSideBets.add_trace(go.Table(
    columnorder = [1,2],
    columnwidth = [65,25],
    header=dict(values=list(df.columns),
                #fill_color='paleturquoise',
                align='center',
                font=dict(color='white', size=20)),
    cells=dict(values=[df['Side Bet'], df.Winner],
               fill_color = [[rowOddColor,rowEvenColor,rowOddColor, rowEvenColor]*5],
               align=['left','center'],
               height = 30,
               font=dict(color='black', size=11))
                            ), row=1, col=2)
figSideBets.update_layout(polar=dict(
        angularaxis=dict(
            showline=False),
        radialaxis=dict(
            tickvals=[0,1,2,3,4],  # Specify the tick values
            ticktext=['', '1 Win','2 Wins']  # Customize tick labels
        )))
figSideBets.update_layout(font_family="Courier New", title_font = dict(size=45), showlegend=False, template = 'plotly_dark')
figSideBets.update_layout(width=1200, height=800, title_text = 'Side Bet Tally')
figSideBets.show()

In [244]:
Matches_2024DF.groupby('Team')['Total'].mean().reset_index().sort_values('Team')

,Team,Total
0,BMoreBallers88,100.616667
1,DirtyCommie,113.196667
2,InfiniteJesse,101.170000
3,JTizzzzle,108.143333
4,RReclam,105.286667
5,RascalHazard,102.833333
6,RossLikeSauce,117.956667
7,bgmaddox,118.856667
8,eegrady,109.966667
9,jhuntmadd,109.283333


In [245]:
WeekSet = 10
Last5 = Matches_2024DF[Matches_2024DF['Week'] > (WeekSet - 5)]
Last5.groupby('Team')['Total'].mean().reset_index().sort_values('Team')

,Team,Total
0,BMoreBallers88,102.674286
1,DirtyCommie,118.982857
2,InfiniteJesse,104.965714
3,JTizzzzle,113.460000
4,RReclam,96.177143
5,RascalHazard,107.994286
6,RossLikeSauce,124.754286
7,bgmaddox,121.542857
8,eegrady,113.374286
9,jhuntmadd,114.251429


In [246]:
AllSeasonsBreakoutDF

,team,matchup,player,points,starter,position,week_x,player_name,recent_teams,week_id,...,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr,Game_date_time,color,Score YTD,TotalYTD,Year,week_id_NFL
0,bgmaddox,1.0,David Johnson,22.70,1,RB,1,David Johnson,ARI,ARI-1,...,0.260256,0.0,19.700001,25.700001,Sunday 4 PM,mediumpurple,22.70,120.70,2019,NaN
1,bgmaddox,1.0,Drew Brees,24.80,1,QB,1,Drew Brees,NO,NO-1,...,NaN,0.0,20.799999,20.799999,Monday 7 PM,mediumpurple,24.80,185.60,2019,NaN
2,bgmaddox,1.0,Jacoby Brissett,19.50,0,QB,1,Jacoby Brissett,IND,IND-1,...,NaN,0.0,16.500000,16.500000,Sunday 4 PM,mediumpurple,19.50,218.54,2019,NaN
3,bgmaddox,1.0,David Njoku,11.70,1,TE,1,David Njoku,CLE,CLE-1,...,0.322005,0.0,9.700000,13.700000,Sunday 1 PM,mediumpurple,11.70,11.70,2019,NaN
4,bgmaddox,1.0,JuJu Smith-Schuster,10.80,1,WR,1,JuJu Smith-Schuster,PIT,PIT-1,...,0.351663,0.0,7.800000,13.800000,Sunday 8 PM,mediumpurple,10.80,86.40,2019,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166,InfiniteJesse,3.0,Tutu Atwell,4.50,0,WR,12,Tutu Atwell,LA,LA-12,...,0.203909,0.0,3.000000,6.000000,Sunday 8 PM,magenta,20.70,20.70,2024,LA-12
167,InfiniteJesse,3.0,Javonte Williams,1.40,0,RB,12,Javonte Williams,DEN,DEN-12,...,0.106299,0.0,0.400000,2.400000,Sunday 4 PM,magenta,102.60,102.60,2024,DEN-12
168,InfiniteJesse,3.0,De'Von Achane,19.10,1,RB,12,De'Von Achane,MIA,MIA-12,...,NaN,NaN,NaN,NaN,Sunday 1 PM,magenta,167.28,167.28,2024,NaN
169,InfiniteJesse,3.0,C.J. Stroud,19.68,1,QB,12,C.J. Stroud,HOU,HOU-12,...,NaN,0.0,15.680000,15.680000,Sunday 1 PM,magenta,192.20,192.20,2024,HOU-12


# TopPlayers():

In [247]:
def TopPlayers(position, df, threshold):
    TopPlayers = df[df['position'] == position]
    TopPlayers = TopPlayers[TopPlayers['TotalYTD']>threshold]

    figLine = px.line(TopPlayers,x='week_x',y='Score YTD', color = 'player',line_shape = 'spline',template='plotly_dark')
    figLine.update_layout(width=1000, height=800)
    figLine.show()
    return figLine


In [248]:
WRs = TopPlayers('WR',Best_2024,125)

In [249]:
RBs = TopPlayers('RB',Best_2024,75)

## Top Players Bar Chart

In [250]:
best_players = pd.DataFrame(Best_2024.groupby('player')['points'].sum()).reset_index().sort_values('points',ascending=False)

In [251]:
best_players

,player,points
46,Lamar Jackson,348.02
36,Joe Burrow,287.22
29,Jalen Hurts,264.64
39,Josh Allen,256.32
34,Jayden Daniels,255.12
52,Saquon Barkley,252.40
21,Derrick Henry,233.10
49,Patrick Mahomes,221.32
32,Jared Goff,218.54
27,Ja'Marr Chase,215.20


# Polar Plots

In [252]:
scaler = MinMaxScaler()
scalerStandard = StandardScaler()
Breakout_2024
Starters = Breakout_2024[Breakout_2024['starter'] == 1]
PosistionPolar = Starters.groupby(['team','position'])['points'].mean()
PosistionPolar = PosistionPolar.reset_index()
PosistionPivot = PosistionPolar.pivot(columns='position', index='team', values='points')
PivotPositions = PosistionPivot.columns
PivotTeams = PosistionPivot.index
PosistionPivot_scaled = scaler.fit_transform(PosistionPivot)

PosistionPivot_Standard_scaled = scalerStandard.fit_transform(PosistionPivot)
PosistionPivot_scaled = pd.DataFrame(PosistionPivot_scaled, columns = PivotPositions, index=PivotTeams)
PosistionPivot_Standard_scaled = pd.DataFrame(PosistionPivot_Standard_scaled, columns = PivotPositions, index=PivotTeams)

In [253]:
PosistionPivot_scaled

position,DEF,K,QB,RB,TE,WR
team,,,,,,
BMoreBallers88,1.000000,0.611336,0.218200,0.391540,0.316400,0.207226
DirtyCommie,0.597403,0.346154,1.000000,0.347998,0.369086,0.601779
InfiniteJesse,0.324675,0.720648,0.000000,0.479822,0.052019,0.623715
JTizzzzle,0.532468,0.000000,0.808112,0.596990,0.000000,0.444735
RReclam,0.779221,0.518219,0.394301,0.366600,0.270669,0.531424
RascalHazard,0.000000,0.951417,0.302272,0.346134,0.475601,0.572024
RossLikeSauce,0.623377,1.000000,0.320113,0.537867,0.260094,1.000000
bgmaddox,0.337662,0.348178,0.476062,0.824422,0.668813,0.629613
eegrady,0.566706,0.273279,0.330510,1.000000,1.000000,0.000000


In [254]:
PosistionPivot_Standard_scaled

position,DEF,K,QB,RB,TE,WR
team,,,,,,
BMoreBallers88,1.567203,0.557443,-0.687546,-0.409416,-0.244549,-1.332034
DirtyCommie,0.018170,-0.284344,2.266841,-0.590235,-0.072223,0.367754
InfiniteJesse,-1.031174,0.904439,-1.512117,-0.042806,-1.109292,0.462258
JTizzzzle,-0.231673,-1.383165,1.541704,0.443764,-1.279437,-0.308810
RReclam,0.717733,0.261853,-0.022069,-0.512984,-0.394126,0.064656
RascalHazard,-2.280393,1.636986,-0.369844,-0.597975,0.276168,0.239568
RossLikeSauce,0.118108,1.791206,-0.302423,0.198241,-0.428716,2.083346
bgmaddox,-0.981205,-0.277918,0.286902,1.388225,0.908132,0.487671
eegrady,-0.099938,-0.515675,-0.263135,2.117353,1.991383,-2.224790


In [255]:
PosistionAvg = Starters.groupby('position')['points'].mean()
PosistionPolar = round(PosistionPolar,2)
PosistionAvg = round(PosistionAvg,2)
PosistionPolar = PosistionPolar.reset_index()
PosistionAvg = PosistionAvg.reset_index()
PosistionPivot_scaled = PosistionPivot_scaled.reset_index()

PosistionPivot_Standard_scaled = PosistionPivot_Standard_scaled.reset_index()
teamlist = PosistionPivot_scaled.team.unique()
teamdict = dict(enumerate(team_list,1))
teamlistorder = teamdict.values()
rowlist = [1,1,1,2,2,2,3,3,3,4,4,4]
collist = [1,2,3,1,2,3,1,2,3,1,2,3]
rowdict = dict(enumerate(rowlist,1))
coldict = dict(enumerate(collist,1))
polars = ['polar']*12

polarlist = []
for i in range(1,13):
   newname = 'polar'+ str(i)
   polarlist.append(newname)
PolarTest = PosistionPivot_scaled[PosistionPivot_scaled['team']=='bgmaddox']
PolarTest = PosistionPivot_scaled[PosistionPivot_scaled['team']=='bgmaddox']
PolarTest = PolarTest.set_index('team').T

PolarTest
PolarTest.columns = ['points']
PolarTest = PolarTest.reset_index()

### Graph

In [256]:
figPolarAll = make_subplots(rows=4, cols=3, specs=[[{'type': 'polar'}, {'type': 'polar'}, {'type': 'polar'}],[{'type': 'polar'}, {'type': 'polar'}, {'type': 'polar'}],
                                                   [{'type': 'polar'}, {'type': 'polar'}, {'type': 'polar'} ],[{'type': 'polar'}, {'type': 'polar'}, {'type': 'polar'} ]],
                            subplot_titles=list(teamlistorder)
                            )
for i in range (1,13):
    PolarTest = PosistionPivot_scaled[PosistionPivot_scaled['team']==teamdict[i]]
    PolarTest = PolarTest.set_index('team').T
    PolarTest.columns = ['points']
    PolarTest = PolarTest.reset_index()
    figPolarAll.add_trace(go.Scatterpolar(r=PolarTest['points'], theta=PolarTest['position'], name=teamdict[i],fill='toself'), row=rowdict[i], col=coldict[i])
    figPolarAll.update_layout(
    polar1=dict(
        radialaxis=dict(
            tickvals=[.25,.5,.75,1,1.25],  # Specify the tick values
            ticktext=None  # Customize tick labels
        )
    ),
    polar2=dict(
        radialaxis=dict(
            tickvals=[.25,.5,.75,1,1.25],  # Specify the tick values
            #ticktext=['25%', '50%', '75%', '100%']  # Customize tick labels
        )
    ),
    polar3=dict(
        radialaxis=dict(
            tickvals=[.25,.5,.75,1,1.25],  # Specify the tick values
            #ticktext=['25%', '50%', '75%', '100%']  # Customize tick labels
        )
    ),
    polar4=dict(
        radialaxis=dict(
            tickvals=[.25,.5,.75,1,1.25],  # Specify the tick values
            ticktext=['25%', '50%', '75%', '100%']  # Customize tick labels
        )
    )
    ,
    polar5=dict(
        radialaxis=dict(
            tickvals=[.25,.5,.75,1,1.25],  # Specify the tick values
            ticktext=['25%', '50%', '75%', '100%']  # Customize tick labels
        )
    )
    ,
    polar6=dict(
        radialaxis=dict(
            tickvals=[.25,.5,.75,1,1.25],  # Specify the tick values
            ticktext=['25%', '50%', '75%', '100%']  # Customize tick labels
        )
    )
    ,
    polar7=dict(
        radialaxis=dict(
            tickvals=[.25,.5,.75,1,1.25],  # Specify the tick values
            ticktext=['25%', '50%', '75%', '100%']  # Customize tick labels
        )
    )
    ,
    polar8=dict(
        radialaxis=dict(
            tickvals=[.25,.5,.75,1,1.25],  # Specify the tick values
            ticktext=['25%', '50%', '75%', '100%']  # Customize tick labels
        )
    )
    ,
    polar9=dict(
        radialaxis=dict(
            tickvals=[.25,.5,.75,1,1.25],  # Specify the tick values
            ticktext=['25%', '50%', '75%', '100%']  # Customize tick labels
        )
    )
    ,
    polar10=dict(
        radialaxis=dict(
            tickvals=[.25,.5,.75,1,1.25],  # Specify the tick values
            ticktext=['25%', '50%', '75%', '100%']  # Customize tick labels
        )
    )
    ,
    polar11=dict(
        radialaxis=dict(
            tickvals=[.25,.5,.75,1,1.25],  # Specify the tick values
            ticktext=['25%', '50%', '75%', '100%']  # Customize tick labels
        )
    )
    ,
    polar12=dict(
        radialaxis=dict(
            tickvals=[.25,.5,.75,1,1.25],  # Specify the tick values
            ticktext=['25%', '50%', '75%', '100%']  # Customize tick labels
        )
    )
)

figPolarAll.update_layout(width=800, height=1200, template = 'plotly_dark', showlegend=False)


figPolarAll.update_layout(font_family="Courier New", title_font = dict(size=45))
figPolarAll.show()

In [257]:
'''
    polar1=dict(
        radialaxis=dict(
            tickvals=[.25,.5,.75,1,1.25],  # Specify the tick values
            ticktext=None  # Customize tick labels
        )
    ),
    polar2=dict(
        radialaxis=dict(
            tickvals=[.25,.5,.75,1,1.25],  # Specify the tick values
            #ticktext=['25%', '50%', '75%', '100%']  # Customize tick labels
        )
    ),
    polar3=dict(
        radialaxis=dict(
            tickvals=[.25,.5,.75,1,1.25],  # Specify the tick values
            #ticktext=['25%', '50%', '75%', '100%']  # Customize tick labels
        )
    ),
    polar4=dict(
        radialaxis=dict(
            tickvals=[.25,.5,.75,1,1.25],  # Specify the tick values
            ticktext=['25%', '50%', '75%', '100%']  # Customize tick labels
        )
    )
    ,
    polar5=dict(
        radialaxis=dict(
            tickvals=[.25,.5,.75,1,1.25],  # Specify the tick values
            ticktext=['25%', '50%', '75%', '100%']  # Customize tick labels
        )
    )
    ,
    polar6=dict(
        radialaxis=dict(
            tickvals=[.25,.5,.75,1,1.25],  # Specify the tick values
            ticktext=['25%', '50%', '75%', '100%']  # Customize tick labels
        )
    )
    ,
    polar7=dict(
        radialaxis=dict(
            tickvals=[.25,.5,.75,1,1.25],  # Specify the tick values
            ticktext=['25%', '50%', '75%', '100%']  # Customize tick labels
        )
    )
    ,
    polar8=dict(
        radialaxis=dict(
            tickvals=[.25,.5,.75,1,1.25],  # Specify the tick values
            ticktext=['25%', '50%', '75%', '100%']  # Customize tick labels
        )
    )
    ,
    polar9=dict(
        radialaxis=dict(
            tickvals=[.25,.5,.75,1,1.25],  # Specify the tick values
            ticktext=['25%', '50%', '75%', '100%']  # Customize tick labels
        )
    )
    ,
    polar10=dict(
        radialaxis=dict(
            tickvals=[.25,.5,.75,1,1.25],  # Specify the tick values
            ticktext=['25%', '50%', '75%', '100%']  # Customize tick labels
        )
    )
    ,
    polar11=dict(
        radialaxis=dict(
            tickvals=[.25,.5,.75,1,1.25],  # Specify the tick values
            ticktext=['25%', '50%', '75%', '100%']  # Customize tick labels
        )
    )
    ,
    polar12=dict(
        radialaxis=dict(
            tickvals=[.25,.5,.75,1,1.25],  # Specify the tick values
            ticktext=['25%', '50%', '75%', '100%']  # Customize tick labels
        )
    )'''

"\n    polar1=dict(\n        radialaxis=dict(\n            tickvals=[.25,.5,.75,1,1.25],  # Specify the tick values\n            ticktext=None  # Customize tick labels\n        )\n    ),\n    polar2=dict(\n        radialaxis=dict(\n            tickvals=[.25,.5,.75,1,1.25],  # Specify the tick values\n            #ticktext=['25%', '50%', '75%', '100%']  # Customize tick labels\n        )\n    ),\n    polar3=dict(\n        radialaxis=dict(\n            tickvals=[.25,.5,.75,1,1.25],  # Specify the tick values\n            #ticktext=['25%', '50%', '75%', '100%']  # Customize tick labels\n        )\n    ),\n    polar4=dict(\n        radialaxis=dict(\n            tickvals=[.25,.5,.75,1,1.25],  # Specify the tick values\n            ticktext=['25%', '50%', '75%', '100%']  # Customize tick labels\n        )\n    )\n    ,\n    polar5=dict(\n        radialaxis=dict(\n            tickvals=[.25,.5,.75,1,1.25],  # Specify the tick values\n            ticktext=['25%', '50%', '75%', '100%']  # Custom

In [258]:
figPolarAll = make_subplots(rows=4, cols=3, specs=[[{'type': 'polar'}, {'type': 'polar'}, {'type': 'polar'}],[{'type': 'polar'}, {'type': 'polar'}, {'type': 'polar'}],
                                                   [{'type': 'polar'}, {'type': 'polar'}, {'type': 'polar'} ],[{'type': 'polar'}, {'type': 'polar'}, {'type': 'polar'} ]],
                            subplot_titles=list(teamlistorder)
                            )
for i in range (1,13):
    PolarTest = PosistionPivot_Standard_scaled[PosistionPivot_Standard_scaled['team']==teamdict[i]]
    PolarTest = PolarTest.set_index('team').T
    PolarTest.columns = ['points']
    PolarTest = PolarTest.reset_index()
    figPolarAll.add_trace(go.Scatterpolar(r=PolarTest['points'], theta=PolarTest['position'], mode='markers+text',textposition='top center',name=teamdict[i],fill='toself', r0 = -2, dr = .5, text=round(PolarTest['points'],1)), row=rowdict[i], col=coldict[i])
    
    

figPolarAll.update_layout(width=800, height=1200, template = 'plotly_dark', showlegend=False)
figPolarAll.update_layout(polar1=dict(
                            radialaxis=dict(
                                range=[-3, 3],  # Set the radial range; adjust to fit your data scale
                                tickvals=[-3,0,3],  # Customize tick values for consistency
                                tickformat=".1f",  # Set the format for tick labels
                                tickangle=0,  # Optional: Adjust tick angle for readability
                                showticklabels=False
                            )),
                          polar2=dict(
                            radialaxis=dict(
                                range=[-3, 3],  # Set the radial range; adjust to fit your data scale
                                tickvals=[-3,0,3],  # Customize tick values for consistency
                                tickformat=".1f",  # Set the format for tick labels
                                tickangle=45,
                                showticklabels=False# Optional: Adjust tick angle for readability
                            )),
                          polar3=dict(
                            radialaxis=dict(
                                range=[-3, 3],  # Set the radial range; adjust to fit your data scale
                                tickvals=[-3,0,3],  # Customize tick values for consistency
                                tickformat=".1f",  # Set the format for tick labels
                                tickangle=45,
                                showticklabels=False# Optional: Adjust tick angle for readability
                            )),
                          polar4=dict(
                            radialaxis=dict(
                                range=[-3, 3],  # Set the radial range; adjust to fit your data scale
                                tickvals=[-3,0,3],  # Customize tick values for consistency
                                tickformat=".1f",  # Set the format for tick labels
                                tickangle=45,
                                showticklabels=False# Optional: Adjust tick angle for readability
                            )),
                          polar5=dict(
                            radialaxis=dict(
                                range=[-3, 3],  # Set the radial range; adjust to fit your data scale
                                tickvals=[-3,0,3],  # Customize tick values for consistency
                                tickformat=".1f",  # Set the format for tick labels
                                tickangle=45,
                                showticklabels=False# Optional: Adjust tick angle for readability
                            )),
                          polar6=dict(
                            radialaxis=dict(
                                range=[-3, 3],  # Set the radial range; adjust to fit your data scale
                                tickvals=[-3,0,3],  # Customize tick values for consistency
                                tickformat=".1f",  # Set the format for tick labels
                                tickangle=45,
                                showticklabels=False# Optional: Adjust tick angle for readability
                            )),
                          polar7=dict(
                            radialaxis=dict(
                                range=[-3, 3],  # Set the radial range; adjust to fit your data scale
                                tickvals=[-3,0,3],  # Customize tick values for consistency
                                tickformat=".1f",  # Set the format for tick labels
                                tickangle=45,
                                showticklabels=False# Optional: Adjust tick angle for readability
                            )),
                          polar8=dict(
                            radialaxis=dict(
                                range=[-3, 3],  # Set the radial range; adjust to fit your data scale
                                tickvals=[-3,0,3],  # Customize tick values for consistency
                                tickformat=".1f",  # Set the format for tick labels
                                tickangle=45,
                                showticklabels=False# Optional: Adjust tick angle for readability
                            )),
                          polar9=dict(
                            radialaxis=dict(
                                range=[-3, 3],  # Set the radial range; adjust to fit your data scale
                                tickvals=[-3,0,3],  # Customize tick values for consistency
                                tickformat=".1f",  # Set the format for tick labels
                                tickangle=45,
                                showticklabels=False# Optional: Adjust tick angle for readability
                            )),
                          polar10=dict(
                            radialaxis=dict(
                                range=[-3, 3],  # Set the radial range; adjust to fit your data scale
                                tickvals=[-3,0,3],  # Customize tick values for consistency
                                tickformat=".1f",  # Set the format for tick labels
                                tickangle=45,
                                showticklabels=False# Optional: Adjust tick angle for readability
                            )),
                          polar11=dict(
                            radialaxis=dict(
                                range=[-3, 3],  # Set the radial range; adjust to fit your data scale
                                tickvals=[-3,0,3],  # Customize tick values for consistency
                                tickformat=".1f",  # Set the format for tick labels
                                tickangle=45,
                                showticklabels=False# Optional: Adjust tick angle for readability
                            )),
                          polar12=dict(
                            radialaxis=dict(
                                range=[-3, 3],  # Set the radial range; adjust to fit your data scale
                                tickvals=[-3,0,3],  # Customize tick values for consistency
                                tickformat=".1f",  # Set the format for tick labels
                                tickangle=45,
                                showticklabels=False# Optional: Adjust tick angle for readability
                            )),
                          )

figPolarAll.update_layout(font_family="Courier New", title_font = dict(size=45))
figPolarAll.update_layout(
    title="Positional Strength",
    title_font=dict(
        
        size=35,
        
    )
)
figPolarAll.add_annotation(
    text="Std Dev from Average",
    xref="paper", yref="paper",
    x=-0.0, y=1.05, # Position relative to figure (right side, middle)
    showarrow=False,
    font=dict(
        family="Courier New, monospace",
        size=18,
        color="white"
    )
    )
figPolarAll.update_traces(textfont_size=10)
figPolarAll.show()

In [259]:
figPolar = px.line_polar(PolarTest, r="points", theta="position", line_close=True,
                    #color_discrete_sequence=px.colors.sequential.Plasma_r,
                    template="plotly_dark",)
figPolar.show()

# Hall of Fame & Shame

## Highest Scoring Losers

In [260]:
Losers = AllMatchesDF[AllMatchesDF['Won'] == 0]
TopTenLosers = Losers.sort_values('Total', ascending=False).head(10)
TopTenLosers['TeamName'] = TopTenLosers['Team'] + '<br>' + "W" +TopTenLosers['Week'].astype(str) + " " + TopTenLosers['Year'].astype(str)
TopTenLosers['Opp'] = round(TopTenLosers['Opp'],2)
TopTenLosers['OppName'] = TopTenLosers['Opp_team'] + ' - ' + TopTenLosers['Opp'].astype(str)
TopTenLosers = TopTenLosers.sort_values('Total')
teamcolors = {'JTizzzzle': 'lightsalmon', 'bgmaddox':'mediumpurple', 'jlglover':'cyan', 'RascalHazard':'pink', 'BMoreBallers88':'gold', 'eegrady':'teal', 'DirtyCommie':'lime', 'jhuntmadd':'orange', 'RReclam':'green', 'sgmaddox':'darkseagreen', 'RossLikeSauce':'red', 'InfiniteJesse':'magenta', 'YouthPastor': 'rosybrown'}

In [445]:
def TopLosers(AllMatches_DF):    
    Losers = AllMatches_DF[AllMatches_DF['Won'] == 0]
    TopTenLosers = Losers.sort_values('Total', ascending=False).head(10)
    TopTenLosers['TeamName'] = TopTenLosers['Team'] + '<br>' + "W" +TopTenLosers['Week'].astype(str) + " " + TopTenLosers['Year'].astype(str)
    TopTenLosers['Opp'] = round(TopTenLosers['Opp'],2)
    TopTenLosers['OppName'] = TopTenLosers['Opp_team'] + ' - ' + TopTenLosers['Opp'].astype(str)
    TopTenLosers = TopTenLosers.sort_values('Total')
    teamcolors = {'JTizzzzle': 'lightsalmon', 'bgmaddox':'mediumpurple', 'jlglover':'cyan', 'RascalHazard':'pink', 'BMoreBallers88':'gold', 'eegrady':'teal', 'DirtyCommie':'lime', 'jhuntmadd':'orange',
                'RReclam':'green', 'sgmaddox':'darkseagreen', 'RossLikeSauce':'red', 'InfiniteJesse':'magenta', 'YouthPastor': 'rosybrown', 'SweetDizzzzzle':'pink'}

    figLosers = go.Figure()
    figLosers.add_trace(go.Bar(
        x = TopTenLosers['Total'],
        y = TopTenLosers['TeamName'],
        name = 'Losers',
        orientation='h',
        text = TopTenLosers['Total'],
        marker_color = [teamcolors[team] for team in TopTenLosers['Team']],
        textfont=dict(size=16)
    ))
    figLosers.add_trace(go.Bar(
        x = TopTenLosers['Opp'],
        y = TopTenLosers['TeamName'],
        name = 'Winners',
        orientation='h',
        opacity=.4,
        text = TopTenLosers['OppName'],
        textfont=dict(color="white",size=14),
        ))
    figLosers.update_layout(template="plotly_dark")
    figLosers.update_layout(width=800, height=1200)
    figLosers.update_layout(font_family="Courier New", title_font = dict(size=45))
    figLosers.update_layout(barcornerradius=13)
    figLosers.update_traces(marker=dict(colorscale='Viridis')) 
    figLosers.update_layout(showlegend=False)
    figLosers.update_yaxes(
            tickfont=dict(
                family='Courier New',  # Font family
                size=18,         # Font size
                color='white'    # Font color
            ),
            title = None,
            #categoryorder="total ascending",
            #categoryarray = Week8GraphInfo['Difference']
        )
    figLosers.update_layout(
        title = "Biggest Losers: Highest Scores in Loss",
        title_font=dict(size=25)
    )
    
    figLosers.add_layout_image(
        dict(
            #source=Image.open(r'C:\Users\Brett\Documents\star1.png'),
            source=Image.open(r'/Users/brettmaddox/Downloads/Award-star-gold-3d.png'),
            xref="paper",  # Use paper coordinates for x-axis
            yref="paper",  # Use paper coordinates for y-axis
            x=.95,  # X-coordinate (0 is left, 1 is right)
            y=0.75,  # Y-coordinate (0 is bottom, 1 is top)
            sizex=.1,  # Width of the image (adjust for your image)
            sizey=.1,  # Height of the image
            xanchor="center", yanchor="middle"  # Center the image over the bar
        )
    )
    

    figLosers.show()
    return figLosers

In [446]:
toplosers = TopLosers(AllMatchesDF)

## Smallest Margins

In [262]:
TenSmallestMargins = AllMatchesDF.sort_values('Abs Margin').head(20)

In [263]:
TenSmallestMargins = TenSmallestMargins.sort_values('Margin').reset_index()

In [264]:
TenSmallestMargins['TeamGraph'] = TenSmallestMargins['Team'] + ' - ' + TenSmallestMargins['Year'].astype(str) + ' [' + TenSmallestMargins['Margin'].astype(str) + ']'

In [265]:
figMargin = px.bar(TenSmallestMargins, x='Margin',y=TenSmallestMargins.index,template = 'plotly_dark',title='Top 10 Smallest Margins', color = 'Week Index', orientation='h', text='TeamGraph' )
figMargin.update_layout(barmode='stack', yaxis={'categoryorder':'total ascending'})
figMargin.update_layout(barcornerradius=13)
figMargin.update_layout(width=1200, height=800)
figMargin.update_layout(title=dict(
            font=dict(
                size=35,
                family="Courier New")))  # Set the width and height in pixels
figMargin.update_traces(textfont_size=20, textangle=0, cliponaxis=True, textposition = 'auto', textfont=dict(weight='bold',family='Courier New',  # Font family
            size=15   # Font color
        ))
figMargin.update_layout(
        xaxis_title="Margin",  # Set x-axis title
        yaxis_title="",   # Set y-axis title
        xaxis=dict(
            title_font=dict(
                family="Courier New",  # Set font family for x-axis title
                size=30,          # Set font size for x-axis title
                color ='red',
                weight = 'bold'
            )
        ),
        yaxis=dict(
            title_font=dict(
                family="Courier New",  # Set font family for y-axis title
                size=30,          # Set font size for y-axis title
                color = 'green',
                weight = 'bold'
            )
        )
    )
figMargin.update_layout(yaxis=dict(showticklabels=False))
#Update the layout to hide the legend:
figMargin.update(layout_coloraxis_showscale=False)
figMargin.update_xaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=22,         # Font size
            color='white'    # Font color
        ),
        title = None,
        dtick=.2
    )
figMargin.show()

## Wall of Shame

In [559]:
Worst10 = AllMatchesDF.sort_values('Total')[0:10]

In [560]:
Best10 = AllMatchesDF.sort_values('Total', ascending=False)[0:10]

In [561]:
Best10

,Team,QB,RB1,RB2,WR1,WR2,TE,WRT,K,DEF,...,Opp,Margin,Week,Season,Week Index,Year,LeagueTotal,PercentTotal,Opp_team,Abs Margin
7,BMoreBallers88,{'Deshaun Watson': 28.16},{'Alvin Kamara': 18.8},{'Todd Gurley': 20.2},{'Davante Adams': 38.1},{'Tyler Lockett': 45.5},{'Dalton Schultz': 3.2},{'Tyler Boyd': 22.24},{'Michael Badgley': 7.0},{'KC': 21.0},...,79.68,124.52,7,Regular,21,2020,1184.92,17.2,JTizzzzle,124.52
3,RReclam,{'Jalen Hurts': 21.88},{'James Cook': 12.2},{'Raheem Mostert': 41.7},{'Justin Jefferson': 24.4},{'Keenan Allen': 37.46},{'Hunter Henry': 2.7},{'Kenneth Walker': 29.1},{'Jake Moody': 11.5},{'CLE': 14.0},...,105.38,89.56,3,Regular,59,2023,1415.86,13.8,DirtyCommie,89.56
7,eegrady,{'Matt Ryan': 34.9},{'Joe Mixon': 11.4},{'Leonard Fournette': 21.7},{'William Fuller': 45.7},{'Robert Woods': 7.3},{'Delanie Walker': 1.5},{'Austin Ekeler': 13.8},{'Justin Tucker': 16.0},{'PHI': 35.0},...,83.54,103.76,5,Regular,5,2019,1421.22,13.2,BMoreBallers88,103.76
0,BMoreBallers88,{'Deshaun Watson': 30.76},{'Alvin Kamara': 54.7},{'Le'Veon Bell': 4.4},{'Davante Adams': 37.7},{'Tyler Lockett': 5.9},{'Mark Andrews': 10.6},{'Jeff Wilson': 26.9},{'Younghoe Koo': 1.0},{'BAL': 11.0},...,142.12,40.84,16,Playoff,30,2020,1231.46,14.9,SweetDizzzzzle,40.84
3,jhuntmadd,{'Jared Goff': 50.06},{'Breece Hall': 12.1},{'Isaac Guerendo': 9.5},{'Davante Adams': 38.3},{'Mike Evans': 32.4},{'Brock Bowers': 5.0},{'Jaxon Smith-Njigba': 13.3},{'Cameron Dicker': 6.1},{'MIN': 9.0},...,95.64,80.12,15,Playoff,85,2024,1403.96,12.5,BMoreBallers88,80.12
10,eegrady,{'Jayden Daniels': 34.4},{'Jahmyr Gibbs': 23.4},{'Saquon Barkley': 44.2},{'Jakobi Meyers': 17.1},{'Michael Pittman': 12.6},{'T.J. Hockenson': 14.9},{'George Kittle': 17.2},{'Jason Sanders': 11.2},{'Missing': 0.0},...,109.02,65.98,12,Regular,82,2024,1298.88,13.5,sgmaddox,65.98
11,RossLikeSauce,{'Tom Brady': 29.92},{'Todd Gurley': 16.2},{'Chris Carson': 18.8},{'Michael Thomas': 35.7},{'Amari Cooper': 34.1},{'O.J. Howard': 1.5},{'Allen Robinson': 25.2},{'Ka'imi Fairbairn': 11.0},{'DAL': 1.0},...,104.82,68.60,5,Regular,5,2019,1421.22,12.2,RascalHazard,68.60
7,BMoreBallers88,{'Matthew Stafford': 29.38},{'James Conner': 22.5},{'Latavius Murray': 32.2},{'Stefon Diggs': 14.8},{'Kenny Stills': 3.7},{'Hunter Henry': 6.7},{'Tevin Coleman': 36.8},{'Robbie Gould': 9.0},{'SF': 18.0},...,97.70,75.38,8,Regular,8,2019,1420.28,12.2,RascalHazard,75.38
0,RascalHazard,{'Lamar Jackson': 47.42},{'Derrick Henry': 22.8},{'David Montgomery': 1.1},{'Hollywood Brown': 21.0},{'DeVonta Smith': 27.38},{'Mark Andrews': 24.9},{'Miles Sanders': 4.9},{'Daniel Carlson': 9.8},{'DAL': 10.0},...,133.30,36.00,3,Regular,45,2022,1206.10,14.0,eegrady,36.00
5,eegrady,{'Jayden Daniels': 44.42},{'Jahmyr Gibbs': 23.4},{'Saquon Barkley': 27.0},{'Jakobi Meyers': 5.1},{'Adam Thielen': 12.8},{'George Kittle': 14.6},{'T.J. Hockenson': 3.7},{'Jason Sanders': 23.7},{'BUF': 13.0},...,148.04,19.68,16,Playoff,86,2024,1488.28,11.3,BMoreBallers88,19.68


In [562]:
Best10Players = AllSeasonsBreakoutDF.sort_values('points', ascending=False)[0:10]

In [563]:
Worst10.sort_values('Year')

,Team,QB,RB1,RB2,WR1,WR2,TE,WRT,K,DEF,...,Opp,Margin,Week,Season,Week Index,Year,LeagueTotal,PercentTotal,Opp_team,Abs Margin
9,SweetDizzzzzle,{'Patrick Mahomes': 9.24},{'Marlon Mack': 7.1},{'Kerryon Johnson': 2.3},{'Kenny Golladay': 2.6},{'Larry Fitzgerald': 1.7},{'Jimmy Graham': 14.5},{'Mike Williams': 6.7},{'Jason Myers': 9.0},{'SEA': 2.0},...,100.22,-45.08,7,Regular,7,2019,1202.42,4.6,akbrown29,45.08
7,BillyRayGonnaGetcha,{'Lamar Jackson': 0.0},{'Ty Johnson': 0.0},{'Saquon Barkley': 6.4},{'Jerry Jeudy': 0.0},{'DJ Moore': 9.8},{'Jared Cook': 4.7},{'Emmanuel Sanders': 0.0},{'Tyler Bass': 7.0},{'CAR': 5.0},...,89.94,-57.04,15,Playoff,43,2021,999.18,3.3,eegrady,57.04
11,BillyRayGonnaGetcha,{'Lamar Jackson': 0.0},{'Ty Johnson': 0.0},{'Saquon Barkley': 3.3},{'Jerry Jeudy': 7.5},{'DJ Moore': 8.0},{'Jared Cook': 7.9},{'Emmanuel Sanders': 3.0},{'Tyler Bass': 9.0},{'CAR': -2.0},...,53.08,-16.38,16,Playoff,44,2021,1219.24,3.0,YouthPastor,16.38
10,YouthPastor,{'Jimmy Garoppolo': 15.18},{'AJ Dillon': 7.1},{'Rhamondre Stevenson': 0.0},{'Hunter Renfrow': 11.5},{'Chris Godwin': 0.0},{'George Kittle': 3.1},{'Aaron Jones': 11.2},{'Matt Prater': 1.0},{'MIN': 4.0},...,36.70,16.38,16,Playoff,44,2021,1219.24,4.4,BillyRayGonnaGetcha,16.38
11,RossLikeSauce,{'Russell Wilson': 0.0},{'Najee Harris': 9.5},{'Jamaal Williams': 5.8},{'Robert Woods': 4.1},{'Courtland Sutton': 3.8},{'Dallas Goedert': 0.0},{'Jerry Jeudy': 13.1},{'Matt Gay': 0.0},{'BUF': 0.0},...,116.54,-80.24,7,Regular,49,2022,1148.74,3.2,BMoreBallers88,80.24
6,RossLikeSauce,{'Russell Wilson': 0.0},{'Najee Harris': 14.6},{'Devin Singletary': 8.5},{'Stefon Diggs': 8.2},{'Courtland Sutton': 0.0},{'Foster Moreau': 3.0},{'Jamaal Williams': 3.3},{'Matt Gay': 7.8},{'BUF': -2.0},...,113.00,-69.60,15,Playoff,57,2022,1271.50,3.4,Just_Here_For_The_Snacks,69.60
7,BMoreBallers88,{'Tua Tagovailoa': 10.4},{'Aaron Jones': 4.4},{'J.K. Dobbins': 5.9},{'Davante Adams': 2.5},{'JuJu Smith-Schuster': 4.2},{'Dalton Schultz': 5.8},{'Jakobi Meyers': 17.3},{'Tyler Bass': 2.0},{'DEN': -5.0},...,74.78,-27.28,16,Playoff,58,2022,1254.66,3.8,JTizzzzle,27.28
2,JTizzzzle,{'Matt Ryan': 8.92},{'Clyde Edwards-Helaire': 9.2},{'D'Andre Swift': 0.0},{'Chase Claypool': 6.6},{'DK Metcalf': 1.7},{'David Njoku': 10.6},{'Rondale Moore': 3.6},{'Justin Tucker': 14.1},{'TB': 1.0},...,89.42,-33.70,7,Regular,49,2022,1148.74,4.9,Just_Here_For_The_Snacks,33.70
5,DirtyCommie,{'Geno Smith': 15.22},{'James Conner': 7.3},{'Austin Ekeler': 6.0},{'Jahan Dotson': 9.8},{'Garrett Wilson': -0.1},{'Trey McBride': 6.8},{'Zay Jones': 4.0},{'Riley Patterson': 3.0},{'LAC': 2.0},...,116.88,-62.86,11,Regular,67,2023,1241.08,4.4,RascalHazard,62.86
1,RascalHazard,{'Brock Purdy': 17.18},{'Tony Pollard': 7.5},{'J.K. Dobbins': 8.1},{'Garrett Wilson': 10.6},{'DeVonta Smith': 0.3},{'Travis Kelce': 3.7},{'Marvin Harrison': 3.6},{'Harrison Butker': 4.0},{'NO': 1.0},...,99.86,-43.88,7,Regular,77,2024,1206.18,4.6,bgmaddox,43.88


In [564]:
Worst10
Worst10['Total'] = Worst10['Total'].astype(int)
Worst10['TeamName'] = '<b>' + Worst10['Team'] + '</b>' + '<br>' + "W" +Worst10['Week'].astype(str) + " " + Worst10['Year'].astype(str)

In [565]:
Best10
Best10['Total'] = Best10['Total'].astype(int)
Best10['TeamName'] = '<b>'+ Best10['Team'] + '</b>' + '<br>' + "W" +Best10['Week'].astype(str) + " " + Best10['Year'].astype(str)

In [566]:
Worst10 = Worst10.sort_values('Total')
Worst10['Opacity'] = .2
Worst10.loc[1,'Opacity'] = 1
OpacityList = Worst10['Opacity'].to_list()

In [567]:
# Create an Opacity column where year == 2024 gets full opacity, otherwise more transparent
Worst10['Opacity'] = Worst10['Year'].astype(int).apply(lambda x: 1 if x == 2024 else 0.3)

In [568]:
figWorst = px.bar(Worst10, x='Total',y='TeamName', color = 'Team', orientation='h', template='plotly_dark', title = '<b>Hall of Shame:</b> Team', text = 'Total')
figWorst.update_layout(height = 1200, width= 800, barcornerradius=13, showlegend=False)
figWorst.update_layout(font_family="Courier New", title_font = dict(size=45))
figWorst.update_xaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=16,         # Font size
            color='white'    # Font color
        ),
        title = None,
        
            #categoryarray=time_order
    )
figWorst.update_yaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=16,         # Font size
            color='white'    # Font color
        ),
        title = None,
        categoryorder="total descending",
    )

# Add the image over a specific bar (adjust xref and yref as needed for placement)

figWorst.show()

## Hall of Fame: Team

In [459]:
figBest = px.bar(Best10, x='Total',y='TeamName', color = 'Team', orientation='h', template='plotly_dark', title = '<b>Hall of Fame:</b> Team', text = 'Total')
figBest.update_layout(height = 1200, width= 800, barcornerradius=13, showlegend=False)
figBest.update_layout(font_family="Courier New", title_font = dict(size=45))
figBest.update_xaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=16,         # Font size
            color='white'    # Font color
        ),
        title = None,
        
            #categoryarray=time_order
    )
figBest.update_yaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=16,         # Font size
            color='white'    # Font color
        ),
        title = None,
        categoryorder="total ascending",
    )
figBest.update_layout(
    font=dict(
        family="Courier New, monospace",
        size=20,  # Set the font size here
        color="white"
    )
)

figBest.add_layout_image(
    dict(
        #source=Image.open(r'/Users/brettmaddox/Library/CloudStorage/GoogleDrive-bgmaddox@gmail.com/Other computers/My Computer/Python Projects/Sleeper/star.png'),
        source=Image.open(r'/Users/brettmaddox/Downloads/Award-star-gold-3d.png'),
        xref="paper",  # Use paper coordinates for x-axis
        yref="paper",  # Use paper coordinates for y-axis
        x=0.92,  # X-coordinate (0 is left, 1 is right)
        y=0.05,  # Y-coordinate (0 is bottom, 1 is top)
        sizex=.1,  # Width of the image (adjust for your image)
        sizey=.1,  # Height of the image
        xanchor="center", yanchor="middle"  # Center the image over the bar
    )
)
# Add the image over a specific bar (adjust xref and yref as needed for placement)

figBest.show()

## Hall of Fame

In [461]:

Best10Players['TeamName'] = Best10Players['team'] + '<br>' + "W" +Best10Players['week_x'].astype(str) + " " + Best10Players['Year'].astype(str)

In [462]:
Best10Players['Player-Points'] = Best10Players.player + ' - ' + Best10Players.points.astype(str)

In [465]:
figBestPlayers = px.bar(Best10Players, x='points',y='TeamName', color = 'team', orientation='h', 
                        template='plotly_dark', title = 'Hall of Fame: Individual Players', text = 'Player-Points',
                        pattern_shape='Season', pattern_shape_sequence=['','+'])
figBestPlayers.update_layout(height = 1200, width= 800, barcornerradius=13, showlegend=False)
figBestPlayers.update_layout(font_family="Courier New", title_font = dict(size=30))
figBestPlayers.update_xaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=16,         # Font size
            color='white'    # Font color
        ),
        title = None,
        
            #categoryarray=time_order
    )
figBestPlayers.update_yaxes(
        tickfont=dict(
            family='Courier New',  # Font family
            size=16,         # Font size
            color='white'    # Font color
        ),
        title = None,
        categoryorder="total ascending",
    )

# Add the image over a specific bar (adjust xref and yref as needed for placement)
figBestPlayers.add_layout_image(
    dict(
        #source=Image.open(r'C:\Users\Brett\Documents\star1.png'),
        source=Image.open(r'/Users/brettmaddox/Downloads/Award-star-gold-3d.png'),
        xref="paper",  # Use paper coordinates for x-axis
        yref="paper",  # Use paper coordinates for y-axis
        x=0.95,  # X-coordinate (0 is left, 1 is right)
        y=0.25,  # Y-coordinate (0 is bottom, 1 is top)
        sizex=.1,  # Width of the image (adjust for your image)
        sizey=.1,  # Height of the image
        xanchor="center", yanchor="middle"  # Center the image over the bar
    )
)
figBestPlayers.show()

# Random Stats

## Unique Players

In [280]:
Breakout_2024.groupby(['team','starter'])['player'].nunique()

team            starter
BMoreBallers88  0          26
                1          30
DirtyCommie     0          21
                1          29
InfiniteJesse   0          17
                1          24
JTizzzzle       0          17
                1          20
RReclam         0          23
                1          41
RascalHazard    0          18
                1          22
RossLikeSauce   0          20
                1          22
bgmaddox        0          23
                1          23
eegrady         0          15
                1          18
jhuntmadd       0          25
                1          27
jlglover        0          24
                1          32
sgmaddox        0          16
                1          24
Name: player, dtype: int64

In [281]:
OpponentPoints = Breakout_2024.groupby(['team','opponent_team'])['points'].sum().reset_index().sort_values('points', ascending=False)
OpponentPoints

,team,opponent_team,points
37,DirtyCommie,CIN,168.60
229,bgmaddox,DAL,151.78
33,DirtyCommie,BAL,150.02
191,RossLikeSauce,BAL,145.44
250,bgmaddox,TB,131.52
...,...,...,...
43,DirtyCommie,HOU,2.90
44,DirtyCommie,IND,1.80
76,InfiniteJesse,KC,1.70
26,BMoreBallers88,SEA,0.80


In [282]:
OpponentPointsNoTeam = Breakout_2024.groupby(['team','opponent_team'])['points'].sum().round(1).reset_index().sort_values('points', ascending=False)
OpponentPointsNoTeamTOP =  OpponentPointsNoTeam.iloc[0:10]

In [283]:
OpponentPointsNoTeamTOP['TeamVs'] = OpponentPointsNoTeamTOP.team + ' vs. ' + OpponentPointsNoTeamTOP.opponent_team
OpponentPointsNoTeamTOP['Purpose'] = 'Points Against...'
OpponentPointsNoTeamTOP


,team,opponent_team,points,TeamVs,Purpose
37,DirtyCommie,CIN,168.6,DirtyCommie vs. CIN,Points Against...
229,bgmaddox,DAL,151.8,bgmaddox vs. DAL,Points Against...
33,DirtyCommie,BAL,150.0,DirtyCommie vs. BAL,Points Against...
191,RossLikeSauce,BAL,145.4,RossLikeSauce vs. BAL,Points Against...
250,bgmaddox,TB,131.5,bgmaddox vs. TB,Points Against...
5,BMoreBallers88,CHI,116.8,BMoreBallers88 vs. CHI,Points Against...
111,JTizzzzle,LV,114.7,JTizzzzle vs. LV,Points Against...
95,JTizzzzle,BAL,110.8,JTizzzzle vs. BAL,Points Against...
152,RReclam,SEA,110.8,RReclam vs. SEA,Points Against...
129,RReclam,CAR,108.7,RReclam vs. CAR,Points Against...


In [284]:
OpponentPointsNoTeamTOP

,team,opponent_team,points,TeamVs,Purpose
37,DirtyCommie,CIN,168.6,DirtyCommie vs. CIN,Points Against...
229,bgmaddox,DAL,151.8,bgmaddox vs. DAL,Points Against...
33,DirtyCommie,BAL,150.0,DirtyCommie vs. BAL,Points Against...
191,RossLikeSauce,BAL,145.4,RossLikeSauce vs. BAL,Points Against...
250,bgmaddox,TB,131.5,bgmaddox vs. TB,Points Against...
5,BMoreBallers88,CHI,116.8,BMoreBallers88 vs. CHI,Points Against...
111,JTizzzzle,LV,114.7,JTizzzzle vs. LV,Points Against...
95,JTizzzzle,BAL,110.8,JTizzzzle vs. BAL,Points Against...
152,RReclam,SEA,110.8,RReclam vs. SEA,Points Against...
129,RReclam,CAR,108.7,RReclam vs. CAR,Points Against...


In [285]:
TeamPoints = Breakout_2024.groupby(['team','recent_teams'])['points'].sum().round(1).reset_index().sort_values('points', ascending=False)
TeamPointsTOP = TeamPoints.iloc[0:10]

In [286]:
TeamPointsTOP['color'] = TeamPointsTOP.team.map(teamcolors)

In [287]:
TeamPointsTOP['TeamVs'] = TeamPointsTOP.team + ' w/ ' + TeamPointsTOP.recent_teams
TeamPointsTOP['Purpose'] = 'Points With...'
TeamPointsTOP

,team,recent_teams,points,color,TeamVs,Purpose
12,BMoreBallers88,MIN,398.8,gold,BMoreBallers88 w/ MIN,Points With...
198,jhuntmadd,SEA,356.7,orange,jhuntmadd w/ SEA,Points With...
231,sgmaddox,GB,350.1,darkseagreen,sgmaddox w/ GB,Points With...
24,DirtyCommie,BAL,348.0,lime,DirtyCommie w/ BAL,Points With...
127,RossLikeSauce,HOU,323.5,red,RossLikeSauce w/ HOU,Points With...
52,InfiniteJesse,DET,319.5,magenta,InfiniteJesse w/ DET,Points With...
66,JTizzzzle,CIN,316.7,lightsalmon,JTizzzzle w/ CIN,Points With...
149,bgmaddox,DET,299.2,mediumpurple,bgmaddox w/ DET,Points With...
204,jlglover,BUF,274.9,cyan,jlglover w/ BUF,Points With...
157,bgmaddox,PHI,264.6,mediumpurple,bgmaddox w/ PHI,Points With...


In [288]:
JointTopBottom = pd.concat([OpponentPointsNoTeamTOP,TeamPointsTOP ])

In [289]:
JointTopBottom

,team,opponent_team,points,TeamVs,Purpose,recent_teams,color
37,DirtyCommie,CIN,168.6,DirtyCommie vs. CIN,Points Against...,NaN,NaN
229,bgmaddox,DAL,151.8,bgmaddox vs. DAL,Points Against...,NaN,NaN
33,DirtyCommie,BAL,150.0,DirtyCommie vs. BAL,Points Against...,NaN,NaN
191,RossLikeSauce,BAL,145.4,RossLikeSauce vs. BAL,Points Against...,NaN,NaN
250,bgmaddox,TB,131.5,bgmaddox vs. TB,Points Against...,NaN,NaN
5,BMoreBallers88,CHI,116.8,BMoreBallers88 vs. CHI,Points Against...,NaN,NaN
111,JTizzzzle,LV,114.7,JTizzzzle vs. LV,Points Against...,NaN,NaN
95,JTizzzzle,BAL,110.8,JTizzzzle vs. BAL,Points Against...,NaN,NaN
152,RReclam,SEA,110.8,RReclam vs. SEA,Points Against...,NaN,NaN
129,RReclam,CAR,108.7,RReclam vs. CAR,Points Against...,NaN,NaN


In [290]:
figTeamPts = px.bar(JointTopBottom, x = 'points', y = 'TeamVs', facet_row='Purpose', template='plotly_dark')
figTeamPts.update_layout(height = 1200, width = 900, template = 'plotly_dark', barcornerradius = 7)
figTeamPts.show()


In [291]:
figTeamPoints = go.Figure()

figTeamPoints = make_subplots(
            rows=2, cols=1, 
            shared_xaxes=False,
            vertical_spacing =  .1,
            #column_widths=[0.5, 0.55],  # Adjust the width of each subplot
            specs=[[{"type": "bar"}],
                    [{"type": "bar"}]],
            subplot_titles=['Points With...','Points vs...']# Specify the chart types
        )
figTeamPoints.add_trace(
            go.Bar(
                x=TeamPointsTOP['points'], 
                y=TeamPointsTOP['TeamVs'], 
                
                text=TeamPointsTOP['points'],
                textangle = 0,
                textposition='auto',
                showlegend=False,
                orientation='h', 
                marker_color = TeamPointsTOP.points,
                opacity=.8,
                
            ),
            row=1, col=1
        )
figTeamPoints.add_trace(
            go.Bar(
                x=OpponentPointsNoTeamTOP['points'], 
                y=OpponentPointsNoTeamTOP['TeamVs'], 
                
                text=OpponentPointsNoTeamTOP['points'],
                textangle = 0,
                textposition='auto',
                showlegend=False,
                orientation='h',
                marker_color = OpponentPointsNoTeamTOP.points,
                opacity=.8
            ),
            row=2, col=1
        )

figTeamPoints.update_layout(height = 1200, width = 900, template = 'plotly_dark', barcornerradius = 7)
 
figTeamPoints.update_annotations(font_size=25)
figTeamPoints.update_layout(font_family="Courier New", title_font = dict(size=55))  
figTeamPoints.update_layout(title="<b>Points With & Against NFL Team</b>")




figTeamPoints.show()

# Experiments

## Margins of Victory & Loss

In [292]:
Matches_2024DF['Margin'] = Matches_2024DF['Margin'].round(2)
Margins = Matches_2024DF.groupby(['Team','Won',])['Margin'].agg(['mean']).round(2).reset_index()

In [293]:
Margins['Won'] = Margins['Won'].replace(0,'Loss').replace(1,'Win')

In [294]:
Margins

,Team,Won,mean
0,BMoreBallers88,Loss,-26.64
1,BMoreBallers88,Win,25.80
2,DirtyCommie,Loss,-15.17
3,DirtyCommie,Win,30.50
4,InfiniteJesse,Loss,-21.19
5,InfiniteJesse,Win,15.98
6,JTizzzzle,Loss,-25.64
7,JTizzzzle,Win,39.62
8,RReclam,Loss,-30.99
9,RReclam,Win,26.90


In [295]:
figMarginsofVictory = px.bar(Margins, y='Team',x='mean',template='plotly_dark', color='Won', title='Victory & Loss Margins', orientation='h', text='mean')
figMarginsofVictory.update_layout(height=1200, width=900)
figMarginsofVictory.update_layout(legend=dict(orientation="h"))
figMarginsofVictory.update_layout(barcornerradius=13)
figMarginsofVictory.update_layout(legend=dict(font=dict(size= 18)))
figMarginsofVictory.update_layout(font_family="Courier New", title_font = dict(size=40))  
figMarginsofVictory.update_layout(legend=dict(
    yanchor="top",
    y=1.05,
    xanchor="right",
    x=0.55
))
figMarginsofVictory.update_layout(
    font=dict(
        family="Courier New, monospace",
        size=18,  # Set the font size here
        color="White"
    )
)
figMarginsofVictory.update_layout(legend_title_text='',xaxis_title_font=dict(size=30))
figMarginsofVictory.update_xaxes(title_text="Avg")
figMarginsofVictory.update_yaxes(title_text="")
figMarginsofVictory.update_layout(
    xaxis=dict(
        tickfont=dict(size=20)  # Sets the x-axis label font size to 14
    )
)
figMarginsofVictory.update_layout(
    yaxis=dict(
        tickfont=dict(size=20)  # Sets the x-axis label font size to 14
    )
)
figMarginsofVictory.show()

## Trades Graph : Experiment

In [296]:


trades_req = requests.get(f'https://api.sleeper.app/v1/league/1122303756063965184/transactions/1')


trade_json = trades_req.json()


trades = json_normalize(trade_json)



In [297]:
trade_json

[{'status': 'failed',
  'type': 'waiver',
  'metadata': {'notes': 'This player was claimed by another owner.'},
  'created': 1725967100969,
  'settings': {'seq': 2},
  'leg': 1,
  'draft_picks': [],
  'creator': '868709284874182656',
  'transaction_id': '1139192225251889152',
  'adds': {'8131': 7},
  'consenter_ids': [7],
  'drops': None,
  'roster_ids': [7],
  'status_updated': 1726038428678,
  'waiver_budget': []},
 {'status': 'complete',
  'type': 'waiver',
  'metadata': {'notes': 'Your waiver claim was processed successfully!'},
  'created': 1725966975944,
  'settings': {'seq': 0},
  'leg': 1,
  'draft_picks': [],
  'creator': '868709284874182656',
  'transaction_id': '1139191700859138048',
  'adds': {'LAC': 7},
  'consenter_ids': [7],
  'drops': {'NO': 7},
  'roster_ids': [7],
  'status_updated': 1726038428678,
  'waiver_budget': []},
 {'status': 'complete',
  'type': 'waiver',
  'metadata': {'notes': 'Your waiver claim was processed successfully!'},
  'created': 1725842852213,
  